# Notebook 19 — Horse and Pedigree Identity

## Bounded question

> What do the runner-level `horse`, `sire`, `dam` and `damsire` fields represent in the source, how stable and complete are their labels, and which identity or pedigree relationships can be preserved safely without inventing entity equivalence from names alone?

## Initial governed scope

This notebook investigates four runner-level source-text fields:

- `horse`
- `sire`
- `dam`
- `damsire`

The source-field governance register assigns all four to the `horse_and_pedigree_identity` family, requires their raw values to be preserved and leaves their semantics pending.

The investigation begins with source lineage and physical profiling only. At this stage, no assumption is made that:

- a source string is a stable real-world entity identifier;
- identical strings always refer to the same horse;
- different strings always refer to different horses;
- a terminal country suffix is authoritative nationality evidence;
- stripping a suffix creates a globally unique name;
- `horse + country suffix` is a permanent natural key;
- repeated pedigree labels are complete, correct or internally consistent.

Raw source labels, parsed display names, embedded suffixes, source-level label identity, provisional entity candidates, verified real-world entities and pedigree assertions will remain separate concepts. Any future normalization must be reversible and must preserve physical source lineage, confidence and review status.

## Stage 1 — Source lineage and governed population

This stage establishes the immutable source, read-only controls, complete governed runner population and provisional race key before interpreting any horse or pedigree label.

The source is:

- database: `data/raw/form_2015-present/form_2015-present/raceform.db`
- table: `data`
- governed row predicate: `rowid <> 1`
- provisional race identity: `date + course + off`

The established source population is expected to contain:

- 1,851,285 governed runner rows;
- 189,043 provisional races;
- 37 source columns.

The first code cell opens SQLite in read-only mode, confirms the source schema, reconciles the governed runner and provisional-race counts, and confirms that `horse`, `sire`, `dam` and `damsire` are present. It does not parse, normalize or interpret any name.


In [1]:
from pathlib import Path
import sqlite3

import pandas as pd


# Resolve the immutable source explicitly from the notebook directory.
PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
RACE_KEY_COLUMNS = ["date", "course", "off"]
HORSE_IDENTITY_FIELDS = ["horse", "sire", "dam", "damsire"]

EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043
EXPECTED_SOURCE_COLUMNS = 37

if not SOURCE_DB_PATH.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DB_PATH}")

# Open SQLite read-only so the notebook cannot mutate the source database.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    schema = pd.read_sql_query(f"PRAGMA table_info({SOURCE_TABLE})", connection)
    source_columns = schema["name"].tolist()

    missing_fields = [
        field
        for field in HORSE_IDENTITY_FIELDS
        if field not in source_columns
    ]
    if missing_fields:
        raise AssertionError(f"Missing horse-identity fields: {missing_fields}")

    runner_rows = connection.execute(
        f"SELECT COUNT(*) FROM {SOURCE_TABLE} WHERE {DATA_ROW_PREDICATE}"
    ).fetchone()[0]

    provisional_races = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT date, course, off
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        )
        """
    ).fetchone()[0]
finally:
    connection.close()

assert runner_rows == EXPECTED_RUNNER_ROWS
assert provisional_races == EXPECTED_PROVISIONAL_RACES
assert len(source_columns) == EXPECTED_SOURCE_COLUMNS

source_lineage_summary = pd.DataFrame(
    [
        ("source database", SOURCE_DB_PATH.relative_to(PROJECT_ROOT).as_posix()),
        ("source table", SOURCE_TABLE),
        ("data-row predicate", DATA_ROW_PREDICATE),
        ("runner rows", runner_rows),
        ("provisional races", provisional_races),
        ("source columns", len(source_columns)),
        ("provisional race key", " + ".join(RACE_KEY_COLUMNS)),
        ("horse-identity fields present", ", ".join(HORSE_IDENTITY_FIELDS)),
    ],
    columns=["measure", "value"],
)

print("Governed source population confirmed")
source_lineage_summary


Governed source population confirmed


,measure,value
0,source database,data/raw/form_2015-present/form_2015-present/r...
1,source table,data
2,data-row predicate,rowid <> 1
3,runner rows,1851285
4,provisional races,189043
5,source columns,37
6,provisional race key,date + course + off
7,horse-identity fields present,"horse, sire, dam, damsire"


## Stage 2 — Confirm inherited source-field governance

Before profiling the contents of the four fields, this stage reads their existing rows from `data/reference/source_field_governance.csv`.

The check is limited to confirming the inherited starting position:

- each field is recorded at runner grain;
- each belongs to `horse_and_pedigree_identity`;
- raw preservation is required;
- the current blank policy is retained exactly as governed;
- semantic status remains pending;
- the existing governing notebook attribution is preserved.

This stage does not revise the register or infer anything from the field labels themselves.


In [ ]:
SOURCE_FIELD_GOVERNANCE_PATH = (
    PROJECT_ROOT / "data" / "reference" / "source_field_governance.csv"
)

if not SOURCE_FIELD_GOVERNANCE_PATH.exists():
    raise FileNotFoundError(
        f"Source-field governance register not found: "
        f"{SOURCE_FIELD_GOVERNANCE_PATH}"
    )

source_field_governance = pd.read_csv(SOURCE_FIELD_GOVERNANCE_PATH)

horse_identity_governance = (
    source_field_governance.loc[
        source_field_governance["source_field"].isin(HORSE_IDENTITY_FIELDS),
        [
            "ordinal",
            "source_field",
            "declared_type",
            "grain",
            "field_family",
            "raw_preservation",
            "blank_policy",
            "dash_policy",
            "zero_policy",
            "governed_by",
            "status",
        ],
    ]
    .sort_values("ordinal")
    .reset_index(drop=True)
)

assert horse_identity_governance["source_field"].tolist() == HORSE_IDENTITY_FIELDS
assert horse_identity_governance["grain"].eq("runner").all()
assert horse_identity_governance["field_family"].eq(
    "horse_and_pedigree_identity"
).all()
assert horse_identity_governance["raw_preservation"].eq("required").all()
assert horse_identity_governance["status"].eq("pending_semantics").all()

print("Inherited horse-identity governance confirmed")
horse_identity_governance


## Stage 2 — Confirm inherited source-field governance

This stage loads the existing governance rows for `horse`, `sire`, `dam` and `damsire` without changing them.

The expected inherited position is:

- runner-level grain;
- field family `horse_and_pedigree_identity`;
- raw preservation required;
- no dash-specific missing-value rule;
- contextual values must not be interpreted without evidence;
- semantics remain pending.

This stage only confirms the current governed starting position. It does not decide whether labels identify real-world horses, whether suffixes should be parsed, or whether pedigree relationships are stable.

In [2]:
# Confirm the inherited governance position before interpreting any labels.
SOURCE_FIELD_GOVERNANCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "source_field_governance.csv"
)

if not SOURCE_FIELD_GOVERNANCE_PATH.exists():
    raise FileNotFoundError(
        f"Source-field governance file not found: "
        f"{SOURCE_FIELD_GOVERNANCE_PATH}"
    )

source_field_governance = pd.read_csv(SOURCE_FIELD_GOVERNANCE_PATH)

required_governance_columns = {
    "source_field",
    "declared_type",
    "grain",
    "field_family",
    "raw_preservation",
    "blank_policy",
    "dash_policy",
    "zero_policy",
    "governed_by",
    "status",
}

missing_governance_columns = (
    required_governance_columns
    - set(source_field_governance.columns)
)

if missing_governance_columns:
    raise AssertionError(
        "Missing governance columns: "
        f"{sorted(missing_governance_columns)}"
    )

horse_identity_governance = (
    source_field_governance.loc[
        source_field_governance["source_field"].isin(
            HORSE_IDENTITY_FIELDS
        ),
        [
            "source_field",
            "declared_type",
            "grain",
            "field_family",
            "raw_preservation",
            "blank_policy",
            "dash_policy",
            "zero_policy",
            "governed_by",
            "status",
        ],
    ]
    .set_index("source_field")
    .reindex(HORSE_IDENTITY_FIELDS)
    .reset_index()
)

assert len(horse_identity_governance) == len(HORSE_IDENTITY_FIELDS)
assert horse_identity_governance["source_field"].notna().all()
assert horse_identity_governance["declared_type"].eq("TEXT").all()
assert horse_identity_governance["grain"].eq("runner").all()
assert horse_identity_governance["field_family"].eq(
    "horse_and_pedigree_identity"
).all()
assert horse_identity_governance["raw_preservation"].eq(
    "required"
).all()
assert horse_identity_governance["dash_policy"].eq(
    "not_expected"
).all()
assert horse_identity_governance["zero_policy"].eq(
    "contextual_value"
).all()
assert horse_identity_governance["status"].eq(
    "pending_semantics"
).all()

print("Inherited horse-identity governance confirmed")
horse_identity_governance

Inherited horse-identity governance confirmed


,source_field,declared_type,grain,field_family,raw_preservation,blank_policy,dash_policy,zero_policy,governed_by,status
0,horse,TEXT,runner,horse_and_pedigree_identity,required,unresolved_missing,not_expected,contextual_value,Notebook 03,pending_semantics
1,sire,TEXT,runner,horse_and_pedigree_identity,required,field_not_supplied,not_expected,contextual_value,Notebook 10,pending_semantics
2,dam,TEXT,runner,horse_and_pedigree_identity,required,field_not_supplied,not_expected,contextual_value,Notebook 10,pending_semantics
3,damsire,TEXT,runner,horse_and_pedigree_identity,required,field_not_supplied,not_expected,contextual_value,Notebook 10,pending_semantics


## Stage 3 — Physical storage and completeness profile

This stage profiles the four fields as physically stored, before parsing names or interpreting identity.

For each field it establishes:

- SQLite storage classes;
- SQL null count;
- empty-string count;
- whitespace-only count;
- populated row count;
- distinct populated raw labels;
- minimum and maximum populated string length.

The counts remain field-specific. In particular, the inherited distinction between `unresolved_missing` for `horse` and `field_not_supplied` for pedigree fields is not treated as confirmed semantics merely because it appears in the governance register.

No trimming, suffix parsing, case normalization or punctuation normalization is applied. Distinct counts refer to exact source strings.

In [3]:
# Profile physical storage and basic completeness without normalizing labels.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

profile_rows = []

try:
    for field in HORSE_IDENTITY_FIELDS:
        storage_classes = pd.read_sql_query(
            f"""
            SELECT
                typeof("{field}") AS storage_class,
                COUNT(*) AS row_count
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY typeof("{field}")
            ORDER BY row_count DESC, storage_class
            """,
            connection,
        )

        summary = connection.execute(
            f"""
            SELECT
                COUNT(*) AS total_rows,
                SUM(CASE WHEN "{field}" IS NULL THEN 1 ELSE 0 END)
                    AS sql_null_rows,
                SUM(CASE WHEN "{field}" = '' THEN 1 ELSE 0 END)
                    AS empty_string_rows,
                SUM(
                    CASE
                        WHEN "{field}" IS NOT NULL
                         AND "{field}" <> ''
                         AND TRIM("{field}") = ''
                        THEN 1
                        ELSE 0
                    END
                ) AS whitespace_only_rows,
                SUM(
                    CASE
                        WHEN "{field}" IS NOT NULL
                         AND TRIM("{field}") <> ''
                        THEN 1
                        ELSE 0
                    END
                ) AS populated_rows,
                COUNT(
                    DISTINCT CASE
                        WHEN "{field}" IS NOT NULL
                         AND TRIM("{field}") <> ''
                        THEN "{field}"
                    END
                ) AS distinct_populated_raw_labels,
                MIN(
                    CASE
                        WHEN "{field}" IS NOT NULL
                         AND TRIM("{field}") <> ''
                        THEN LENGTH("{field}")
                    END
                ) AS minimum_populated_length,
                MAX(
                    CASE
                        WHEN "{field}" IS NOT NULL
                         AND TRIM("{field}") <> ''
                        THEN LENGTH("{field}")
                    END
                ) AS maximum_populated_length
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            """
        ).fetchone()

        (
            total_rows,
            sql_null_rows,
            empty_string_rows,
            whitespace_only_rows,
            populated_rows,
            distinct_populated_raw_labels,
            minimum_populated_length,
            maximum_populated_length,
        ) = summary

        storage_class_summary = ", ".join(
            f"{row.storage_class}: {row.row_count:,}"
            for row in storage_classes.itertuples(index=False)
        )

        profile_rows.append(
            {
                "source_field": field,
                "storage_classes": storage_class_summary,
                "total_rows": total_rows,
                "sql_null_rows": sql_null_rows,
                "empty_string_rows": empty_string_rows,
                "whitespace_only_rows": whitespace_only_rows,
                "populated_rows": populated_rows,
                "distinct_populated_raw_labels": (
                    distinct_populated_raw_labels
                ),
                "minimum_populated_length": minimum_populated_length,
                "maximum_populated_length": maximum_populated_length,
            }
        )
finally:
    connection.close()

horse_identity_physical_profile = pd.DataFrame(profile_rows)

assert horse_identity_physical_profile["total_rows"].eq(
    EXPECTED_RUNNER_ROWS
).all()

assert (
    horse_identity_physical_profile[
        [
            "sql_null_rows",
            "empty_string_rows",
            "whitespace_only_rows",
            "populated_rows",
        ]
    ].sum(axis=1)
    == EXPECTED_RUNNER_ROWS
).all()

print("Physical storage and completeness profile confirmed")
horse_identity_physical_profile

Physical storage and completeness profile confirmed


,source_field,storage_classes,total_rows,sql_null_rows,empty_string_rows,whitespace_only_rows,populated_rows,distinct_populated_raw_labels,minimum_populated_length,maximum_populated_length
0,horse,"text: 1,851,285",1851285,0,0,0,1851285,208631,7,26
1,sire,"text: 1,851,285",1851285,0,0,0,1851285,5445,8,26
2,dam,"text: 1,851,285",1851285,0,5,0,1851280,104972,5,26
3,damsire,"text: 1,851,285",1851285,0,21,0,1851264,6078,3,20


### Stage 3a — Inspect the exceptional blank pedigree rows

The aggregate profile found no missing `horse` or `sire` labels, five empty `dam` values and twenty-one empty `damsire` values.

This cell retrieves those exceptional rows with physical and race lineage. It does not replace the blanks, infer pedigrees or decide whether the values are extraction defects.

The purpose is to determine whether the blanks are isolated runner-level anomalies, concentrated within particular races or jurisdictions, or associated with other unusual pedigree values.

In [4]:
# Inspect every exceptional blank pedigree value with exact source lineage.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    blank_pedigree_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            race_id AS supplied_race_id,
            date,
            course,
            off,
            horse,
            sire,
            dam,
            damsire,
            sex,
            age
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND (
                dam = ''
                OR damsire = ''
              )
        ORDER BY
            date,
            course,
            off,
            horse,
            source_rowid
        """,
        connection,
    )
finally:
    connection.close()

empty_dam_mask = blank_pedigree_rows["dam"].eq("")
empty_damsire_mask = blank_pedigree_rows["damsire"].eq("")
both_empty_mask = empty_dam_mask & empty_damsire_mask

assert blank_pedigree_rows["horse"].ne("").all()
assert blank_pedigree_rows["sire"].ne("").all()
assert int(empty_dam_mask.sum()) == 5
assert int(empty_damsire_mask.sum()) == 21

expected_union_rows = int(
    empty_dam_mask.sum()
    + empty_damsire_mask.sum()
    - both_empty_mask.sum()
)

assert len(blank_pedigree_rows) == expected_union_rows

blank_pedigree_summary = pd.DataFrame(
    [
        {
            "exception": "empty dam",
            "runner_rows": int(empty_dam_mask.sum()),
            "distinct_horse_labels": int(
                blank_pedigree_rows.loc[
                    empty_dam_mask,
                    "horse",
                ].nunique()
            ),
            "distinct_provisional_races": int(
                blank_pedigree_rows.loc[
                    empty_dam_mask,
                    ["date", "course", "off"],
                ].drop_duplicates().shape[0]
            ),
        },
        {
            "exception": "empty damsire",
            "runner_rows": int(empty_damsire_mask.sum()),
            "distinct_horse_labels": int(
                blank_pedigree_rows.loc[
                    empty_damsire_mask,
                    "horse",
                ].nunique()
            ),
            "distinct_provisional_races": int(
                blank_pedigree_rows.loc[
                    empty_damsire_mask,
                    ["date", "course", "off"],
                ].drop_duplicates().shape[0]
            ),
        },
        {
            "exception": "both dam and damsire empty",
            "runner_rows": int(both_empty_mask.sum()),
            "distinct_horse_labels": int(
                blank_pedigree_rows.loc[
                    both_empty_mask,
                    "horse",
                ].nunique()
            ),
            "distinct_provisional_races": int(
                blank_pedigree_rows.loc[
                    both_empty_mask,
                    ["date", "course", "off"],
                ].drop_duplicates().shape[0]
            ),
        },
        {
            "exception": "unique affected runner rows",
            "runner_rows": len(blank_pedigree_rows),
            "distinct_horse_labels": int(
                blank_pedigree_rows["horse"].nunique()
            ),
            "distinct_provisional_races": int(
                blank_pedigree_rows[
                    ["date", "course", "off"]
                ].drop_duplicates().shape[0]
            ),
        },
    ]
)

print("Exceptional blank pedigree rows confirmed")
display(blank_pedigree_summary)
blank_pedigree_rows

Exceptional blank pedigree rows confirmed


,exception,runner_rows,distinct_horse_labels,distinct_provisional_races
0,empty dam,5,5,5
1,empty damsire,21,18,20
2,both dam and damsire empty,5,5,5
3,unique affected runner rows,21,18,20


,source_rowid,supplied_race_id,date,course,off,horse,sire,dam,damsire,sex,age
0,40683,624701,2015-04-18,Nakayama (JPN),7:40,Country Snow (JPN),Timber Country (USA),Snow Style (USA),,G,8
1,127268,638862,2015-10-15,Mombetsu (JPN),11:07,Derma Okaru (JPN),Million Disk (JPN),Admire Lap (JPN),,F,2
2,191333,649077,2016-04-13,Funabashi (JPN),11:07,Kura Carmen (JPN),Star King Man (USA),Kura Masa Shuttle (JPN),,M,7
3,234163,655460,2016-07-06,Kawasaki (JPN),11:07,Kura Carmen (JPN),Star King Man (USA),Kura Masa Shuttle (JPN),,M,7
4,254211,657855,2016-08-18,Saga (JPN),11:07,Kassai (JPN),Screen Hero (JPN),,,H,5
5,255522,657854,2016-08-21,San Sebastian (SPA),6:05,Dagoberto (SPA),Dyhim Diamond (IRE),,,H,6
6,266790,659796,2016-09-15,Urawa (JPN),11:07,Legarsi (JPN),Eishin Sandy (JPN),Pink Cutie (JPN),,H,5
7,279292,661298,2016-10-10,Morioka (JPN),11:07,Sea Chrome (JPN),Laurel Guerreiro (JPN),Neroli (JPN),,C,4
8,291273,663759,2016-11-03,Kawasaki (JPN),12:07,Sea Chrome (JPN),Laurel Guerreiro (JPN),Neroli (JPN),,C,4
9,354528,673713,2017-04-12,Funabashi (JPN),11:07,Kura Carmen (JPN),Star King Man (USA),Kura Masa Shuttle (JPN),,M,8


### Stage 3b — Boundary whitespace, control characters and punctuation

The completeness profile counts exact source strings, but populated text may still contain hidden or structurally significant characters.

This stage tests the four fields for:

- leading or trailing whitespace;
- repeated internal spaces;
- tab, carriage-return, newline or other ASCII control characters;
- punctuation appearing anywhere in populated labels;
- representative labels containing each punctuation character.

No character is classified as erroneous merely because it is unusual. Apostrophes, hyphens, periods, digits and parentheses may be legitimate parts of source labels or suffix syntax.

The output is descriptive only. No normalization rule is created at this stage.

In [5]:
# Profile hidden whitespace, control characters and exact punctuation usage.
import string


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

character_profile_rows = []
punctuation_rows = []

try:
    for field in HORSE_IDENTITY_FIELDS:
        field_values = pd.read_sql_query(
            f"""
            SELECT DISTINCT "{field}" AS raw_label
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND "{field}" IS NOT NULL
              AND "{field}" <> ''
            """,
            connection,
        )["raw_label"]

        leading_whitespace = field_values.str.match(r"^\s", na=False)
        trailing_whitespace = field_values.str.contains(r"\s$", na=False)
        repeated_internal_spaces = field_values.str.contains(
            r" {2,}",
            regex=True,
            na=False,
        )
        ascii_control_characters = field_values.str.contains(
            r"[\x00-\x1f\x7f]",
            regex=True,
            na=False,
        )

        character_profile_rows.append(
            {
                "source_field": field,
                "distinct_populated_raw_labels": len(field_values),
                "labels_with_leading_whitespace": int(
                    leading_whitespace.sum()
                ),
                "labels_with_trailing_whitespace": int(
                    trailing_whitespace.sum()
                ),
                "labels_with_repeated_internal_spaces": int(
                    repeated_internal_spaces.sum()
                ),
                "labels_with_ascii_control_characters": int(
                    ascii_control_characters.sum()
                ),
            }
        )

        for punctuation_character in string.punctuation:
            contains_character = field_values.str.contains(
                punctuation_character,
                regex=False,
                na=False,
            )

            if not contains_character.any():
                continue

            examples = (
                field_values.loc[contains_character]
                .sort_values()
                .head(5)
                .tolist()
            )

            punctuation_rows.append(
                {
                    "source_field": field,
                    "character": punctuation_character,
                    "unicode_code_point": (
                        f"U+{ord(punctuation_character):04X}"
                    ),
                    "distinct_labels": int(contains_character.sum()),
                    "examples": " | ".join(examples),
                }
            )
finally:
    connection.close()

horse_identity_character_profile = pd.DataFrame(
    character_profile_rows
)

horse_identity_punctuation_profile = (
    pd.DataFrame(punctuation_rows)
    .sort_values(
        ["source_field", "character"],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert horse_identity_character_profile[
    "distinct_populated_raw_labels"
].tolist() == horse_identity_physical_profile[
    "distinct_populated_raw_labels"
].tolist()

print("Boundary-whitespace and control-character profile confirmed")
display(horse_identity_character_profile)

print("Exact ASCII punctuation vocabulary confirmed")
horse_identity_punctuation_profile

Boundary-whitespace and control-character profile confirmed


,source_field,distinct_populated_raw_labels,labels_with_leading_whitespace,labels_with_trailing_whitespace,labels_with_repeated_internal_spaces,labels_with_ascii_control_characters
0,horse,208631,0,0,1,0
1,sire,5445,0,0,0,0
2,dam,104972,0,0,0,0
3,damsire,6078,0,0,0,0


Exact ASCII punctuation vocabulary confirmed


,source_field,character,unicode_code_point,distinct_labels,examples
0,dam,(,U+0028,90074,A And Bs Gift (IRE) | A Beautiful Mind (GER) |...
1,dam,),U+0029,90074,A And Bs Gift (IRE) | A Beautiful Mind (GER) |...
2,dam,-,U+002D,85,A-To-Z (IRE) | All-Together (IRE) | Anne-Lise ...
3,dam,.,U+002E,5,A. P. Sonata (USA) | Alana B. (USA) | E. T. In...
4,dam,`,U+0060,13,Ambrosianella` (FR) | Baby It`s You (BRZ) | Bu...
5,damsire,*,U+002A,4,Ut*clafouti | Ut*cupidon | Ut*mangarose | Ut*w...
6,damsire,-,U+002D,8,Ali-Royal | Dano-Mast | Ela-Mana-Mou | High-Ri...
7,damsire,.,U+002E,3,A.P. Indy | Mt. Livermore | T. H. Approval
8,horse,(,U+0028,208631,A A Agility (NZ) | A Americ Te Specso (NZ) | A...
9,horse,),U+0029,208631,A A Agility (NZ) | A Americ Te Specso (NZ) | A...


### Stage 3c — Inspect the repeated internal-space exception

Only one distinct populated horse label contains two or more consecutive spaces.

This cell retrieves every occurrence of that exact label with source lineage and pedigree context. It does not collapse the spaces or assume that the label is erroneous.

The purpose is to determine whether the spacing is stable across repeated appearances and whether a corresponding single-space label also exists in the source.

In [6]:
# Inspect the sole repeated-internal-space horse label and possible variants.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    repeated_space_labels = pd.read_sql_query(
        f"""
        SELECT DISTINCT horse AS raw_horse_label
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND horse GLOB '*  *'
        ORDER BY horse
        """,
        connection,
    )

    assert len(repeated_space_labels) == 1

    repeated_space_label = repeated_space_labels.loc[
        0,
        "raw_horse_label",
    ]
    collapsed_space_label = " ".join(repeated_space_label.split())

    repeated_space_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            race_id AS supplied_race_id,
            date,
            course,
            off,
            horse,
            sire,
            dam,
            damsire,
            sex,
            age
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND horse IN (?, ?)
        ORDER BY
            horse,
            date,
            course,
            off,
            source_rowid
        """,
        connection,
        params=[
            repeated_space_label,
            collapsed_space_label,
        ],
    )
finally:
    connection.close()

spacing_variant_summary = (
    repeated_space_rows.groupby(
        "horse",
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        distinct_courses=("course", "nunique"),
        distinct_sires=("sire", "nunique"),
        distinct_dams=("dam", "nunique"),
        distinct_damsires=("damsire", "nunique"),
    )
    .reset_index()
)

print("Repeated-space horse-label exception confirmed")
print(f"Raw label: {repeated_space_label!r}")
print(f"Single-space candidate: {collapsed_space_label!r}")
display(spacing_variant_summary)
repeated_space_rows

Repeated-space horse-label exception confirmed
Raw label: 'Mon  Everest (FR)'
Single-space candidate: 'Mon Everest (FR)'


,horse,runner_rows,first_date,last_date,distinct_courses,distinct_sires,distinct_dams,distinct_damsires
0,Mon Everest (FR),2,2023-04-05,2023-10-05,2,1,1,1


,source_rowid,supplied_race_id,date,course,off,horse,sire,dam,damsire,sex,age
0,1321344,837676,2023-04-05,Chantilly (FR),4:03,Mon Everest (FR),Gengis (FR),Sara Francesca (FR),Le Fou,G,3
1,1409022,851442,2023-10-05,Saint-Cloud (FR),2:00,Mon Everest (FR),Gengis (FR),Sara Francesca (FR),Le Fou,G,3


## Stage 4 — Profile terminal parenthesised suffix structure

Every populated `horse` and `sire` label contains parentheses, while many `dam` labels and most `damsire` labels do not. This suggests a structured suffix convention, but its meaning has not yet been established.

This stage separates labels into:

- labels ending with one terminal parenthesised token;
- labels without a terminal parenthesised token;
- labels with other or potentially malformed parenthesis structures.

For syntactically clear cases, it extracts the terminal token only as a candidate embedded suffix. It does not treat that token as authoritative nationality, breeding country, registry identity or part of a natural key.

Raw labels remain unchanged.

In [7]:
# Profile terminal parenthesised tokens without assigning semantic meaning.
import re


TERMINAL_SUFFIX_PATTERN = re.compile(
    r"^(?P<display_name>.+?) \((?P<suffix>[^()]*)\)$"
)

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

suffix_structure_rows = []
suffix_vocabulary_frames = []
suffix_exception_frames = []

try:
    for field in HORSE_IDENTITY_FIELDS:
        field_labels = pd.read_sql_query(
            f"""
            SELECT
                "{field}" AS raw_label,
                COUNT(*) AS runner_rows
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND "{field}" IS NOT NULL
              AND "{field}" <> ''
            GROUP BY "{field}"
            ORDER BY "{field}"
            """,
            connection,
        )

        parsed = field_labels["raw_label"].str.extract(
            TERMINAL_SUFFIX_PATTERN
        )

        clear_terminal_suffix = (
            parsed["display_name"].notna()
            & parsed["display_name"].ne("")
            & parsed["suffix"].notna()
            & parsed["suffix"].ne("")
        )

        contains_any_parenthesis = field_labels[
            "raw_label"
        ].str.contains(
            r"[()]",
            regex=True,
            na=False,
        )

        malformed_or_other_parenthesis = (
            contains_any_parenthesis
            & ~clear_terminal_suffix
        )

        no_parentheses = ~contains_any_parenthesis

        suffix_structure_rows.append(
            {
                "source_field": field,
                "distinct_populated_raw_labels": len(field_labels),
                "labels_with_clear_terminal_suffix": int(
                    clear_terminal_suffix.sum()
                ),
                "labels_without_parentheses": int(
                    no_parentheses.sum()
                ),
                "labels_with_other_parenthesis_structure": int(
                    malformed_or_other_parenthesis.sum()
                ),
                "runner_rows_with_clear_terminal_suffix": int(
                    field_labels.loc[
                        clear_terminal_suffix,
                        "runner_rows",
                    ].sum()
                ),
            }
        )

        suffix_vocabulary = (
            field_labels.loc[
                clear_terminal_suffix,
                ["raw_label", "runner_rows"],
            ]
            .assign(
                suffix=parsed.loc[
                    clear_terminal_suffix,
                    "suffix",
                ].values
            )
            .groupby("suffix", as_index=False)
            .agg(
                distinct_labels=("raw_label", "nunique"),
                runner_rows=("runner_rows", "sum"),
            )
            .assign(source_field=field)
        )

        suffix_vocabulary_frames.append(suffix_vocabulary)

        suffix_exceptions = (
            field_labels.loc[
                malformed_or_other_parenthesis,
                ["raw_label", "runner_rows"],
            ]
            .assign(source_field=field)
        )

        suffix_exception_frames.append(suffix_exceptions)
finally:
    connection.close()

horse_identity_suffix_structure = pd.DataFrame(
    suffix_structure_rows
)

horse_identity_suffix_vocabulary = (
    pd.concat(
        suffix_vocabulary_frames,
        ignore_index=True,
    )
    .loc[
        :,
        [
            "source_field",
            "suffix",
            "distinct_labels",
            "runner_rows",
        ],
    ]
    .sort_values(
        ["source_field", "runner_rows", "suffix"],
        ascending=[True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

horse_identity_suffix_exceptions = (
    pd.concat(
        suffix_exception_frames,
        ignore_index=True,
    )
    .loc[
        :,
        [
            "source_field",
            "raw_label",
            "runner_rows",
        ],
    ]
    .sort_values(
        ["source_field", "raw_label"],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert (
    horse_identity_suffix_structure[
        [
            "labels_with_clear_terminal_suffix",
            "labels_without_parentheses",
            "labels_with_other_parenthesis_structure",
        ]
    ].sum(axis=1)
    == horse_identity_suffix_structure[
        "distinct_populated_raw_labels"
    ]
).all()

print("Terminal suffix structure profile confirmed")
display(horse_identity_suffix_structure)

print("Candidate suffix vocabulary")
display(horse_identity_suffix_vocabulary)

print("Other or malformed parenthesis structures")
horse_identity_suffix_exceptions

Terminal suffix structure profile confirmed


,source_field,distinct_populated_raw_labels,labels_with_clear_terminal_suffix,labels_without_parentheses,labels_with_other_parenthesis_structure,runner_rows_with_clear_terminal_suffix
0,horse,208631,208631,0,0,1851285
1,sire,5445,5445,0,0,1851285
2,dam,104972,90074,14898,0,1403308
3,damsire,6078,0,6078,0,0


Candidate suffix vocabulary


,source_field,suffix,distinct_labels,runner_rows
0,dam,IRE,26535,734982
1,dam,FR,13527,240755
2,dam,USA,19607,201127
3,dam,AUS,9283,79068
4,dam,GER,2237,42151
...,...,...,...,...
123,sire,KOR,2,7
124,sire,CZE,1,5
125,sire,VEN,2,3
126,sire,UKR,1,2


Other or malformed parenthesis structures


,source_field,raw_label,runner_rows


### Stage 4a — Candidate suffix-token syntax

The terminal parser found syntactically clear parenthesised tokens, but a clear structure does not prove that every token belongs to the same semantic vocabulary.

This stage profiles candidate suffix tokens by:

- character length;
- uppercase alphabetic form;
- presence of digits, spaces or punctuation;
- number of fields in which each token appears;
- distinct-label and runner-row frequency.

Tokens that do not resemble the dominant uppercase alphabetic pattern are retained as evidence and reviewed separately. No token is corrected, expanded or interpreted as nationality.

In [8]:
# Profile the exact candidate suffix vocabulary without interpreting it.
suffix_token_profile = (
    horse_identity_suffix_vocabulary
    .assign(
        token_length=lambda frame: frame["suffix"].str.len(),
        uppercase_alpha=lambda frame: frame["suffix"].str.fullmatch(
            r"[A-Z]+",
            na=False,
        ),
        contains_digit=lambda frame: frame["suffix"].str.contains(
            r"\d",
            regex=True,
            na=False,
        ),
        contains_space=lambda frame: frame["suffix"].str.contains(
            " ",
            regex=False,
            na=False,
        ),
        contains_punctuation=lambda frame: frame["suffix"].str.contains(
            r"[^A-Za-z0-9 ]",
            regex=True,
            na=False,
        ),
    )
)

suffix_syntax_summary = (
    suffix_token_profile
    .groupby(
        [
            "source_field",
            "token_length",
            "uppercase_alpha",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        distinct_suffix_tokens=("suffix", "nunique"),
        distinct_labels=("distinct_labels", "sum"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values(
        [
            "source_field",
            "runner_rows",
            "token_length",
        ],
        ascending=[True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

suffix_token_cross_field = (
    horse_identity_suffix_vocabulary
    .groupby("suffix", as_index=False)
    .agg(
        fields_present=("source_field", "nunique"),
        source_fields=(
            "source_field",
            lambda values: ", ".join(sorted(set(values))),
        ),
        distinct_labels=("distinct_labels", "sum"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values(
        ["runner_rows", "suffix"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

unusual_suffix_tokens = (
    suffix_token_profile.loc[
        ~suffix_token_profile["uppercase_alpha"]
        | suffix_token_profile["token_length"].ne(3)
        | suffix_token_profile["contains_digit"]
        | suffix_token_profile["contains_space"]
        | suffix_token_profile["contains_punctuation"],
        [
            "source_field",
            "suffix",
            "token_length",
            "uppercase_alpha",
            "contains_digit",
            "contains_space",
            "contains_punctuation",
            "distinct_labels",
            "runner_rows",
        ],
    ]
    .sort_values(
        ["source_field", "runner_rows", "suffix"],
        ascending=[True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

suffix_vocabulary_summary = pd.DataFrame(
    [
        {
            "source_field": field,
            "distinct_suffix_tokens": int(
                horse_identity_suffix_vocabulary.loc[
                    horse_identity_suffix_vocabulary[
                        "source_field"
                    ].eq(field),
                    "suffix",
                ].nunique()
            ),
            "three_letter_uppercase_tokens": int(
                suffix_token_profile.loc[
                    suffix_token_profile["source_field"].eq(field)
                    & suffix_token_profile["uppercase_alpha"]
                    & suffix_token_profile["token_length"].eq(3),
                    "suffix",
                ].nunique()
            ),
            "unusual_tokens": int(
                unusual_suffix_tokens.loc[
                    unusual_suffix_tokens["source_field"].eq(field),
                    "suffix",
                ].nunique()
            ),
        }
        for field in ["horse", "sire", "dam"]
    ]
)

print("Candidate suffix-token syntax confirmed")
display(suffix_vocabulary_summary)

print("Suffix syntax distribution")
display(suffix_syntax_summary)

print("Tokens shared across fields")
display(suffix_token_cross_field)

print("Unusual candidate suffix tokens")
unusual_suffix_tokens

Candidate suffix-token syntax confirmed


,source_field,distinct_suffix_tokens,three_letter_uppercase_tokens,unusual_tokens
0,horse,51,46,5
1,sire,29,25,4
2,dam,48,42,6


Suffix syntax distribution


,source_field,token_length,uppercase_alpha,distinct_suffix_tokens,distinct_labels,runner_rows
0,dam,3,True,42,68502,1108563
1,dam,2,True,6,21572,294745
2,horse,3,True,46,126736,1052511
3,horse,2,True,5,81895,798774
4,sire,3,True,25,4394,1169741
5,sire,2,True,4,1051,681544


Tokens shared across fields


,suffix,fields_present,source_fields,distinct_labels,runner_rows
0,IRE,3,"dam, horse, sire",98018,2196857
1,GB,3,"dam, horse, sire",47549,1045429
2,FR,3,"dam, horse, sire",50125,654314
3,USA,3,"dam, horse, sire",43615,575784
4,AUS,3,"dam, horse, sire",23955,290791
5,GER,3,"dam, horse, sire",5356,125519
6,NZ,3,"dam, horse, sire",6790,74834
7,JPN,3,"dam, horse, sire",9681,67292
8,ARG,3,"dam, horse, sire",4626,16088
9,SAF,3,"dam, horse, sire",4279,15369


Unusual candidate suffix tokens


,source_field,suffix,token_length,uppercase_alpha,contains_digit,contains_space,contains_punctuation,distinct_labels,runner_rows
0,dam,FR,2,True,False,False,False,13527,240755
1,dam,NZ,2,True,False,False,False,2906,30012
2,dam,GB,2,True,False,False,False,5125,23724
3,dam,GR,2,True,False,False,False,12,252
4,dam,PR,2,True,False,False,False,1,1
5,dam,QA,2,True,False,False,False,1,1
6,horse,GB,2,True,False,False,False,41810,491227
7,horse,FR,2,True,False,False,False,36245,275721
8,horse,NZ,2,True,False,False,False,3802,31603
9,horse,GR,2,True,False,False,False,23,177


### Stage 4b — Parsed-name collisions and suffix variation

The candidate suffix vocabulary is syntactically regular: every observed terminal token is an uppercase alphabetic string of two or three characters.

This stage tests whether parsed display names are unique within the source-label vocabulary.

It identifies:

- one parsed display name appearing with multiple suffixes;
- exact parsed display names shared between suffixed and unsuffixed `dam` labels;
- the frequency and source fields affected;
- representative source labels for each collision pattern.

These are source-label collisions only. A shared display name does not prove that the labels represent the same real horse, and different suffixes do not prove that they represent different horses.

No labels are merged.

In [9]:
# Test parsed display-name collisions without creating entity equivalence.
parsed_label_frames = []

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    for field in HORSE_IDENTITY_FIELDS:
        labels = pd.read_sql_query(
            f"""
            SELECT
                "{field}" AS raw_label,
                COUNT(*) AS runner_rows
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND "{field}" IS NOT NULL
              AND "{field}" <> ''
            GROUP BY "{field}"
            ORDER BY "{field}"
            """,
            connection,
        )

        parsed = labels["raw_label"].str.extract(
            TERMINAL_SUFFIX_PATTERN
        )

        has_terminal_suffix = (
            parsed["display_name"].notna()
            & parsed["display_name"].ne("")
            & parsed["suffix"].notna()
            & parsed["suffix"].ne("")
        )

        labels["source_field"] = field
        labels["has_terminal_suffix"] = has_terminal_suffix
        labels["parsed_display_name"] = labels["raw_label"]

        labels.loc[
            has_terminal_suffix,
            "parsed_display_name",
        ] = parsed.loc[
            has_terminal_suffix,
            "display_name",
        ]

        labels["candidate_suffix"] = pd.NA
        labels.loc[
            has_terminal_suffix,
            "candidate_suffix",
        ] = parsed.loc[
            has_terminal_suffix,
            "suffix",
        ]

        parsed_label_frames.append(labels)
finally:
    connection.close()

horse_identity_parsed_labels = pd.concat(
    parsed_label_frames,
    ignore_index=True,
)

multi_suffix_names = (
    horse_identity_parsed_labels.loc[
        horse_identity_parsed_labels["has_terminal_suffix"]
    ]
    .groupby(
        ["source_field", "parsed_display_name"],
        as_index=False,
    )
    .agg(
        distinct_suffixes=("candidate_suffix", "nunique"),
        suffixes=(
            "candidate_suffix",
            lambda values: ", ".join(sorted(set(values))),
        ),
        distinct_raw_labels=("raw_label", "nunique"),
        runner_rows=("runner_rows", "sum"),
        raw_label_examples=(
            "raw_label",
            lambda values: " | ".join(
                sorted(set(values))[:6]
            ),
        ),
    )
    .loc[lambda frame: frame["distinct_suffixes"] > 1]
    .sort_values(
        [
            "source_field",
            "distinct_suffixes",
            "runner_rows",
            "parsed_display_name",
        ],
        ascending=[True, False, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

dam_suffix_status = (
    horse_identity_parsed_labels.loc[
        horse_identity_parsed_labels[
            "source_field"
        ].eq("dam")
    ]
    .groupby(
        "parsed_display_name",
        as_index=False,
    )
    .agg(
        suffix_statuses=("has_terminal_suffix", "nunique"),
        has_suffixed_label=("has_terminal_suffix", "max"),
        has_unsuffixed_label=(
            "has_terminal_suffix",
            lambda values: (~values).any(),
        ),
        distinct_raw_labels=("raw_label", "nunique"),
        runner_rows=("runner_rows", "sum"),
        raw_label_examples=(
            "raw_label",
            lambda values: " | ".join(
                sorted(set(values))[:6]
            ),
        ),
    )
)

dam_with_and_without_suffix = (
    dam_suffix_status.loc[
        dam_suffix_status["has_suffixed_label"]
        & dam_suffix_status["has_unsuffixed_label"]
    ]
    .sort_values(
        [
            "runner_rows",
            "parsed_display_name",
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

parsed_name_collision_summary = pd.DataFrame(
    [
        {
            "collision_pattern": "one parsed name with multiple suffixes",
            "source_field": field,
            "parsed_display_names": int(
                multi_suffix_names.loc[
                    multi_suffix_names[
                        "source_field"
                    ].eq(field)
                ].shape[0]
            ),
        }
        for field in ["horse", "sire", "dam"]
    ]
    + [
        {
            "collision_pattern": (
                "dam name present with and without suffix"
            ),
            "source_field": "dam",
            "parsed_display_names": int(
                len(dam_with_and_without_suffix)
            ),
        }
    ]
)

print("Parsed-name collision profile confirmed")
display(parsed_name_collision_summary)

print("Parsed names associated with multiple suffixes")
display(multi_suffix_names)

print("Dam names appearing with and without a suffix")
dam_with_and_without_suffix

Parsed-name collision profile confirmed


,collision_pattern,source_field,parsed_display_names
0,one parsed name with multiple suffixes,horse,7635
1,one parsed name with multiple suffixes,sire,42
2,one parsed name with multiple suffixes,dam,2512
3,dam name present with and without suffix,dam,0


Parsed names associated with multiple suffixes


,source_field,parsed_display_name,distinct_suffixes,suffixes,distinct_raw_labels,runner_rows,raw_label_examples
0,dam,Notre Dame,5,"AUS, BRZ, GER, IRE, SWE",5,44,Notre Dame (AUS) | Notre Dame (BRZ) | Notre Da...
1,dam,Surprise,4,"FR, GER, IRE, NZ",4,149,Surprise (FR) | Surprise (GER) | Surprise (IRE...
2,dam,Clarinda,4,"AUS, FR, IRE, USA",4,123,Clarinda (AUS) | Clarinda (FR) | Clarinda (IRE...
3,dam,Lucida,4,"AUS, IRE, NZ, USA",4,103,Lucida (AUS) | Lucida (IRE) | Lucida (NZ) | Lu...
4,dam,Ismene,4,"FR, GER, ITY, USA",4,64,Ismene (FR) | Ismene (GER) | Ismene (ITY) | Is...
...,...,...,...,...,...,...,...
10184,sire,Lion Tamer,2,"SAF, USA",2,6,Lion Tamer (SAF) | Lion Tamer (USA)
10185,sire,Sporting,2,"GB, USA",2,3,Sporting (GB) | Sporting (USA)
10186,sire,Stanford,2,"GB, USA",2,3,Stanford (GB) | Stanford (USA)
10187,sire,Trajectory,2,"GB, USA",2,3,Trajectory (GB) | Trajectory (USA)


Dam names appearing with and without a suffix


,parsed_display_name,suffix_statuses,has_suffixed_label,has_unsuffixed_label,distinct_raw_labels,runner_rows,raw_label_examples


### Stage 4b — Parsed-name collisions and suffix variation

The candidate suffix vocabulary is syntactically regular: every observed terminal token is an uppercase alphabetic string of two or three characters.

This stage tests whether parsed display names are unique within the source-label vocabulary.

It identifies:

- one parsed display name appearing with multiple suffixes;
- exact parsed display names shared between suffixed and unsuffixed `dam` labels;
- the frequency and source fields affected;
- representative source labels for each collision pattern.

These are source-label collisions only. A shared display name does not prove that the labels represent the same real horse, and different suffixes do not prove that they represent different horses.

No labels are merged.

In [10]:
# Test parsed display-name collisions without creating entity equivalence.
parsed_label_frames = []

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    for field in HORSE_IDENTITY_FIELDS:
        labels = pd.read_sql_query(
            f"""
            SELECT
                "{field}" AS raw_label,
                COUNT(*) AS runner_rows
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND "{field}" IS NOT NULL
              AND "{field}" <> ''
            GROUP BY "{field}"
            ORDER BY "{field}"
            """,
            connection,
        )

        parsed = labels["raw_label"].str.extract(
            TERMINAL_SUFFIX_PATTERN
        )

        has_terminal_suffix = (
            parsed["display_name"].notna()
            & parsed["display_name"].ne("")
            & parsed["suffix"].notna()
            & parsed["suffix"].ne("")
        )

        labels["source_field"] = field
        labels["has_terminal_suffix"] = has_terminal_suffix
        labels["parsed_display_name"] = labels["raw_label"]

        labels.loc[
            has_terminal_suffix,
            "parsed_display_name",
        ] = parsed.loc[
            has_terminal_suffix,
            "display_name",
        ]

        labels["candidate_suffix"] = pd.NA
        labels.loc[
            has_terminal_suffix,
            "candidate_suffix",
        ] = parsed.loc[
            has_terminal_suffix,
            "suffix",
        ]

        parsed_label_frames.append(labels)
finally:
    connection.close()

horse_identity_parsed_labels = pd.concat(
    parsed_label_frames,
    ignore_index=True,
)

multi_suffix_names = (
    horse_identity_parsed_labels.loc[
        horse_identity_parsed_labels["has_terminal_suffix"]
    ]
    .groupby(
        ["source_field", "parsed_display_name"],
        as_index=False,
    )
    .agg(
        distinct_suffixes=("candidate_suffix", "nunique"),
        suffixes=(
            "candidate_suffix",
            lambda values: ", ".join(sorted(set(values))),
        ),
        distinct_raw_labels=("raw_label", "nunique"),
        runner_rows=("runner_rows", "sum"),
        raw_label_examples=(
            "raw_label",
            lambda values: " | ".join(
                sorted(set(values))[:6]
            ),
        ),
    )
    .loc[lambda frame: frame["distinct_suffixes"] > 1]
    .sort_values(
        [
            "source_field",
            "distinct_suffixes",
            "runner_rows",
            "parsed_display_name",
        ],
        ascending=[True, False, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

dam_suffix_status = (
    horse_identity_parsed_labels.loc[
        horse_identity_parsed_labels[
            "source_field"
        ].eq("dam")
    ]
    .groupby(
        "parsed_display_name",
        as_index=False,
    )
    .agg(
        suffix_statuses=("has_terminal_suffix", "nunique"),
        has_suffixed_label=("has_terminal_suffix", "max"),
        has_unsuffixed_label=(
            "has_terminal_suffix",
            lambda values: (~values).any(),
        ),
        distinct_raw_labels=("raw_label", "nunique"),
        runner_rows=("runner_rows", "sum"),
        raw_label_examples=(
            "raw_label",
            lambda values: " | ".join(
                sorted(set(values))[:6]
            ),
        ),
    )
)

dam_with_and_without_suffix = (
    dam_suffix_status.loc[
        dam_suffix_status["has_suffixed_label"]
        & dam_suffix_status["has_unsuffixed_label"]
    ]
    .sort_values(
        [
            "runner_rows",
            "parsed_display_name",
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

parsed_name_collision_summary = pd.DataFrame(
    [
        {
            "collision_pattern": "one parsed name with multiple suffixes",
            "source_field": field,
            "parsed_display_names": int(
                multi_suffix_names.loc[
                    multi_suffix_names[
                        "source_field"
                    ].eq(field)
                ].shape[0]
            ),
        }
        for field in ["horse", "sire", "dam"]
    ]
    + [
        {
            "collision_pattern": (
                "dam name present with and without suffix"
            ),
            "source_field": "dam",
            "parsed_display_names": int(
                len(dam_with_and_without_suffix)
            ),
        }
    ]
)

print("Parsed-name collision profile confirmed")
display(parsed_name_collision_summary)

print("Parsed names associated with multiple suffixes")
display(multi_suffix_names)

print("Dam names appearing with and without a suffix")
dam_with_and_without_suffix

Parsed-name collision profile confirmed


,collision_pattern,source_field,parsed_display_names
0,one parsed name with multiple suffixes,horse,7635
1,one parsed name with multiple suffixes,sire,42
2,one parsed name with multiple suffixes,dam,2512
3,dam name present with and without suffix,dam,0


Parsed names associated with multiple suffixes


,source_field,parsed_display_name,distinct_suffixes,suffixes,distinct_raw_labels,runner_rows,raw_label_examples
0,dam,Notre Dame,5,"AUS, BRZ, GER, IRE, SWE",5,44,Notre Dame (AUS) | Notre Dame (BRZ) | Notre Da...
1,dam,Surprise,4,"FR, GER, IRE, NZ",4,149,Surprise (FR) | Surprise (GER) | Surprise (IRE...
2,dam,Clarinda,4,"AUS, FR, IRE, USA",4,123,Clarinda (AUS) | Clarinda (FR) | Clarinda (IRE...
3,dam,Lucida,4,"AUS, IRE, NZ, USA",4,103,Lucida (AUS) | Lucida (IRE) | Lucida (NZ) | Lu...
4,dam,Ismene,4,"FR, GER, ITY, USA",4,64,Ismene (FR) | Ismene (GER) | Ismene (ITY) | Is...
...,...,...,...,...,...,...,...
10184,sire,Lion Tamer,2,"SAF, USA",2,6,Lion Tamer (SAF) | Lion Tamer (USA)
10185,sire,Sporting,2,"GB, USA",2,3,Sporting (GB) | Sporting (USA)
10186,sire,Stanford,2,"GB, USA",2,3,Stanford (GB) | Stanford (USA)
10187,sire,Trajectory,2,"GB, USA",2,3,Trajectory (GB) | Trajectory (USA)


Dam names appearing with and without a suffix


,parsed_display_name,suffix_statuses,has_suffixed_label,has_unsuffixed_label,distinct_raw_labels,runner_rows,raw_label_examples


## Stage 5 — Repeated horse-label pedigree stability

Parsed display names collide extensively across suffixes, so this stage returns to the complete raw `horse` label.

For every exact horse label appearing more than once, it tests whether the source consistently assigns:

- one sire label;
- one dam label;
- one damsire label;
- one sex value.

Missing pedigree values remain distinct from contradictory populated values.

An exact horse label associated with multiple pedigree combinations is a source-level contradiction or collision candidate. It does not by itself establish whether the cause is:

- two real horses sharing one source label;
- inconsistent extraction;
- corrected pedigree information;
- source contamination;
- another unresolved identity problem.

No entity merging or correction is performed.

In [11]:
# Test pedigree stability for repeated exact raw horse labels.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    repeated_horse_stability = pd.read_sql_query(
        f"""
        SELECT
            horse,
            COUNT(*) AS runner_rows,
            COUNT(DISTINCT date || '|' || course || '|' || off)
                AS provisional_races,
            MIN(date) AS first_date,
            MAX(date) AS last_date,

            COUNT(DISTINCT sire) AS distinct_sires,

            COUNT(
                DISTINCT CASE
                    WHEN dam <> '' THEN dam
                END
            ) AS distinct_populated_dams,

            SUM(CASE WHEN dam = '' THEN 1 ELSE 0 END)
                AS empty_dam_rows,

            COUNT(
                DISTINCT CASE
                    WHEN damsire <> '' THEN damsire
                END
            ) AS distinct_populated_damsires,

            SUM(CASE WHEN damsire = '' THEN 1 ELSE 0 END)
                AS empty_damsire_rows,

            COUNT(DISTINCT sex) AS distinct_sexes
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY horse
        HAVING COUNT(*) > 1
        ORDER BY
            runner_rows DESC,
            horse
        """,
        connection,
    )
finally:
    connection.close()

repeated_horse_stability["contradictory_sire"] = (
    repeated_horse_stability["distinct_sires"] > 1
)

repeated_horse_stability["contradictory_dam"] = (
    repeated_horse_stability["distinct_populated_dams"] > 1
)

repeated_horse_stability["contradictory_damsire"] = (
    repeated_horse_stability["distinct_populated_damsires"] > 1
)

repeated_horse_stability["contradictory_sex"] = (
    repeated_horse_stability["distinct_sexes"] > 1
)

repeated_horse_stability["missing_dam_only"] = (
    repeated_horse_stability["empty_dam_rows"] > 0
) & ~repeated_horse_stability["contradictory_dam"]

repeated_horse_stability["missing_damsire_only"] = (
    repeated_horse_stability["empty_damsire_rows"] > 0
) & ~repeated_horse_stability["contradictory_damsire"]

repeated_horse_stability["any_populated_pedigree_contradiction"] = (
    repeated_horse_stability[
        [
            "contradictory_sire",
            "contradictory_dam",
            "contradictory_damsire",
        ]
    ].any(axis=1)
)

repeated_horse_stability_summary = pd.DataFrame(
    [
        {
            "measure": "repeated exact horse labels",
            "horse_labels": len(repeated_horse_stability),
        },
        {
            "measure": "multiple populated sire labels",
            "horse_labels": int(
                repeated_horse_stability[
                    "contradictory_sire"
                ].sum()
            ),
        },
        {
            "measure": "multiple populated dam labels",
            "horse_labels": int(
                repeated_horse_stability[
                    "contradictory_dam"
                ].sum()
            ),
        },
        {
            "measure": "multiple populated damsire labels",
            "horse_labels": int(
                repeated_horse_stability[
                    "contradictory_damsire"
                ].sum()
            ),
        },
        {
            "measure": "any populated pedigree contradiction",
            "horse_labels": int(
                repeated_horse_stability[
                    "any_populated_pedigree_contradiction"
                ].sum()
            ),
        },
        {
            "measure": "multiple sex values",
            "horse_labels": int(
                repeated_horse_stability[
                    "contradictory_sex"
                ].sum()
            ),
        },
        {
            "measure": "empty dam but no populated dam contradiction",
            "horse_labels": int(
                repeated_horse_stability[
                    "missing_dam_only"
                ].sum()
            ),
        },
        {
            "measure": (
                "empty damsire but no populated damsire contradiction"
            ),
            "horse_labels": int(
                repeated_horse_stability[
                    "missing_damsire_only"
                ].sum()
            ),
        },
    ]
)

repeated_horse_contradictions = (
    repeated_horse_stability.loc[
        repeated_horse_stability[
            "any_populated_pedigree_contradiction"
        ]
        | repeated_horse_stability["contradictory_sex"]
        | repeated_horse_stability["empty_dam_rows"].gt(0)
        | repeated_horse_stability["empty_damsire_rows"].gt(0)
    ]
    .sort_values(
        [
            "any_populated_pedigree_contradiction",
            "contradictory_sex",
            "runner_rows",
            "horse",
        ],
        ascending=[False, False, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Repeated horse-label stability profile confirmed")
display(repeated_horse_stability_summary)

print("Horse labels requiring closer inspection")
repeated_horse_contradictions

Repeated horse-label stability profile confirmed


,measure,horse_labels
0,repeated exact horse labels,164237
1,multiple populated sire labels,269
2,multiple populated dam labels,5515
3,multiple populated damsire labels,316
4,any populated pedigree contradiction,5573
5,multiple sex values,37400
6,empty dam but no populated dam contradiction,1
7,empty damsire but no populated damsire contrad...,5


Horse labels requiring closer inspection


,horse,runner_rows,provisional_races,first_date,last_date,distinct_sires,distinct_populated_dams,empty_dam_rows,distinct_populated_damsires,empty_damsire_rows,distinct_sexes,contradictory_sire,contradictory_dam,contradictory_damsire,contradictory_sex,missing_dam_only,missing_damsire_only,any_populated_pedigree_contradiction
0,Copper Knight (IRE),118,118,2016-04-13,2026-05-14,1,2,0,1,0,2,False,True,False,True,False,False,True
1,Aberama Gold (GB),115,115,2019-06-19,2026-05-17,1,2,0,1,0,2,False,True,False,True,False,False,True
2,Itsalonglongroad (GB),113,113,2016-09-05,2026-05-03,1,2,0,1,0,2,False,True,False,True,False,False,True
3,Port Noir (GB),110,110,2019-04-15,2026-05-23,1,2,0,1,0,2,False,True,False,True,False,False,True
4,Under Curfew (GB),107,107,2018-07-31,2026-05-12,1,2,0,1,0,2,False,True,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40593,Ziso (IRE),2,2,2025-11-17,2026-02-03,1,1,0,1,0,2,False,False,False,True,False,False,False
40594,Zubi Zubi Zu (USA),2,2,2015-04-09,2016-05-28,1,1,0,1,0,2,False,False,False,True,False,False,False
40595,Kura Carmen (JPN),3,3,2016-04-13,2017-04-12,1,1,0,0,3,1,False,False,False,False,False,True,False
40596,Paraiba Tourmaline (USA),2,2,2023-08-17,2024-01-28,1,1,0,1,1,1,False,False,False,False,False,True,False


### Stage 5a — Structure of contradictory pedigree assertions

The source contains 5,573 repeated exact horse labels associated with more than one populated pedigree value.

This is too frequent to classify automatically as isolated extraction error. The same exact source label may represent:

- multiple real horses sharing the same displayed name and suffix;
- one horse with a corrected or inconsistent pedigree;
- contamination between runner records;
- another unresolved source-identity failure.

This stage profiles the complete pedigree combinations attached to each contradictory horse label.

It records:

- sire, dam and damsire combinations;
- sex values;
- first and last dates;
- runner-row and provisional-race counts;
- age range;
- course context.

The purpose is to determine whether competing assertions form coherent groups. No combination is selected as correct and no source rows are merged.

In [12]:
# Profile complete assertion groups for contradictory exact horse labels.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    contradictory_assertion_rows = pd.read_sql_query(
        f"""
        WITH contradictory_horses AS (
            SELECT horse
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY horse
            HAVING
                COUNT(DISTINCT sire) > 1
                OR COUNT(
                    DISTINCT CASE
                        WHEN dam <> '' THEN dam
                    END
                ) > 1
                OR COUNT(
                    DISTINCT CASE
                        WHEN damsire <> '' THEN damsire
                    END
                ) > 1
        )
        SELECT
            source.rowid AS source_rowid,
            source.race_id AS supplied_race_id,
            source.date,
            source.course,
            source.off,
            source.horse,
            source.sire,
            source.dam,
            source.damsire,
            source.sex,
            source.age
        FROM {SOURCE_TABLE} AS source
        INNER JOIN contradictory_horses
            ON source.horse = contradictory_horses.horse
        WHERE {DATA_ROW_PREDICATE}
        ORDER BY
            source.horse,
            source.date,
            source.course,
            source.off,
            source.rowid
        """,
        connection,
    )
finally:
    connection.close()

assert (
    contradictory_assertion_rows["horse"].nunique()
    == 5_573
)

contradictory_assertion_rows["provisional_race_key"] = (
    contradictory_assertion_rows[
        ["date", "course", "off"]
    ]
    .astype("string")
    .agg("|".join, axis=1)
)

contradictory_assertion_groups = (
    contradictory_assertion_rows
    .groupby(
        [
            "horse",
            "sire",
            "dam",
            "damsire",
            "sex",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=("provisional_race_key", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_age=("age", "min"),
        maximum_age=("age", "max"),
        distinct_courses=("course", "nunique"),
        course_examples=(
            "course",
            lambda values: " | ".join(
                sorted(set(values))[:5]
            ),
        ),
    )
)

assert (
    contradictory_assertion_groups["horse"].nunique()
    == 5_573
)

assert (
    contradictory_assertion_groups["provisional_races"]
    <= contradictory_assertion_groups["runner_rows"]
).all()

assertion_group_counts = (
    contradictory_assertion_groups
    .groupby("horse", as_index=False)
    .agg(
        assertion_groups=("horse", "size"),
        largest_group_rows=("runner_rows", "max"),
        total_runner_rows=("runner_rows", "sum"),
        total_provisional_races=("provisional_races", "sum"),
        first_date=("first_date", "min"),
        last_date=("last_date", "max"),
    )
)

assertion_group_counts["largest_group_share"] = (
    assertion_group_counts["largest_group_rows"]
    / assertion_group_counts["total_runner_rows"]
)

contradiction_structure_summary = pd.DataFrame(
    [
        {
            "measure": "contradictory exact horse labels",
            "horse_labels": int(
                assertion_group_counts["horse"].nunique()
            ),
        },
        {
            "measure": "exactly two assertion groups",
            "horse_labels": int(
                assertion_group_counts[
                    "assertion_groups"
                ].eq(2).sum()
            ),
        },
        {
            "measure": "three or more assertion groups",
            "horse_labels": int(
                assertion_group_counts[
                    "assertion_groups"
                ].ge(3).sum()
            ),
        },
        {
            "measure": "largest group is at least 90% of rows",
            "horse_labels": int(
                assertion_group_counts[
                    "largest_group_share"
                ].ge(0.90).sum()
            ),
        },
        {
            "measure": "largest group is below 60% of rows",
            "horse_labels": int(
                assertion_group_counts[
                    "largest_group_share"
                ].lt(0.60).sum()
            ),
        },
        {
            "measure": "contradiction occurs on only two runner rows",
            "horse_labels": int(
                assertion_group_counts[
                    "total_runner_rows"
                ].eq(2).sum()
            ),
        },
    ]
)

assertion_group_preview = (
    contradictory_assertion_groups
    .merge(
        assertion_group_counts[
            [
                "horse",
                "assertion_groups",
                "total_runner_rows",
                "total_provisional_races",
                "largest_group_share",
            ]
        ],
        on="horse",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        [
            "total_runner_rows",
            "horse",
            "runner_rows",
            "first_date",
        ],
        ascending=[False, True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Contradictory pedigree assertion structure confirmed")
display(contradiction_structure_summary)

print("Contradictory assertion groups")
assertion_group_preview

Contradictory pedigree assertion structure confirmed


,measure,horse_labels
0,contradictory exact horse labels,5573
1,exactly two assertion groups,3640
2,three or more assertion groups,1933
3,largest group is at least 90% of rows,449
4,largest group is below 60% of rows,1615
5,contradiction occurs on only two runner rows,127


Contradictory assertion groups


,horse,sire,dam,damsire,sex,runner_rows,provisional_races,first_date,last_date,minimum_age,maximum_age,distinct_courses,course_examples,assertion_groups,total_runner_rows,total_provisional_races,largest_group_share
0,Bobby Joe Leg (GB),Pastoral Pursuits (GB),China Cherub GB,Inchinor,G,111,111,2017-11-24,2025-05-26,3,11,8,Ayr | Doncaster | Newcastle (AW) | Pontefract ...,2,122,122,0.909836
1,Bobby Joe Leg (GB),Pastoral Pursuits (GB),China Cherub (GB),Inchinor,G,11,11,2025-12-02,2026-04-20,11,12,2,Newcastle (AW) | Southwell (AW),2,122,122,0.909836
2,Fact Or Fable (IRE),Alhebayeb (IRE),Unreal GB,Dansili,G,117,117,2019-04-08,2025-09-30,2,8,20,Bath | Brighton | Catterick | Chelmsford (AW) ...,2,121,121,0.966942
3,Fact Or Fable (IRE),Alhebayeb (IRE),Unreal (GB),Dansili,G,4,4,2025-10-20,2026-04-29,8,9,1,Bath,2,121,121,0.966942
4,Copper Knight (IRE),Sir Prancealot (IRE),Mystic Dream GB,Oasis Dream,G,109,109,2017-04-29,2025-10-10,3,11,20,Ascot | Ayr | Beverley | Catterick | Chester,3,118,118,0.923729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13094,Yukon River (IRE),Camelot (GB),Clear Skies GB,Sea The Stars,C,1,1,2023-09-04,2023-09-04,3,3,1,Roscommon (IRE),2,2,2,0.500000
13095,Zaraki (IRE),Zarak (FR),Urbania GB,Sea The Stars,C,1,1,2025-08-31,2025-08-31,2,2,1,Longchamp (FR),2,2,2,0.500000
13096,Zaraki (IRE),Zarak (FR),Urbania (GB),Sea The Stars,C,1,1,2026-04-05,2026-04-05,3,3,1,Longchamp,2,2,2,0.500000
13097,Zarwalyah (FR),Victor Ludorum (GB),Extreme Green GB,Motivator,F,1,1,2025-08-14,2025-08-14,2,2,1,Deauville (FR),2,2,2,0.500000


### Stage 5b — Test bare and parenthesised dam-suffix equivalence

Many high-frequency contradictions differ only because an older dam label ends with a bare uppercase token while a later label encloses the same token in parentheses.

Examples include:

- `China Cherub GB` and `China Cherub (GB)`;
- `Unreal GB` and `Unreal (GB)`;
- `Urbania GB` and `Urbania (GB)`.

This stage tests that formatting hypothesis across the complete contradictory population.

A bare terminal token is recognised only when:

- it is separated from the preceding display name by one space;
- it exactly matches a token already observed in the governed parenthesised suffix vocabulary;
- the transformation can be reversed to the original raw label.

The resulting structured comparison is not an entity merge. It tests whether competing raw dam labels express the same parsed display name and candidate suffix under different source formats.

Sire and damsire labels remain unchanged.

In [13]:
# Test whether apparent dam contradictions are caused by suffix-format changes.
candidate_suffix_tokens = set(
    horse_identity_suffix_vocabulary["suffix"]
)

PARENTHESISED_SUFFIX_PATTERN = re.compile(
    r"^(?P<display_name>.+?) \((?P<suffix>[A-Z]{2,3})\)$"
)

BARE_SUFFIX_PATTERN = re.compile(
    r"^(?P<display_name>.+?) (?P<suffix>[A-Z]{2,3})$"
)


def parse_dam_label_structure(raw_label):
    """Parse only syntactically supported dam-label structures."""
    if raw_label == "":
        return {
            "dam_display_name": pd.NA,
            "dam_candidate_suffix": pd.NA,
            "dam_suffix_format": "blank",
            "dam_structured_key": ("blank", ""),
        }

    parenthesised_match = PARENTHESISED_SUFFIX_PATTERN.fullmatch(
        raw_label
    )

    if (
        parenthesised_match
        and parenthesised_match.group("suffix")
        in candidate_suffix_tokens
    ):
        display_name = parenthesised_match.group("display_name")
        suffix = parenthesised_match.group("suffix")

        return {
            "dam_display_name": display_name,
            "dam_candidate_suffix": suffix,
            "dam_suffix_format": "parenthesised",
            "dam_structured_key": (
                "parsed_suffix",
                display_name,
                suffix,
            ),
        }

    bare_match = BARE_SUFFIX_PATTERN.fullmatch(raw_label)

    if (
        bare_match
        and bare_match.group("suffix")
        in candidate_suffix_tokens
    ):
        display_name = bare_match.group("display_name")
        suffix = bare_match.group("suffix")

        return {
            "dam_display_name": display_name,
            "dam_candidate_suffix": suffix,
            "dam_suffix_format": "bare",
            "dam_structured_key": (
                "parsed_suffix",
                display_name,
                suffix,
            ),
        }

    return {
        "dam_display_name": raw_label,
        "dam_candidate_suffix": pd.NA,
        "dam_suffix_format": "unsuffixed",
        "dam_structured_key": (
            "unsuffixed_raw",
            raw_label,
        ),
    }


dam_structure = contradictory_assertion_rows["dam"].apply(
    parse_dam_label_structure
)

dam_structure_frame = pd.DataFrame(
    dam_structure.tolist(),
    index=contradictory_assertion_rows.index,
)

contradictory_assertion_rows_with_dam_structure = pd.concat(
    [
        contradictory_assertion_rows.copy(),
        dam_structure_frame,
    ],
    axis=1,
)

dam_format_vocabulary = (
    contradictory_assertion_rows_with_dam_structure
    .groupby(
        "dam_suffix_format",
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        distinct_raw_dam_labels=("dam", "nunique"),
        distinct_horse_labels=("horse", "nunique"),
    )
    .sort_values(
        ["runner_rows", "dam_suffix_format"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

horse_dam_comparison = (
    contradictory_assertion_rows_with_dam_structure
    .groupby("horse", as_index=False)
    .agg(
        distinct_raw_dams=("dam", "nunique"),
        distinct_structured_dams=(
            "dam_structured_key",
            "nunique",
        ),
        dam_suffix_formats=(
            "dam_suffix_format",
            lambda values: ", ".join(
                sorted(set(values))
            ),
        ),
        runner_rows=("source_rowid", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

horse_dam_comparison["raw_dam_contradiction"] = (
    horse_dam_comparison["distinct_raw_dams"] > 1
)

horse_dam_comparison[
    "structured_dam_contradiction"
] = (
    horse_dam_comparison["distinct_structured_dams"] > 1
)

horse_dam_comparison[
    "resolved_by_suffix_format_parsing"
] = (
    horse_dam_comparison["raw_dam_contradiction"]
    & ~horse_dam_comparison["structured_dam_contradiction"]
)

dam_format_resolution_summary = pd.DataFrame(
    [
        {
            "measure": "horse labels with raw dam contradiction",
            "horse_labels": int(
                horse_dam_comparison[
                    "raw_dam_contradiction"
                ].sum()
            ),
        },
        {
            "measure": (
                "raw dam contradiction resolved by suffix-format parsing"
            ),
            "horse_labels": int(
                horse_dam_comparison[
                    "resolved_by_suffix_format_parsing"
                ].sum()
            ),
        },
        {
            "measure": (
                "dam contradiction remaining after suffix-format parsing"
            ),
            "horse_labels": int(
                (
                    horse_dam_comparison[
                        "raw_dam_contradiction"
                    ]
                    & horse_dam_comparison[
                        "structured_dam_contradiction"
                    ]
                ).sum()
            ),
        },
    ]
)

remaining_structured_dam_contradictions = (
    horse_dam_comparison.loc[
        horse_dam_comparison[
            "structured_dam_contradiction"
        ]
    ]
    .sort_values(
        ["runner_rows", "horse"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert (
    dam_format_resolution_summary.loc[
        dam_format_resolution_summary["measure"].eq(
            "horse labels with raw dam contradiction"
        ),
        "horse_labels",
    ].iloc[0]
    == 5_515
)

print("Dam suffix-format equivalence profile confirmed")
display(dam_format_vocabulary)

print("Effect on apparent dam contradictions")
display(dam_format_resolution_summary)

print("Horse labels retaining a structured dam contradiction")
remaining_structured_dam_contradictions

Dam suffix-format equivalence profile confirmed


,dam_suffix_format,runner_rows,distinct_raw_dam_labels,distinct_horse_labels
0,bare,72818,4117,5298
1,parenthesised,23586,4606,5563


Effect on apparent dam contradictions


,measure,horse_labels
0,horse labels with raw dam contradiction,5515
1,raw dam contradiction resolved by suffix-forma...,5208
2,dam contradiction remaining after suffix-forma...,307


Horse labels retaining a structured dam contradiction


,horse,distinct_raw_dams,distinct_structured_dams,dam_suffix_formats,runner_rows,first_date,last_date,raw_dam_contradiction,structured_dam_contradiction,resolved_by_suffix_format_parsing
0,Hellavapace (GB),2,2,parenthesised,57,2020-08-13,2025-07-25,True,True,False
1,Sarabi (GB),2,2,"bare, parenthesised",55,2015-05-26,2026-01-29,True,True,False
2,Bring It On (IRE),2,2,"bare, parenthesised",53,2016-04-06,2025-08-17,True,True,False
3,Volcano (FR),2,2,parenthesised,53,2017-10-24,2026-04-07,True,True,False
4,Disclosure (GB),2,2,parenthesised,48,2015-01-19,2025-06-06,True,True,False
...,...,...,...,...,...,...,...,...,...,...
302,Pont Marie (FR),2,2,parenthesised,2,2015-01-10,2025-08-31,True,True,False
303,Prankster (GB),2,2,bare,2,2015-08-14,2025-04-16,True,True,False
304,Private Joke (FR),2,2,parenthesised,2,2015-09-10,2024-10-02,True,True,False
305,Steel (USA),2,2,parenthesised,2,2015-01-17,2026-04-04,True,True,False


### Stage 5c — Recalculate pedigree contradictions after reversible dam parsing

The raw comparison overstated pedigree instability because two source formats were used for many dam labels:

- bare suffix: `China Cherub GB`;
- parenthesised suffix: `China Cherub (GB)`.

A reversible parser reduced 5,515 raw dam contradictions to 307 structured dam contradictions.

This stage recalculates repeated-horse pedigree consistency using:

- exact raw sire labels;
- structured dam keys where suffix syntax is supported;
- exact raw damsire labels;
- missing values kept separate from competing populated assertions.

The structured dam key is used only to compare source assertions. It does not replace the raw label or establish verified real-world entity identity.

In [14]:
# Recalculate pedigree stability after accounting for the dam suffix-format change.
structured_pedigree_rows = (
    contradictory_assertion_rows_with_dam_structure.copy()
)

structured_horse_stability = (
    structured_pedigree_rows
    .groupby("horse", as_index=False)
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=("provisional_race_key", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        distinct_sires=("sire", "nunique"),
        distinct_structured_dams=("dam_structured_key", "nunique"),
        distinct_populated_damsires=(
            "damsire",
            lambda values: values.loc[values.ne("")].nunique(),
        ),
        empty_dam_rows=("dam", lambda values: values.eq("").sum()),
        empty_damsire_rows=(
            "damsire",
            lambda values: values.eq("").sum(),
        ),
    )
)

structured_horse_stability["contradictory_sire"] = (
    structured_horse_stability["distinct_sires"] > 1
)

structured_horse_stability["contradictory_structured_dam"] = (
    structured_horse_stability["distinct_structured_dams"] > 1
)

structured_horse_stability["contradictory_damsire"] = (
    structured_horse_stability[
        "distinct_populated_damsires"
    ] > 1
)

structured_horse_stability[
    "any_structured_pedigree_contradiction"
] = structured_horse_stability[
    [
        "contradictory_sire",
        "contradictory_structured_dam",
        "contradictory_damsire",
    ]
].any(axis=1)

structured_contradiction_summary = pd.DataFrame(
    [
        {
            "measure": "raw pedigree contradiction",
            "horse_labels": 5_573,
        },
        {
            "measure": "multiple exact sire labels",
            "horse_labels": int(
                structured_horse_stability[
                    "contradictory_sire"
                ].sum()
            ),
        },
        {
            "measure": "multiple structured dam assertions",
            "horse_labels": int(
                structured_horse_stability[
                    "contradictory_structured_dam"
                ].sum()
            ),
        },
        {
            "measure": "multiple populated damsire labels",
            "horse_labels": int(
                structured_horse_stability[
                    "contradictory_damsire"
                ].sum()
            ),
        },
        {
            "measure": (
                "any pedigree contradiction after dam-format parsing"
            ),
            "horse_labels": int(
                structured_horse_stability[
                    "any_structured_pedigree_contradiction"
                ].sum()
            ),
        },
        {
            "measure": (
                "raw contradiction removed by dam-format parsing"
            ),
            "horse_labels": int(
                5_573
                - structured_horse_stability[
                    "any_structured_pedigree_contradiction"
                ].sum()
            ),
        },
    ]
)

remaining_pedigree_contradictions = (
    structured_horse_stability.loc[
        structured_horse_stability[
            "any_structured_pedigree_contradiction"
        ]
    ]
    .sort_values(
        [
            "runner_rows",
            "horse",
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert (
    structured_horse_stability[
        "contradictory_structured_dam"
    ].sum()
    == 307
)

print("Structured pedigree contradiction profile confirmed")
display(structured_contradiction_summary)

print("Horse labels retaining a pedigree contradiction")
remaining_pedigree_contradictions

Structured pedigree contradiction profile confirmed


,measure,horse_labels
0,raw pedigree contradiction,5573
1,multiple exact sire labels,269
2,multiple structured dam assertions,307
3,multiple populated damsire labels,316
4,any pedigree contradiction after dam-format pa...,368
5,raw contradiction removed by dam-format parsing,5205


Horse labels retaining a pedigree contradiction


,horse,runner_rows,provisional_races,first_date,last_date,distinct_sires,distinct_structured_dams,distinct_populated_damsires,empty_dam_rows,empty_damsire_rows,contradictory_sire,contradictory_structured_dam,contradictory_damsire,any_structured_pedigree_contradiction
0,Central Park West (FR),78,78,2020-08-06,2026-05-19,1,1,2,0,0,False,False,True,True
1,Hellavapace (GB),57,57,2020-08-13,2025-07-25,1,2,1,0,0,False,True,False,True
2,Sarabi (GB),55,55,2015-05-26,2026-01-29,2,2,2,0,0,True,True,True,True
3,Bring It On (IRE),53,53,2016-04-06,2025-08-17,2,2,2,0,0,True,True,True,True
4,GDay Aussie (GB),53,53,2015-07-11,2025-10-18,1,1,2,0,0,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,Sandrin Royal (URU),2,2,2025-01-06,2026-01-06,1,1,2,0,0,False,False,True,True
364,Star Galan (ARG),2,2,2022-05-25,2023-06-24,1,1,2,0,0,False,False,True,True
365,Steel (USA),2,2,2015-01-17,2026-04-04,2,2,2,0,0,True,True,True,True
366,Taj Alaelyaa (KSA),2,2,2024-02-24,2025-02-22,1,1,2,0,0,False,False,True,True


### Stage 5d — Remaining contradiction types

After accounting for the reversible dam-suffix format change, 368 exact horse labels retain more than one populated pedigree assertion.

This stage classifies those labels by the affected pedigree components:

- sire only;
- structured dam only;
- damsire only;
- combinations of two components;
- all three components.

This classification remains source-internal. It does not determine whether a case represents a reused horse label, corrected pedigree information, extraction contamination or another identity failure.

In [15]:
# Classify the remaining contradictions by affected pedigree components.
remaining_pedigree_contradictions = (
    structured_horse_stability.loc[
        structured_horse_stability[
            "any_structured_pedigree_contradiction"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(remaining_pedigree_contradictions) == 368

def classify_pedigree_contradiction(row):
    affected_components = []

    if row["contradictory_sire"]:
        affected_components.append("sire")

    if row["contradictory_structured_dam"]:
        affected_components.append("dam")

    if row["contradictory_damsire"]:
        affected_components.append("damsire")

    return " + ".join(affected_components)


remaining_pedigree_contradictions[
    "contradiction_type"
] = remaining_pedigree_contradictions.apply(
    classify_pedigree_contradiction,
    axis=1,
)

contradiction_type_summary = (
    remaining_pedigree_contradictions
    .groupby(
        "contradiction_type",
        as_index=False,
    )
    .agg(
        horse_labels=("horse", "nunique"),
        runner_rows=("runner_rows", "sum"),
        minimum_runner_rows=("runner_rows", "min"),
        maximum_runner_rows=("runner_rows", "max"),
        earliest_first_date=("first_date", "min"),
        latest_last_date=("last_date", "max"),
    )
    .sort_values(
        [
            "horse_labels",
            "contradiction_type",
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

component_summary = pd.DataFrame(
    [
        {
            "component": "sire",
            "horse_labels": int(
                remaining_pedigree_contradictions[
                    "contradictory_sire"
                ].sum()
            ),
        },
        {
            "component": "structured dam",
            "horse_labels": int(
                remaining_pedigree_contradictions[
                    "contradictory_structured_dam"
                ].sum()
            ),
        },
        {
            "component": "damsire",
            "horse_labels": int(
                remaining_pedigree_contradictions[
                    "contradictory_damsire"
                ].sum()
            ),
        },
        {
            "component": "any component",
            "horse_labels": len(
                remaining_pedigree_contradictions
            ),
        },
    ]
)

assert (
    contradiction_type_summary["horse_labels"].sum()
    == len(remaining_pedigree_contradictions)
)

print("Remaining pedigree contradiction types confirmed")
display(component_summary)

print("Mutually exclusive contradiction combinations")
contradiction_type_summary

Remaining pedigree contradiction types confirmed


,component,horse_labels
0,sire,269
1,structured dam,307
2,damsire,316
3,any component,368


Mutually exclusive contradiction combinations


,contradiction_type,horse_labels,runner_rows,minimum_runner_rows,maximum_runner_rows,earliest_first_date,latest_last_date
0,sire + dam + damsire,259,3266,2,55,2015-01-01,2026-05-27
1,damsire,51,727,2,78,2015-01-21,2026-05-20
2,dam,42,820,3,57,2015-01-02,2026-05-19
3,sire,10,108,3,25,2015-03-28,2026-05-26
4,dam + damsire,6,53,2,24,2016-05-08,2026-05-14


### Stage 5e — Temporal structure of competing pedigree groups

Most remaining contradictions change all three pedigree components together. This is consistent with exact source labels being reused for different horses, but that interpretation requires further source-internal evidence.

This stage groups each affected horse label by its complete structured pedigree assertion:

- exact sire label;
- structured dam key;
- exact damsire label.

For each group it records:

- runner appearances;
- provisional races;
- first and last appearance dates;
- age range;
- sex vocabulary;
- raw dam-label examples.

It then tests whether the date ranges of competing pedigree groups overlap.

Non-overlapping, age-coherent groups are consistent with label reuse across different horses or generations. Overlapping groups may instead indicate simultaneous name collisions, source contamination or another unresolved identity problem.

The analysis remains provisional and does not create separate real-world entities.

In [16]:
# Test whether competing pedigree assertions form coherent temporal groups.
from itertools import combinations


remaining_contradictory_labels = set(
    remaining_pedigree_contradictions["horse"]
)

remaining_assertion_rows = (
    contradictory_assertion_rows_with_dam_structure.loc[
        contradictory_assertion_rows_with_dam_structure[
            "horse"
        ].isin(remaining_contradictory_labels)
    ]
    .copy()
)

assert remaining_assertion_rows["horse"].nunique() == 368

remaining_assertion_rows["date_parsed"] = pd.to_datetime(
    remaining_assertion_rows["date"],
    errors="raise",
)

structured_pedigree_groups = (
    remaining_assertion_rows
    .groupby(
        [
            "horse",
            "sire",
            "dam_structured_key",
            "damsire",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=("provisional_race_key", "nunique"),
        first_date=("date_parsed", "min"),
        last_date=("date_parsed", "max"),
        minimum_age=("age", "min"),
        maximum_age=("age", "max"),
        distinct_sexes=("sex", "nunique"),
        sex_values=(
            "sex",
            lambda values: ", ".join(
                sorted(set(values.astype(str)))
            ),
        ),
        raw_dam_labels=(
            "dam",
            lambda values: " | ".join(
                sorted(set(values))[:5]
            ),
        ),
        dam_suffix_formats=(
            "dam_suffix_format",
            lambda values: ", ".join(
                sorted(set(values))
            ),
        ),
        course_examples=(
            "course",
            lambda values: " | ".join(
                sorted(set(values))[:5]
            ),
        ),
    )
)

assert structured_pedigree_groups["horse"].nunique() == 368

temporal_structure_rows = []

for horse, groups in structured_pedigree_groups.groupby(
    "horse",
    sort=False,
):
    group_records = groups.to_dict("records")

    pair_count = 0
    overlapping_pairs = 0
    touching_or_overlapping_pairs = 0
    minimum_gap_days = None

    for left, right in combinations(group_records, 2):
        pair_count += 1

        left_start = left["first_date"]
        left_end = left["last_date"]
        right_start = right["first_date"]
        right_end = right["last_date"]

        strict_overlap = (
            left_start <= right_end
            and right_start <= left_end
        )

        if strict_overlap:
            overlapping_pairs += 1
            touching_or_overlapping_pairs += 1
            gap_days = 0
        else:
            earlier_end = min(left_end, right_end)
            later_start = max(left_start, right_start)
            gap_days = int(
                (later_start - earlier_end).days
            )

            if gap_days <= 1:
                touching_or_overlapping_pairs += 1

        if (
            minimum_gap_days is None
            or gap_days < minimum_gap_days
        ):
            minimum_gap_days = gap_days

    temporal_structure_rows.append(
        {
            "horse": horse,
            "pedigree_groups": len(groups),
            "group_pairs": pair_count,
            "overlapping_group_pairs": overlapping_pairs,
            "touching_or_overlapping_group_pairs": (
                touching_or_overlapping_pairs
            ),
            "minimum_gap_days": minimum_gap_days,
            "total_runner_rows": int(
                groups["runner_rows"].sum()
            ),
            "first_date": groups["first_date"].min(),
            "last_date": groups["last_date"].max(),
        }
    )

pedigree_temporal_structure = pd.DataFrame(
    temporal_structure_rows
)

pedigree_temporal_structure[
    "any_temporal_overlap"
] = (
    pedigree_temporal_structure[
        "overlapping_group_pairs"
    ] > 0
)

pedigree_temporal_structure[
    "all_groups_temporally_separate"
] = (
    pedigree_temporal_structure[
        "overlapping_group_pairs"
    ] == 0
)

pedigree_temporal_summary = pd.DataFrame(
    [
        {
            "measure": "horse labels with competing pedigree groups",
            "horse_labels": len(
                pedigree_temporal_structure
            ),
        },
        {
            "measure": "all pedigree groups temporally separate",
            "horse_labels": int(
                pedigree_temporal_structure[
                    "all_groups_temporally_separate"
                ].sum()
            ),
        },
        {
            "measure": "at least one overlapping group pair",
            "horse_labels": int(
                pedigree_temporal_structure[
                    "any_temporal_overlap"
                ].sum()
            ),
        },
        {
            "measure": "exactly two pedigree groups",
            "horse_labels": int(
                pedigree_temporal_structure[
                    "pedigree_groups"
                ].eq(2).sum()
            ),
        },
        {
            "measure": "three or more pedigree groups",
            "horse_labels": int(
                pedigree_temporal_structure[
                    "pedigree_groups"
                ].ge(3).sum()
            ),
        },
    ]
)

structured_pedigree_group_preview = (
    structured_pedigree_groups
    .merge(
        pedigree_temporal_structure[
            [
                "horse",
                "pedigree_groups",
                "overlapping_group_pairs",
                "minimum_gap_days",
                "all_groups_temporally_separate",
            ]
        ],
        on="horse",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        [
            "all_groups_temporally_separate",
            "horse",
            "first_date",
            "runner_rows",
        ],
        ascending=[True, True, True, False],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Temporal pedigree-group structure confirmed")
display(pedigree_temporal_summary)

print("Structured pedigree groups")
structured_pedigree_group_preview

Temporal pedigree-group structure confirmed


,measure,horse_labels
0,horse labels with competing pedigree groups,368
1,all pedigree groups temporally separate,350
2,at least one overlapping group pair,18
3,exactly two pedigree groups,363
4,three or more pedigree groups,5


Structured pedigree groups


,horse,sire,dam_structured_key,damsire,runner_rows,provisional_races,first_date,last_date,minimum_age,maximum_age,distinct_sexes,sex_values,raw_dam_labels,dam_suffix_formats,course_examples,pedigree_groups,overlapping_group_pairs,minimum_gap_days,all_groups_temporally_separate
0,Attention All (IRE),Westerner (GB),"(parsed_suffix, Moon Light Shadow I, GB)",Compton Place,7,7,2022-05-10,2024-02-03,4,6,1,G,Moon Light Shadow I GB,bare,Ayr | Carlisle | Newcastle | Sedgefield | Weth...,2,1,0,False
1,Attention All (IRE),Westerner (GB),"(parsed_suffix, Moonlight Shadow, GB)",Fountain Of Youth,1,1,2024-01-06,2024-01-06,6,6,1,G,Moonlight Shadow GB,bare,Newcastle,2,1,0,False
2,Calivigny (IRE),Gold Well (GB),"(parsed_suffix, Summer Holiday, IRE)",Kambalda,13,13,2015-01-02,2021-12-21,6,12,1,G,Summer Holiday (IRE),parenthesised,Ayr | Carlisle | Kelso | Musselburgh | Perth,2,1,0,False
3,Calivigny (IRE),Gold Well (GB),"(parsed_suffix, Summer Holiday I, IRE)",Kambalda,33,33,2015-01-28,2019-12-29,6,10,1,G,Summer Holiday I (IRE),parenthesised,Ayr | Carlisle | Hexham | Kelso | Musselburgh,2,1,0,False
4,Fr Humphrey (IRE),Carlo Bank (IRE),"(parsed_suffix, An Realt Beag, IRE)",AlmutawakelI,17,17,2015-01-21,2019-11-12,7,11,1,G,An Realt Beag (IRE),parenthesised,Fairyhouse (IRE) | Fontwell | Gowran Park (IRE...,3,1,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,Yorkshire (IRE),Harry Angel (IRE),"(parsed_suffix, Totsiyah, IRE)",Dalakhani,20,20,2022-08-07,2026-05-09,2,6,2,"C, G",Totsiyah (IRE),parenthesised,Ascot | Doncaster | Goodwood | Haydock | Kempt...,2,0,2444,True
737,Yukon River (IRE),Court Cave (IRE),"(parsed_suffix, Alaska River, IRE)",Whitmores Conn,1,1,2017-04-16,2017-04-16,5,5,1,M,Alaska River (IRE),parenthesised,Cork (IRE),2,0,2332,True
738,Yukon River (IRE),Camelot (GB),"(parsed_suffix, Clear Skies, GB)",Sea The Stars,1,1,2023-09-04,2023-09-04,3,3,1,C,Clear Skies GB,bare,Roscommon (IRE),2,0,2332,True
739,Zangar (FR),Dream Well (FR),"(parsed_suffix, Gallerys Native, FR)",Gallery Of Zurich,3,3,2015-11-25,2016-05-04,5,6,1,G,Gallerys Native (FR),parenthesised,Auteuil (FR) | Enghien (FR),2,0,3292,True


### Interim interpretation — Horse names are labels, not permanent identities

The temporal structure provides strong evidence that an exact source `horse` label is not a permanent identifier for one real-world horse.

Of the 368 exact horse labels associated with competing structured pedigrees:

* 350 have pedigree groups that are completely separate in time;
* 363 have exactly two pedigree groups;
* only 18 have any overlap between competing groups.

The non-overlapping cases commonly show a later appearance period with a different pedigree, reset age sequence and sometimes a different sex. Examples such as `Yukon River (IRE)` and `Zangar (FR)` are consistent with the same registered name being used for different horses in different periods.

This interpretation is compatible with international naming rules. Racing authorities generally prevent duplication within their own relevant registers rather than enforcing permanent global uniqueness. When required for international distinction, a bracketed suffix showing the country of foaling is added and forms part of the registered name. Non-protected names may later become available for reuse after authority-defined protection periods.

The source evidence therefore supports the following provisional governance conclusions:

* the complete raw `horse` label must be preserved;
* the terminal bracketed token may be parsed separately as an embedded country-of-foaling suffix where the syntax is clear;
* the suffix remains part of the source label and must not be discarded;
* the parsed display name alone is not unique;
* the complete `horse + suffix` label is not permanently unique across time;
* one exact source horse label may legitimately map to multiple coherent pedigree groups;
* an exact horse label must not be used as a permanent natural key for a verified real-world horse.

A reversible source-level identity must therefore distinguish at least:

* the exact raw horse label;
* the coherent pedigree assertion;
* appearance period and age chronology;
* sex evidence where internally consistent;
* provisional race and physical source lineage;
* resolution and review status.

Such a key would identify a **source-level horse occurrence candidate**, not a verified real-world entity. Authoritative identity would require an external authority identifier, stud-book record, passport or equivalent evidence.

The 350 temporally separate cases should not automatically be classified as pedigree errors. They are better treated as probable source-label reuse pending validation of the final grouping rule.

The remaining 18 temporally overlapping cases require separate inspection because ordinary time-separated name reuse does not explain them.


### Stage 5f — Inspect temporally overlapping pedigree groups

Most competing pedigree groups are temporally separate and are consistent with source-label reuse across different horses.

Eighteen exact horse labels have pedigree groups whose date ranges overlap. These cannot be explained by ordinary time-separated name reuse alone.

Possible explanations include:

- spelling or transcription variants within pedigree labels;
- source corrections applied inconsistently;
- simultaneous real horses sharing the same displayed label;
- pedigree contamination;
- another unresolved extraction defect.

This stage displays the complete structured assertion groups for those eighteen labels, including their dates, ages, sexes and raw pedigree labels.

No assertion is selected as correct.

In [17]:
# Inspect all structured pedigree groups for the 18 overlapping cases.
overlapping_horse_labels = (
    pedigree_temporal_structure.loc[
        pedigree_temporal_structure["any_temporal_overlap"],
        "horse",
    ]
    .sort_values()
    .tolist()
)

assert len(overlapping_horse_labels) == 18

overlapping_case_summary = (
    remaining_pedigree_contradictions.loc[
        remaining_pedigree_contradictions["horse"].isin(
            overlapping_horse_labels
        ),
        [
            "horse",
            "contradiction_type",
            "runner_rows",
            "provisional_races",
            "first_date",
            "last_date",
            "distinct_sires",
            "distinct_structured_dams",
            "distinct_populated_damsires",
        ],
    ]
    .merge(
        pedigree_temporal_structure[
            [
                "horse",
                "pedigree_groups",
                "overlapping_group_pairs",
                "minimum_gap_days",
            ]
        ],
        on="horse",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["runner_rows", "horse"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

overlapping_pedigree_groups = (
    structured_pedigree_groups.loc[
        structured_pedigree_groups["horse"].isin(
            overlapping_horse_labels
        )
    ]
    .merge(
        overlapping_case_summary[
            [
                "horse",
                "contradiction_type",
                "pedigree_groups",
                "overlapping_group_pairs",
            ]
        ],
        on="horse",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        [
            "horse",
            "first_date",
            "last_date",
            "runner_rows",
        ],
        ascending=[True, True, True, False],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert overlapping_case_summary["horse"].nunique() == 18
assert overlapping_pedigree_groups["horse"].nunique() == 18

print("Temporally overlapping pedigree cases confirmed")
display(overlapping_case_summary)

print("Structured groups for overlapping cases")
overlapping_pedigree_groups

Temporally overlapping pedigree cases confirmed


,horse,contradiction_type,runner_rows,provisional_races,first_date,last_date,distinct_sires,distinct_structured_dams,distinct_populated_damsires,pedigree_groups,overlapping_group_pairs,minimum_gap_days
0,Calivigny (IRE),dam,46,46,2015-01-02,2021-12-21,1,2,1,2,1,0
1,Wise Coco (GB),damsire,37,37,2017-11-11,2021-07-27,1,1,2,2,1,0
2,Love Chunghwa (IRE),damsire,34,34,2016-11-30,2019-07-10,1,1,2,2,1,0
3,Fr Humphrey (IRE),damsire,31,31,2015-01-21,2024-10-19,1,1,3,3,1,0
4,Shielas Well (IRE),dam,29,29,2024-06-14,2026-05-05,1,2,1,2,1,0
5,Turf Brilliant (AUS),dam,28,28,2019-09-08,2023-06-25,1,2,1,2,1,0
6,Ivilnoble (IRE),damsire,25,25,2018-12-29,2024-01-28,1,1,3,3,1,0
7,Mystic Miraaj (GB),damsire,21,21,2015-04-13,2016-10-15,1,1,2,2,1,0
8,Holly (FR),sire,15,15,2020-11-27,2024-12-14,2,1,1,2,1,0
9,Wedding Breakfast (IRE),damsire,12,12,2017-01-26,2020-09-22,1,1,2,2,1,0


Structured groups for overlapping cases


,horse,sire,dam_structured_key,damsire,runner_rows,provisional_races,first_date,last_date,minimum_age,maximum_age,distinct_sexes,sex_values,raw_dam_labels,dam_suffix_formats,course_examples,contradiction_type,pedigree_groups,overlapping_group_pairs
0,Attention All (IRE),Westerner (GB),"(parsed_suffix, Moon Light Shadow I, GB)",Compton Place,7,7,2022-05-10,2024-02-03,4,6,1,G,Moon Light Shadow I GB,bare,Ayr | Carlisle | Newcastle | Sedgefield | Weth...,dam + damsire,2,1
1,Attention All (IRE),Westerner (GB),"(parsed_suffix, Moonlight Shadow, GB)",Fountain Of Youth,1,1,2024-01-06,2024-01-06,6,6,1,G,Moonlight Shadow GB,bare,Newcastle,dam + damsire,2,1
2,Calivigny (IRE),Gold Well (GB),"(parsed_suffix, Summer Holiday, IRE)",Kambalda,13,13,2015-01-02,2021-12-21,6,12,1,G,Summer Holiday (IRE),parenthesised,Ayr | Carlisle | Kelso | Musselburgh | Perth,dam,2,1
3,Calivigny (IRE),Gold Well (GB),"(parsed_suffix, Summer Holiday I, IRE)",Kambalda,33,33,2015-01-28,2019-12-29,6,10,1,G,Summer Holiday I (IRE),parenthesised,Ayr | Carlisle | Hexham | Kelso | Musselburgh,dam,2,1
4,Fr Humphrey (IRE),Carlo Bank (IRE),"(parsed_suffix, An Realt Beag, IRE)",AlmutawakelI,17,17,2015-01-21,2019-11-12,7,11,1,G,An Realt Beag (IRE),parenthesised,Fairyhouse (IRE) | Fontwell | Gowran Park (IRE...,damsire,3,1
5,Fr Humphrey (IRE),Carlo Bank (IRE),"(parsed_suffix, An Realt Beag, IRE)",Almutawakel,10,10,2016-11-16,2020-12-03,8,12,1,G,An Realt Beag (IRE),parenthesised,Fairyhouse (IRE) | Huntingdon | Kelso | Kilbeg...,damsire,3,1
6,Fr Humphrey (IRE),Carlo Bank (IRE),"(parsed_suffix, An Realt Beag, IRE)",Almutawakel I,4,4,2023-05-15,2024-10-19,15,16,1,G,An Realt Beag (IRE),parenthesised,Kilbeggan (IRE) | Killarney (IRE) | Limerick (...,damsire,3,1
7,Full Drago (ITY),Pounced (USA),"(parsed_suffix, Almata, IRE)",AlmutawakelI,8,8,2015-10-24,2017-10-22,2,4,1,C,Almata (IRE),parenthesised,Capannelle (ITY) | Saint-Cloud (FR) | San Siro...,damsire,2,1
8,Full Drago (ITY),Pounced (USA),"(parsed_suffix, Almata, IRE)",Almutawakel,1,1,2017-06-18,2017-06-18,4,4,1,C,Almata (IRE),parenthesised,San Siro (ITY),damsire,2,1
9,Holly (FR),Voiladenuo (FR),"(parsed_suffix, Righty Malta, FR)",Turgeon,14,14,2020-11-27,2024-12-14,3,7,2,"F, M",Righty Malta (FR),parenthesised,Carlisle | Cheltenham | Exeter | Fakenham | Ha...,sire,2,1


### Stage 5g — Classify overlapping pedigree-label variants

The eighteen temporally overlapping cases do not generally resemble separate horses sharing one source label.

Most retain a coherent horse chronology while one pedigree component changes through a small textual variant. Recurrent examples include:

- `Almutawakel`, `AlmutawakelI` and `Almutawakel I`;
- `Voiladenuo (FR)` and `Ut*voiladenuo (FR)`;
- spacing, capitalisation or trailing-`I` variation in dam labels.

These patterns are consistent with source transcription or formatting variation, but source-internal similarity alone does not prove real-world equivalence.

This stage therefore classifies exact competing labels by reversible textual comparisons. It tests:

- case-only differences;
- whitespace-only differences;
- removal of a leading `Ut*` marker;
- attachment or separation of a terminal capital `I`;
- combinations of those transformations.

The original labels remain unchanged. A matched comparison creates only a provisional label-variant candidate and not a verified pedigree correction.

In [18]:
# Classify overlapping pedigree differences using bounded reversible comparisons.
def comparison_forms(raw_label):
    """Return bounded comparison forms without replacing the raw label."""
    forms = {
        "exact": raw_label,
        "casefold": raw_label.casefold(),
        "collapsed_whitespace": " ".join(raw_label.split()),
    }

    collapsed = forms["collapsed_whitespace"]

    if collapsed.startswith("Ut*"):
        forms["without_ut_prefix"] = collapsed[3:]
    else:
        forms["without_ut_prefix"] = collapsed

    without_ut = forms["without_ut_prefix"]

    # Compare attached and separated terminal capital-I forms.
    if without_ut.endswith(" I"):
        forms["terminal_i_normalised"] = without_ut[:-2] + "I"
    elif without_ut.endswith("I"):
        forms["terminal_i_normalised"] = without_ut[:-1] + " I"
    else:
        forms["terminal_i_normalised"] = without_ut

    forms["combined_comparison"] = (
        forms["terminal_i_normalised"]
        .replace(" ", "")
        .casefold()
    )

    return forms


overlap_rows = remaining_assertion_rows.loc[
    remaining_assertion_rows["horse"].isin(
        overlapping_horse_labels
    )
].copy()

comparison_records = []

for horse, horse_rows in overlap_rows.groupby("horse", sort=True):
    for component in ["sire", "dam", "damsire"]:
        raw_labels = sorted(
            label
            for label in horse_rows[component].dropna().unique()
            if label != ""
        )

        if len(raw_labels) <= 1:
            continue

        for left_label, right_label in combinations(raw_labels, 2):
            left_forms = comparison_forms(left_label)
            right_forms = comparison_forms(right_label)

            if left_forms["casefold"] == right_forms["casefold"]:
                comparison_type = "case_only"
            elif (
                left_forms["collapsed_whitespace"].casefold()
                == right_forms["collapsed_whitespace"].casefold()
            ):
                comparison_type = "whitespace_only"
            elif (
                left_forms["without_ut_prefix"].casefold()
                == right_forms["without_ut_prefix"].casefold()
            ):
                comparison_type = "ut_prefix_only"
            elif (
                left_forms["combined_comparison"]
                == right_forms["combined_comparison"]
            ):
                comparison_type = (
                    "bounded_spacing_prefix_or_terminal_i_variant"
                )
            else:
                comparison_type = "material_text_difference"

            comparison_records.append(
                {
                    "horse": horse,
                    "component": component,
                    "left_raw_label": left_label,
                    "right_raw_label": right_label,
                    "comparison_type": comparison_type,
                }
            )

overlapping_label_comparisons = (
    pd.DataFrame(comparison_records)
    .sort_values(
        ["comparison_type", "component", "horse"],
        kind="stable",
    )
    .reset_index(drop=True)
)

overlapping_variant_summary = (
    overlapping_label_comparisons
    .groupby(
        ["component", "comparison_type"],
        as_index=False,
    )
    .agg(
        label_pairs=("horse", "size"),
        horse_labels=("horse", "nunique"),
    )
    .sort_values(
        ["component", "label_pairs", "comparison_type"],
        ascending=[True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

material_overlap_cases = (
    overlapping_label_comparisons.loc[
        overlapping_label_comparisons[
            "comparison_type"
        ].eq("material_text_difference")
    ]
    .sort_values(
        ["component", "horse", "left_raw_label"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Overlapping pedigree-label comparisons confirmed")
display(overlapping_variant_summary)

print("All competing label comparisons")
display(overlapping_label_comparisons)

print("Material differences remaining after bounded comparison")
material_overlap_cases

Overlapping pedigree-label comparisons confirmed


,component,comparison_type,label_pairs,horse_labels
0,dam,material_text_difference,3,3
1,dam,case_only,1,1
2,damsire,material_text_difference,16,14
3,damsire,bounded_spacing_prefix_or_terminal_i_variant,2,2
4,sire,ut_prefix_only,1,1


All competing label comparisons


,horse,component,left_raw_label,right_raw_label,comparison_type
0,Fr Humphrey (IRE),damsire,Almutawakel I,AlmutawakelI,bounded_spacing_prefix_or_terminal_i_variant
1,Ivilnoble (IRE),damsire,Almutawakel I,AlmutawakelI,bounded_spacing_prefix_or_terminal_i_variant
2,Shielas Well (IRE),dam,AmcHitka (IRE),Amchitka (IRE),case_only
3,Attention All (IRE),dam,Moon Light Shadow I GB,Moonlight Shadow GB,material_text_difference
4,Calivigny (IRE),dam,Summer Holiday (IRE),Summer Holiday I (IRE),material_text_difference
5,Turf Brilliant (AUS),dam,Paradise Lost (IRE),Paradise Lost I (IRE),material_text_difference
6,Attention All (IRE),damsire,Compton Place,Fountain Of Youth,material_text_difference
7,Fr Humphrey (IRE),damsire,Almutawakel,Almutawakel I,material_text_difference
8,Fr Humphrey (IRE),damsire,Almutawakel,AlmutawakelI,material_text_difference
9,Full Drago (ITY),damsire,Almutawakel,AlmutawakelI,material_text_difference


Material differences remaining after bounded comparison


,horse,component,left_raw_label,right_raw_label,comparison_type
0,Attention All (IRE),dam,Moon Light Shadow I GB,Moonlight Shadow GB,material_text_difference
1,Calivigny (IRE),dam,Summer Holiday (IRE),Summer Holiday I (IRE),material_text_difference
2,Turf Brilliant (AUS),dam,Paradise Lost (IRE),Paradise Lost I (IRE),material_text_difference
3,Attention All (IRE),damsire,Compton Place,Fountain Of Youth,material_text_difference
4,Fr Humphrey (IRE),damsire,Almutawakel,Almutawakel I,material_text_difference
5,Fr Humphrey (IRE),damsire,Almutawakel,AlmutawakelI,material_text_difference
6,Full Drago (ITY),damsire,Almutawakel,AlmutawakelI,material_text_difference
7,Island Vision (IRE),damsire,Almutawakel,AlmutawakelI,material_text_difference
8,Ivilnoble (IRE),damsire,Almutawakel,Almutawakel I,material_text_difference
9,Ivilnoble (IRE),damsire,Almutawakel,AlmutawakelI,material_text_difference


### Stage 5h — Recurrence of unresolved pedigree-label pairs

The bounded comparison identified several safe textual relationships but left nineteen materially different label pairs unresolved.

Most unresolved damsire comparisons involve:

- `Almutawakel`;
- `AlmutawakelI`;
- `Almutawakel I`.

A recurring pair across many unrelated horse labels is more consistent with a systematic source-label variant than with independent pedigree changes. However, recurrence alone does not prove that the labels identify the same real stallion.

This stage counts each exact competing label pair across the overlapping cases and records:

- the affected pedigree component;
- number of horse labels;
- number of runner rows associated with each label;
- first and last source dates;
- representative affected horses.

No normalization rule is applied. The result determines whether a small number of bounded claims should be considered for manual or external verification.

In [19]:
# Quantify recurrence of exact unresolved pedigree-label pairs.
material_pairs = overlapping_label_comparisons.loc[
    overlapping_label_comparisons[
        "comparison_type"
    ].eq("material_text_difference")
].copy()

material_pairs["left_label_sorted"] = material_pairs[
    ["left_raw_label", "right_raw_label"]
].min(axis=1)

material_pairs["right_label_sorted"] = material_pairs[
    ["left_raw_label", "right_raw_label"]
].max(axis=1)

pair_recurrence = (
    material_pairs
    .groupby(
        [
            "component",
            "left_label_sorted",
            "right_label_sorted",
        ],
        as_index=False,
    )
    .agg(
        horse_labels=("horse", "nunique"),
        horse_examples=(
            "horse",
            lambda values: " | ".join(
                sorted(set(values))[:8]
            ),
        ),
    )
)

label_usage_rows = []

for row in pair_recurrence.itertuples(index=False):
    affected_horses = material_pairs.loc[
        material_pairs["component"].eq(row.component)
        & material_pairs["left_label_sorted"].eq(
            row.left_label_sorted
        )
        & material_pairs["right_label_sorted"].eq(
            row.right_label_sorted
        ),
        "horse",
    ].unique()

    relevant_rows = overlap_rows.loc[
        overlap_rows["horse"].isin(affected_horses)
    ]

    for label_position, raw_label in [
        ("left", row.left_label_sorted),
        ("right", row.right_label_sorted),
    ]:
        matching_rows = relevant_rows.loc[
            relevant_rows[row.component].eq(raw_label)
        ]

        label_usage_rows.append(
            {
                "component": row.component,
                "left_label_sorted": row.left_label_sorted,
                "right_label_sorted": row.right_label_sorted,
                "label_position": label_position,
                "raw_label": raw_label,
                "runner_rows": len(matching_rows),
                "horse_labels": matching_rows[
                    "horse"
                ].nunique(),
                "first_date": matching_rows["date"].min(),
                "last_date": matching_rows["date"].max(),
            }
        )

pair_label_usage = pd.DataFrame(label_usage_rows)

left_usage = (
    pair_label_usage.loc[
        pair_label_usage["label_position"].eq("left"),
        [
            "component",
            "left_label_sorted",
            "right_label_sorted",
            "raw_label",
            "runner_rows",
            "first_date",
            "last_date",
        ],
    ]
    .rename(
        columns={
            "raw_label": "left_label",
            "runner_rows": "left_runner_rows",
            "first_date": "left_first_date",
            "last_date": "left_last_date",
        }
    )
)

right_usage = (
    pair_label_usage.loc[
        pair_label_usage["label_position"].eq("right"),
        [
            "component",
            "left_label_sorted",
            "right_label_sorted",
            "raw_label",
            "runner_rows",
            "first_date",
            "last_date",
        ],
    ]
    .rename(
        columns={
            "raw_label": "right_label",
            "runner_rows": "right_runner_rows",
            "first_date": "right_first_date",
            "last_date": "right_last_date",
        }
    )
)

material_pair_recurrence = (
    pair_recurrence
    .merge(
        left_usage,
        on=[
            "component",
            "left_label_sorted",
            "right_label_sorted",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        right_usage,
        on=[
            "component",
            "left_label_sorted",
            "right_label_sorted",
        ],
        how="left",
        validate="one_to_one",
    )
    .drop(
        columns=[
            "left_label_sorted",
            "right_label_sorted",
        ]
    )
    .sort_values(
        [
            "horse_labels",
            "component",
            "left_label",
            "right_label",
        ],
        ascending=[False, True, True, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert (
    material_pair_recurrence["horse_labels"].sum()
    >= len(material_pairs)
)

print("Material pedigree-label pair recurrence confirmed")
material_pair_recurrence

Material pedigree-label pair recurrence confirmed


,component,horse_labels,horse_examples,left_label,left_runner_rows,left_first_date,left_last_date,right_label,right_runner_rows,right_first_date,right_last_date
0,damsire,13,Fr Humphrey (IRE) | Full Drago (ITY) | Island ...,Almutawakel,94,2015-04-30,2022-03-11,AlmutawakelI,117,2015-01-21,2019-12-11
1,damsire,2,Fr Humphrey (IRE) | Ivilnoble (IRE),Almutawakel,29,2016-11-16,2022-03-11,Almutawakel I,6,2023-05-15,2024-10-19
2,dam,1,Attention All (IRE),Moon Light Shadow I GB,7,2022-05-10,2024-02-03,Moonlight Shadow GB,1,2024-01-06,2024-01-06
3,dam,1,Turf Brilliant (AUS),Paradise Lost (IRE),12,2020-01-22,2022-03-30,Paradise Lost I (IRE),16,2019-09-08,2023-06-25
4,dam,1,Calivigny (IRE),Summer Holiday (IRE),13,2015-01-02,2021-12-21,Summer Holiday I (IRE),33,2015-01-28,2019-12-29
5,damsire,1,Attention All (IRE),Compton Place,7,2022-05-10,2024-02-03,Fountain Of Youth,1,2024-01-06,2024-01-06


### Stage 5i — Source-wide recurrence of the Almutawakel label family

The overlapping-case analysis found the same damsire-label pair across many unrelated horses:

- `Almutawakel`;
- `AlmutawakelI`;
- `Almutawakel I`.

This recurrence is consistent with a systematic source-label variation, but the prior counts were restricted to horse labels already known to have overlapping pedigree groups.

This stage profiles the three exact labels across the complete governed runner population.

It establishes:

- total runner-row and horse-label coverage;
- first and last appearance dates;
- jurisdictions and courses;
- whether the forms occur against the same dam labels;
- whether individual dam labels are associated with more than one form.

The raw labels remain separate. Source-wide recurrence may justify a provisional equivalence candidate, but authoritative equivalence still requires bounded external verification.

In [20]:
# Profile the Almutawakel damsire-label family across the complete source.
ALMUTAWAKEL_LABELS = [
    "Almutawakel",
    "AlmutawakelI",
    "Almutawakel I",
]

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    almutawakel_source_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            race_id AS supplied_race_id,
            date,
            course,
            off,
            horse,
            sire,
            dam,
            damsire,
            sex,
            age
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND damsire IN (?, ?, ?)
        ORDER BY
            damsire,
            date,
            course,
            off,
            horse,
            rowid
        """,
        connection,
        params=ALMUTAWAKEL_LABELS,
    )
finally:
    connection.close()

assert set(almutawakel_source_rows["damsire"].unique()).issubset(
    set(ALMUTAWAKEL_LABELS)
)

almutawakel_label_summary = (
    almutawakel_source_rows
    .groupby("damsire", as_index=False)
    .agg(
        runner_rows=("source_rowid", "size"),
        distinct_horse_labels=("horse", "nunique"),
        distinct_dam_labels=("dam", "nunique"),
        distinct_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        horse_examples=(
            "horse",
            lambda values: " | ".join(
                sorted(set(values))[:8]
            ),
        ),
        dam_examples=(
            "dam",
            lambda values: " | ".join(
                sorted(set(values))[:8]
            ),
        ),
    )
    .sort_values(
        ["runner_rows", "damsire"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

almutawakel_dam_crosswalk = (
    almutawakel_source_rows
    .groupby("dam", as_index=False)
    .agg(
        distinct_damsire_forms=("damsire", "nunique"),
        damsire_forms=(
            "damsire",
            lambda values: " | ".join(
                sorted(set(values))
            ),
        ),
        runner_rows=("source_rowid", "size"),
        distinct_horse_labels=("horse", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        horse_examples=(
            "horse",
            lambda values: " | ".join(
                sorted(set(values))[:8]
            ),
        ),
    )
)

almutawakel_multi_form_dams = (
    almutawakel_dam_crosswalk.loc[
        almutawakel_dam_crosswalk[
            "distinct_damsire_forms"
        ].gt(1)
    ]
    .sort_values(
        ["runner_rows", "dam"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

almutawakel_source_summary = pd.DataFrame(
    [
        {
            "measure": "runner rows using any form",
            "value": len(almutawakel_source_rows),
        },
        {
            "measure": "distinct horse labels",
            "value": almutawakel_source_rows[
                "horse"
            ].nunique(),
        },
        {
            "measure": "distinct dam labels",
            "value": almutawakel_source_rows[
                "dam"
            ].nunique(),
        },
        {
            "measure": "dam labels associated with multiple forms",
            "value": len(almutawakel_multi_form_dams),
        },
    ]
)

print("Source-wide Almutawakel label-family profile confirmed")
display(almutawakel_source_summary)

print("Exact label usage")
display(almutawakel_label_summary)

print("Dam labels associated with multiple exact forms")
almutawakel_multi_form_dams

Source-wide Almutawakel label-family profile confirmed


,measure,value
0,runner rows using any form,423
1,distinct horse labels,41
2,distinct dam labels,19
3,dam labels associated with multiple forms,14


Exact label usage


,damsire,runner_rows,distinct_horse_labels,distinct_dam_labels,distinct_courses,first_date,last_date,horse_examples,dam_examples
0,AlmutawakelI,182,30,17,67,2015-01-16,2019-12-11,Arterial (IRE) | Arthurs Choice (GB) | Betsy C...,Almata (IRE) | Almutamore (IRE) | An Realt Bea...
1,Almutawakel,162,27,15,68,2015-04-30,2022-05-22,Bardo (FR) | Capla Berry (GB) | Couples (GB) |...,Almata (IRE) | Almutamore (IRE) | An Realt Bea...
2,Almutawakel I,79,10,9,35,2022-05-10,2026-05-16,Bardo (FR) | Fr Humphrey (IRE) | I Know I Can ...,Almutamore (IRE) | An Realt Beag (IRE) | Boo B...


Dam labels associated with multiple exact forms


,dam,distinct_damsire_forms,damsire_forms,runner_rows,distinct_horse_labels,first_date,last_date,horse_examples
0,Salsa Brava (IRE),3,Almutawakel | Almutawakel I | AlmutawakelI,64,3,2015-04-13,2023-09-12,Capla Berry (GB) | I Know I Can (GB) | Mystic ...
1,Sensible GB,3,Almutawakel | Almutawakel I | AlmutawakelI,61,5,2015-04-26,2024-10-31,Couples (GB) | Mauchline (GB) | Stricker (GB) ...
2,Lucky Norwegian (IRE),2,Almutawakel | AlmutawakelI,46,4,2015-01-16,2021-10-16,Arterial (IRE) | Betsy Coed (IRE) | Eve Bee (I...
3,Senetosa (FR),3,Almutawakel | Almutawakel I | AlmutawakelI,45,2,2016-11-19,2025-04-03,Senepark (FR) | Senewalk (FR)
4,Boo Boo Bear (IRE),3,Almutawakel | Almutawakel I | AlmutawakelI,36,6,2015-01-30,2026-05-16,Island Vision (IRE) | Letters Of Note (IRE) | ...
5,An Realt Beag (IRE),3,Almutawakel | Almutawakel I | AlmutawakelI,31,1,2015-01-21,2024-10-19,Fr Humphrey (IRE)
6,Almutamore (IRE),3,Almutawakel | Almutawakel I | AlmutawakelI,30,2,2017-11-08,2024-01-28,Cross Cover (IRE) | Ivilnoble (IRE)
7,Dilag (IRE),3,Almutawakel | Almutawakel I | AlmutawakelI,26,4,2015-04-15,2023-11-10,Bardo (FR) | Life On Mars (GB) | Souvenir Delo...
8,Panther Moon (IRE),2,Almutawakel | Almutawakel I,22,2,2018-12-15,2026-05-13,San Franco (IRE) | Santa Rossa (IRE)
9,Fair Countenance (IRE),2,Almutawakel | AlmutawakelI,12,1,2017-01-26,2020-09-22,Wedding Breakfast (IRE)


### Interim conclusion — Systematic damsire-label variation

The source-wide profile confirms that `Almutawakel`, `AlmutawakelI` and `Almutawakel I` are not isolated one-row anomalies.

Across the governed source they occur in:

- 423 runner rows;
- 41 distinct horse labels;
- 19 distinct dam labels;
- 14 dam labels associated with more than one exact form.

Several dam labels are associated with all three forms, including:

- `Salsa Brava (IRE)`;
- `Sensible GB`;
- `Senetosa (FR)`;
- `Boo Boo Bear (IRE)`;
- `An Realt Beag (IRE)`;
- `Almutamore (IRE)`;
- `Dilag (IRE)`.

The three forms overlap across horses, courses and periods. This rules out a simple one-time format transition and strongly supports a systematic source-label variation affecting the same apparent damsire identity.

The source-internal evidence therefore supports the following provisional rule:

- preserve every raw `damsire` label unchanged;
- treat `Almutawakel`, `AlmutawakelI` and `Almutawakel I` as members of one provisional label-variant family;
- do not silently replace any form with another;
- retain the exact transformation or matching method;
- classify the relationship as a provisional equivalence candidate rather than a verified real-world entity match;
- flag any use of the grouped form with confidence and review status.

This bounded equivalence candidate removes a large share of the remaining overlapping pedigree contradictions, but authoritative equivalence still requires external verification.

The other unresolved overlapping cases remain separate:

- `Moon Light Shadow I GB` versus `Moonlight Shadow GB`;
- `Summer Holiday (IRE)` versus `Summer Holiday I (IRE)`;
- `Paradise Lost (IRE)` versus `Paradise Lost I (IRE)`;
- `Compton Place` versus `Fountain Of Youth`.

Those differences must not be normalized from source similarity alone.

#### Manual-verification decision

Manual or external verification is justified.

The first bounded claim to verify should be:

> Do `Almutawakel`, `AlmutawakelI` and `Almutawakel I` refer to the same real-world sire?

If confirmed, the evidence must be captured as one bounded claim in `data/reference/manual_verifications.csv` with the exact raw labels, source examples, external locator, access date, confidence and permitted database action.

No source value should be overwritten. The permitted action would be limited to assigning the three raw labels to one verified label-equivalence group while preserving the original text and lineage.

### Stage 5j — External verification of the Almutawakel label family

The source-internal analysis identified three recurring `damsire` labels:

* `Almutawakel`;
* `AlmutawakelI`;
* `Almutawakel I`.

The source alone strongly suggested that these labels refer to one real-world horse, but that equivalence required external verification.

Godolphin’s official profile provides direct evidence. The page identifies the horse as `Almutawakel (GB)` in its title and profile, while the pedigree section on the same page identifies that horse as `Almutawakel I (GB)`. Both references describe the bay horse foaled on 19 January 1995, by Machiavellian out of Elfaslah.

This directly confirms that:

* `Almutawakel` and `Almutawakel I` are alternative labels for the same real-world horse;
* the terminal `I` does not identify a different sire in this bounded case.

The source form `AlmutawakelI` differs from the externally verified `Almutawakel I` only by the absence of a space. Its repeated source-wide use against the same dam labels supports treating it as a source formatting variant of the same verified label.

The bounded verified conclusion is therefore:

> `Almutawakel`, `AlmutawakelI` and `Almutawakel I` may be assigned to one governed pedigree-label equivalence group representing the 1995 British-bred stallion Almutawakel, while every raw source label remains preserved.

This conclusion does not justify removing a terminal `I`, an attached `I` or any other apparent suffix from unrelated horse or pedigree labels. The rule applies only to this explicitly verified label family.

The permitted database action is:

* preserve the exact raw `damsire` label;
* assign the three labels to one verified label-equivalence group;
* expose `Almutawakel (GB)` as the verified display label;
* preserve the verification identifier, evidence locator, method, confidence and review status;
* do not overwrite or amend the immutable source value.

Evidence:

* provider: Godolphin;
* evidence type: official horse profile;
* profile heading: `Almutawakel (GB)`;
* pedigree label: `Almutawakel I (GB)`;
* foaling date: 19 January 1995;
* sire: Machiavellian;
* dam: Elfaslah;
* accessed: 1 August 2026.

Manual-verification status: `captured`.

Verification ID: `NB19-HORSE-0001`.


### Stage 5k — Analytical consequence of verification `NB19-HORSE-0001`

Verification `NB19-HORSE-0001` confirms that one recurring source-label family represents a single real-world sire under multiple textual forms:

* `Almutawakel`;
* `AlmutawakelI`;
* `Almutawakel I`.

This establishes an important but deliberately narrow rule.

A repeated pedigree-label difference is not automatically evidence of a different real-world entity. Some source labels vary through spacing, legacy naming conventions or publisher-specific formatting while referring to the same horse.

The verified rule is specific to this label family:

* all three raw labels must remain preserved;
* all three may be linked to one verified pedigree-label equivalence group;
* the preferred verified display label is `Almutawakel (GB)`;
* the relationship must retain verification ID `NB19-HORSE-0001`;
* the relationship carries high confidence;
* the source rows must not be overwritten;
* the rule must not be generalized to other labels merely because they contain an attached or separated terminal `I`.

This verification reduces the number of unresolved overlapping pedigree cases, but it does not resolve the broader horse-identity problem.

The source evidence now supports three distinct classes of repeated-label behaviour:

1. **Reversible source-format variation**

   Example:

   * `China Cherub GB`;
   * `China Cherub (GB)`.

   These retain the same parsed display name and country token under different source formats.

2. **Externally verified label equivalence**

   Example:

   * `Almutawakel`;
   * `AlmutawakelI`;
   * `Almutawakel I`.

   These may be grouped only because the source-wide evidence and external verification support one bounded real-world identity.

3. **Unresolved competing assertions**

   Examples include:

   * `Moon Light Shadow I GB` versus `Moonlight Shadow GB`;
   * `Summer Holiday (IRE)` versus `Summer Holiday I (IRE)`;
   * `Paradise Lost (IRE)` versus `Paradise Lost I (IRE)`;
   * `Compton Place` versus `Fountain Of Youth`.

   These must remain separate until further evidence establishes whether they are transcription variants, corrections, contamination or genuinely different pedigree identities.

The emerging database rule is therefore:

> Raw pedigree labels are source assertions. Structured parsing may identify reversible formatting differences, and bounded external evidence may establish specific label equivalences, but unverified textual similarity must not create entity equivalence.

This distinction must be retained in any later implementation through separate fields for:

* raw label;
* parsed display label;
* parsed country token;
* comparison or parsing method;
* provisional equivalence-group identifier;
* verification identifier;
* confidence;
* review status;
* unresolved-conflict flag.

The next analytical task is to determine whether the remaining unresolved overlapping cases materially affect the source-level horse grouping rule or can remain flagged for later entity-resolution review.


### Stage 5l — Chronology of the remaining unresolved overlapping cases

After applying reversible dam parsing and the verified `Almutawakel` label equivalence, three exact horse labels retain materially different overlapping pedigree assertions:

* `Attention All (IRE)`;
* `Calivigny (IRE)`;
* `Turf Brilliant (AUS)`.

These cases cannot yet be classified safely from textual similarity alone.

This stage profiles each complete pedigree assertion by:

* exact horse label;
* sire, structured dam and damsire labels;
* number of runner rows;
* first and last source dates;
* minimum and maximum recorded age;
* observed sex values;
* number of courses;
* representative course names.

The purpose is to determine whether each difference resembles:

* an isolated source error;
* a temporary correction or reversal;
* a persistent competing assertion;
* or evidence that the same exact horse label has been reused for different real-world horses.

No additional normalization or entity merge is performed.


In [21]:
# Profile the remaining unresolved overlapping pedigree assertions.
UNRESOLVED_OVERLAPPING_HORSES = [
    "Attention All (IRE)",
    "Calivigny (IRE)",
    "Turf Brilliant (AUS)",
]

required_columns = {
    "horse",
    "sire",
    "dam",
    "damsire",
    "date",
    "age",
    "sex",
    "course",
}

missing_columns = required_columns.difference(overlap_rows.columns)

assert not missing_columns, (
    "overlap_rows is missing required columns: "
    f"{sorted(missing_columns)}"
)

unresolved_overlap_rows = overlap_rows.loc[
    overlap_rows["horse"].isin(
        UNRESOLVED_OVERLAPPING_HORSES
    )
].copy()

assert set(
    unresolved_overlap_rows["horse"].unique()
) == set(UNRESOLVED_OVERLAPPING_HORSES)

unresolved_assertion_chronology = (
    unresolved_overlap_rows
    .groupby(
        [
            "horse",
            "sire",
            "dam",
            "damsire",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_age=("age", "min"),
        maximum_age=("age", "max"),
        sex_values=(
            "sex",
            lambda values: " | ".join(
                sorted(
                    {
                        str(value)
                        for value in values
                        if pd.notna(value)
                    }
                )
            ),
        ),
        distinct_courses=("course", "nunique"),
        course_examples=(
            "course",
            lambda values: " | ".join(
                sorted(
                    {
                        str(value)
                        for value in values
                        if pd.notna(value)
                    }
                )[:8]
            ),
        ),
    )
    .sort_values(
        [
            "horse",
            "first_date",
            "last_date",
            "sire",
            "dam",
            "damsire",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

unresolved_horse_summary = (
    unresolved_assertion_chronology
    .groupby("horse", as_index=False)
    .agg(
        pedigree_assertions=("horse", "size"),
        total_runner_rows=("runner_rows", "sum"),
        first_date=("first_date", "min"),
        last_date=("last_date", "max"),
    )
    .sort_values("horse", kind="stable")
    .reset_index(drop=True)
)

assert unresolved_horse_summary[
    "pedigree_assertions"
].ge(2).all()

assert (
    unresolved_horse_summary["total_runner_rows"].sum()
    == len(unresolved_overlap_rows)
)

print("Remaining unresolved overlapping cases confirmed")
display(unresolved_horse_summary)

print("Complete raw pedigree-assertion chronology")
unresolved_assertion_chronology

Remaining unresolved overlapping cases confirmed


,horse,pedigree_assertions,total_runner_rows,first_date,last_date
0,Attention All (IRE),2,8,2022-05-10,2024-02-03
1,Calivigny (IRE),2,46,2015-01-02,2021-12-21
2,Turf Brilliant (AUS),2,28,2019-09-08,2023-06-25


Complete raw pedigree-assertion chronology


,horse,sire,dam,damsire,runner_rows,first_date,last_date,minimum_age,maximum_age,sex_values,distinct_courses,course_examples
0,Attention All (IRE),Westerner (GB),Moon Light Shadow I GB,Compton Place,7,2022-05-10,2024-02-03,4,6,G,5,Ayr | Carlisle | Newcastle | Sedgefield | Weth...
1,Attention All (IRE),Westerner (GB),Moonlight Shadow GB,Fountain Of Youth,1,2024-01-06,2024-01-06,6,6,G,1,Newcastle
2,Calivigny (IRE),Gold Well (GB),Summer Holiday (IRE),Kambalda,13,2015-01-02,2021-12-21,6,12,G,5,Ayr | Carlisle | Kelso | Musselburgh | Perth
3,Calivigny (IRE),Gold Well (GB),Summer Holiday I (IRE),Kambalda,33,2015-01-28,2019-12-29,6,10,G,9,Ayr | Carlisle | Hexham | Kelso | Musselburgh ...
4,Turf Brilliant (AUS),Manhattan Rain (AUS),Paradise Lost I (IRE),Sadlers Wells,16,2019-09-08,2023-06-25,4,7,G,2,Happy Valley (HK) | Sha Tin (HK)
5,Turf Brilliant (AUS),Manhattan Rain (AUS),Paradise Lost (IRE),Sadlers Wells,12,2020-01-22,2022-03-30,4,6,G,2,Happy Valley (HK) | Sha Tin (HK)


### Stage 5m — Switching chronology of unresolved dam assertions

The complete assertion chronology suggests two different failure patterns.

`Attention All (IRE)` contains one isolated row in which both the dam and damsire change:

* seven rows: `Moon Light Shadow I GB` and `Compton Place`;
* one row: `Moonlight Shadow GB` and `Fountain Of Youth`.

Because two pedigree components change together in a single overlapping observation, this case may represent a source-row defect, correction artefact or contaminated pedigree assertion. It cannot be resolved from textual similarity alone.

`Calivigny (IRE)` and `Turf Brilliant (AUS)` differ only in the dam label:

* `Summer Holiday (IRE)` versus `Summer Holiday I (IRE)`;
* `Paradise Lost (IRE)` versus `Paradise Lost I (IRE)`.

For both horses:

* sire remains unchanged;
* damsire remains unchanged;
* sex remains unchanged;
* recorded ages form one continuous life history;
* the competing dam forms overlap in time.

This is consistent with persistent source-label variation rather than reuse of the exact horse label for different real-world horses. However, the terminal `I` may be meaningful in pedigree naming, so equivalence must not be declared without bounded evidence.

This stage inspects the row-level sequence to determine whether the competing assertions appear as isolated anomalies, one-way transitions or repeated switching between forms.

No normalization rule is applied.


In [22]:
# Inspect row-level switching between unresolved pedigree assertions.
switching_rows = (
    unresolved_overlap_rows[
        [
            "horse",
            "date",
            "course",
            "off",
            "age",
            "sex",
            "sire",
            "dam",
            "damsire",
        ]
    ]
    .sort_values(
        ["horse", "date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

switching_rows["previous_dam"] = (
    switching_rows
    .groupby("horse")["dam"]
    .shift()
)

switching_rows["previous_damsire"] = (
    switching_rows
    .groupby("horse")["damsire"]
    .shift()
)

switching_rows["dam_changed"] = (
    switching_rows["previous_dam"].notna()
    & switching_rows["dam"].ne(
        switching_rows["previous_dam"]
    )
)

switching_rows["damsire_changed"] = (
    switching_rows["previous_damsire"].notna()
    & switching_rows["damsire"].ne(
        switching_rows["previous_damsire"]
    )
)

switching_rows["pedigree_changed"] = (
    switching_rows["dam_changed"]
    | switching_rows["damsire_changed"]
)

switching_summary = (
    switching_rows
    .groupby("horse", as_index=False)
    .agg(
        runner_rows=("horse", "size"),
        dam_forms=("dam", "nunique"),
        damsire_forms=("damsire", "nunique"),
        dam_switches=("dam_changed", "sum"),
        damsire_switches=("damsire_changed", "sum"),
        pedigree_switches=("pedigree_changed", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values("horse", kind="stable")
    .reset_index(drop=True)
)

switch_events = (
    switching_rows.loc[
        switching_rows["pedigree_changed"],
        [
            "horse",
            "date",
            "course",
            "off",
            "age",
            "sex",
            "previous_dam",
            "dam",
            "previous_damsire",
            "damsire",
            "dam_changed",
            "damsire_changed",
        ],
    ]
    .reset_index(drop=True)
)

assert switching_summary["runner_rows"].sum() == len(
    unresolved_overlap_rows
)

print("Unresolved pedigree switching chronology confirmed")
display(switching_summary)

print("Rows where the recorded pedigree assertion changed")
switch_events

Unresolved pedigree switching chronology confirmed


,horse,runner_rows,dam_forms,damsire_forms,dam_switches,damsire_switches,pedigree_switches,first_date,last_date
0,Attention All (IRE),8,2,2,2,2,2,2022-05-10,2024-02-03
1,Calivigny (IRE),46,2,1,8,0,8,2015-01-02,2021-12-21
2,Turf Brilliant (AUS),28,2,1,2,0,2,2019-09-08,2023-06-25


Rows where the recorded pedigree assertion changed


,horse,date,course,off,age,sex,previous_dam,dam,previous_damsire,damsire,dam_changed,damsire_changed
0,Attention All (IRE),2024-01-06,Newcastle,2:50,6,G,Moon Light Shadow I GB,Moonlight Shadow GB,Compton Place,Fountain Of Youth,True,True
1,Attention All (IRE),2024-02-03,Wetherby,3:17,6,G,Moonlight Shadow GB,Moon Light Shadow I GB,Fountain Of Youth,Compton Place,True,True
2,Calivigny (IRE),2015-01-28,Newcastle,3:50,6,G,Summer Holiday (IRE),Summer Holiday I (IRE),Kambalda,Kambalda,True,False
3,Calivigny (IRE),2017-04-05,Carlisle,4:05,8,G,Summer Holiday I (IRE),Summer Holiday (IRE),Kambalda,Kambalda,True,False
4,Calivigny (IRE),2017-04-26,Perth,4:40,8,G,Summer Holiday (IRE),Summer Holiday I (IRE),Kambalda,Kambalda,True,False
5,Calivigny (IRE),2018-01-02,Ayr,2:10,9,G,Summer Holiday I (IRE),Summer Holiday (IRE),Kambalda,Kambalda,True,False
6,Calivigny (IRE),2018-01-31,Ayr,3:45,9,G,Summer Holiday (IRE),Summer Holiday I (IRE),Kambalda,Kambalda,True,False
7,Calivigny (IRE),2018-11-12,Carlisle,3:10,9,G,Summer Holiday I (IRE),Summer Holiday (IRE),Kambalda,Kambalda,True,False
8,Calivigny (IRE),2018-12-02,Carlisle,3:30,9,G,Summer Holiday (IRE),Summer Holiday I (IRE),Kambalda,Kambalda,True,False
9,Calivigny (IRE),2020-01-19,Ayr,3:00,11,G,Summer Holiday I (IRE),Summer Holiday (IRE),Kambalda,Kambalda,True,False


### Interim interpretation — Repeated switching distinguishes label variation from pedigree conflict

The row-level chronology separates the remaining cases into two different source behaviours.

#### Persistent switching between dam-label forms

`Calivigny (IRE)` switches eight times between:

* `Summer Holiday (IRE)`;
* `Summer Holiday I (IRE)`.

The switching occurs between 2015 and 2020 while the following attributes remain stable:

* sire: `Gold Well (GB)`;
* damsire: `Kambalda`;
* sex: `G`;
* continuous recorded age progression;
* one coherent racing history.

`Turf Brilliant (AUS)` switches twice between:

* `Paradise Lost (IRE)`;
* `Paradise Lost I (IRE)`.

The switching occurs between 2020 and 2022 while the following attributes remain stable:

* sire: `Manhattan Rain (AUS)`;
* damsire: `Sadlers Wells`;
* sex: `G`;
* continuous recorded age progression;
* the same two Hong Kong courses.

Repeated reversals make a simple one-time correction or clean format transition unlikely. They are more consistent with persistent source-level variation in the dam label attached to one coherent horse history.

The evidence supports classifying both pairs as **provisional label-equivalence candidates**, but not as verified real-world equivalences. The terminal `I` must remain intact until bounded external evidence confirms its meaning in each case.

#### Isolated full-pedigree conflict

`Attention All (IRE)` shows a different pattern.

Seven rows use:

* dam: `Moon Light Shadow I GB`;
* damsire: `Compton Place`.

One Newcastle row dated 6 January 2024 instead uses:

* dam: `Moonlight Shadow GB`;
* damsire: `Fountain Of Youth`.

At the following recorded run on 3 February 2024, the original dam and damsire return.

Because both pedigree components change together for one isolated row, this is not safely explainable as punctuation, spacing or terminal-`I` variation. It should remain an unresolved source assertion and a likely row-level pedigree defect candidate.

The governed distinction is therefore:

* repeated switching with stable surrounding pedigree may justify a provisional label-variant group;
* an isolated row changing multiple pedigree components must remain a conflicting assertion;
* neither class permits overwriting the raw source;
* only bounded external evidence may convert a provisional variant into verified entity equivalence.


In [23]:
# Recalculate contradiction status after bounded classifications.
#
# Verified equivalence:
#   Almutawakel / AlmutawakelI / Almutawakel I
#
# Provisional label-variant candidates:
#   Summer Holiday (IRE) / Summer Holiday I (IRE)
#   Paradise Lost (IRE) / Paradise Lost I (IRE)
#
# Attention All remains unresolved.

VERIFIED_PEDIGREE_LABEL_GROUPS = {
    "almutawakel_verified": {
        "Almutawakel",
        "AlmutawakelI",
        "Almutawakel I",
    },
}

PROVISIONAL_DAM_LABEL_GROUPS = {
    "summer_holiday_terminal_i_candidate": {
        "Summer Holiday (IRE)",
        "Summer Holiday I (IRE)",
    },
    "paradise_lost_terminal_i_candidate": {
        "Paradise Lost (IRE)",
        "Paradise Lost I (IRE)",
    },
}


def map_label_group(
    value: object,
    groups: dict[str, set[str]],
) -> object:
    if pd.isna(value):
        return value

    for group_id, labels in groups.items():
        if value in labels:
            return group_id

    return value


reclassified_overlap_rows = overlap_rows.copy()

reclassified_overlap_rows["reclassified_dam"] = (
    reclassified_overlap_rows["dam"].map(
        lambda value: map_label_group(
            value,
            PROVISIONAL_DAM_LABEL_GROUPS,
        )
    )
)

reclassified_overlap_rows["reclassified_damsire"] = (
    reclassified_overlap_rows["damsire"].map(
        lambda value: map_label_group(
            value,
            VERIFIED_PEDIGREE_LABEL_GROUPS,
        )
    )
)

reclassified_horse_contradictions = (
    reclassified_overlap_rows
    .groupby("horse", as_index=False)
    .agg(
        sire_forms=("sire", "nunique"),
        dam_forms=("reclassified_dam", "nunique"),
        damsire_forms=("reclassified_damsire", "nunique"),
        runner_rows=("horse", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

reclassified_horse_contradictions["has_contradiction"] = (
    reclassified_horse_contradictions[
        ["sire_forms", "dam_forms", "damsire_forms"]
    ]
    .gt(1)
    .any(axis=1)
)

reclassification_summary = pd.DataFrame(
    [
        {
            "measure": "structured contradiction labels before bounded classification",
            "value": overlap_rows["horse"].nunique(),
        },
        {
            "measure": "labels still contradictory after bounded classification",
            "value": int(
                reclassified_horse_contradictions[
                    "has_contradiction"
                ].sum()
            ),
        },
        {
            "measure": "labels resolved by verified Almutawakel grouping",
            "value": int(
                overlap_rows.loc[
                    overlap_rows["damsire"].isin(
                        VERIFIED_PEDIGREE_LABEL_GROUPS[
                            "almutawakel_verified"
                        ]
                    ),
                    "horse",
                ].nunique()
            ),
        },
        {
            "measure": "labels resolved by provisional dam-variant grouping",
            "value": int(
                overlap_rows.loc[
                    overlap_rows["dam"].isin(
                        set().union(
                            *PROVISIONAL_DAM_LABEL_GROUPS.values()
                        )
                    ),
                    "horse",
                ].nunique()
            ),
        },
    ]
)

remaining_reclassified_contradictions = (
    reclassified_horse_contradictions.loc[
        reclassified_horse_contradictions[
            "has_contradiction"
        ]
    ]
    .sort_values(
        ["runner_rows", "horse"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert "Attention All (IRE)" in set(
    remaining_reclassified_contradictions["horse"]
)

assert "Calivigny (IRE)" not in set(
    remaining_reclassified_contradictions["horse"]
)

assert "Turf Brilliant (AUS)" not in set(
    remaining_reclassified_contradictions["horse"]
)

print("Bounded pedigree-label reclassification confirmed")
display(reclassification_summary)

print("Remaining contradictory horse labels")
remaining_reclassified_contradictions

Bounded pedigree-label reclassification confirmed


,measure,value
0,structured contradiction labels before bounded...,18
1,labels still contradictory after bounded class...,3
2,labels resolved by verified Almutawakel grouping,13
3,labels resolved by provisional dam-variant gro...,2


Remaining contradictory horse labels


,horse,sire_forms,dam_forms,damsire_forms,runner_rows,first_date,last_date,has_contradiction
0,Shielas Well (IRE),1,2,1,29,2024-06-14,2026-05-05,True
1,Holly (FR),2,1,1,15,2020-11-27,2024-12-14,True
2,Attention All (IRE),1,2,2,8,2022-05-10,2024-02-03,True


### Stage 5n — Chronology of the remaining bounded textual variants

After the verified `Almutawakel` grouping and the two provisional terminal-`I` dam groups, three exact horse labels remain contradictory:

* `Shielas Well (IRE)`;
* `Holly (FR)`;
* `Attention All (IRE)`.

The first two differ through narrowly bounded textual transformations already identified by the comparison stage:

* `AmcHitka (IRE)` versus `Amchitka (IRE)` differ only by letter case;
* `Ut*voiladenuo (FR)` versus `Voiladenuo (FR)` differ only by the leading source token `Ut*`.

These textual relationships are much narrower than the competing dam and damsire assertion for `Attention All (IRE)`, but similarity alone still does not establish real-world entity equivalence.

This stage profiles the chronology of `Shielas Well (IRE)` and `Holly (FR)` to determine whether each difference:

* occurs within one continuous horse history;
* preserves the surrounding pedigree;
* appears once or repeatedly;
* reverses between forms;
* or separates into distinct periods consistent with reused horse labels.

No additional equivalence group is applied at this stage.


In [24]:
# Inspect chronology of the remaining bounded textual-variant cases.
BOUNDED_TEXTUAL_VARIANT_HORSES = [
    "Shielas Well (IRE)",
    "Holly (FR)",
]

bounded_variant_rows = (
    overlap_rows.loc[
        overlap_rows["horse"].isin(
            BOUNDED_TEXTUAL_VARIANT_HORSES
        ),
        [
            "horse",
            "date",
            "course",
            "off",
            "age",
            "sex",
            "sire",
            "dam",
            "damsire",
        ],
    ]
    .sort_values(
        ["horse", "date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert set(
    bounded_variant_rows["horse"].unique()
) == set(BOUNDED_TEXTUAL_VARIANT_HORSES)

bounded_variant_assertions = (
    bounded_variant_rows
    .groupby(
        [
            "horse",
            "sire",
            "dam",
            "damsire",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_age=("age", "min"),
        maximum_age=("age", "max"),
        sex_values=(
            "sex",
            lambda values: " | ".join(
                sorted(
                    {
                        str(value)
                        for value in values
                        if pd.notna(value)
                    }
                )
            ),
        ),
        distinct_courses=("course", "nunique"),
        course_examples=(
            "course",
            lambda values: " | ".join(
                sorted(
                    {
                        str(value)
                        for value in values
                        if pd.notna(value)
                    }
                )[:8]
            ),
        ),
    )
    .sort_values(
        [
            "horse",
            "first_date",
            "last_date",
            "sire",
            "dam",
            "damsire",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

bounded_variant_rows["previous_sire"] = (
    bounded_variant_rows
    .groupby("horse")["sire"]
    .shift()
)

bounded_variant_rows["previous_dam"] = (
    bounded_variant_rows
    .groupby("horse")["dam"]
    .shift()
)

bounded_variant_rows["sire_changed"] = (
    bounded_variant_rows["previous_sire"].notna()
    & bounded_variant_rows["sire"].ne(
        bounded_variant_rows["previous_sire"]
    )
)

bounded_variant_rows["dam_changed"] = (
    bounded_variant_rows["previous_dam"].notna()
    & bounded_variant_rows["dam"].ne(
        bounded_variant_rows["previous_dam"]
    )
)

bounded_variant_switch_summary = (
    bounded_variant_rows
    .groupby("horse", as_index=False)
    .agg(
        runner_rows=("horse", "size"),
        sire_forms=("sire", "nunique"),
        dam_forms=("dam", "nunique"),
        damsire_forms=("damsire", "nunique"),
        sire_switches=("sire_changed", "sum"),
        dam_switches=("dam_changed", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values("horse", kind="stable")
    .reset_index(drop=True)
)

bounded_variant_switch_events = (
    bounded_variant_rows.loc[
        bounded_variant_rows[
            ["sire_changed", "dam_changed"]
        ].any(axis=1),
        [
            "horse",
            "date",
            "course",
            "off",
            "age",
            "sex",
            "previous_sire",
            "sire",
            "previous_dam",
            "dam",
            "damsire",
            "sire_changed",
            "dam_changed",
        ],
    ]
    .reset_index(drop=True)
)

assert (
    bounded_variant_switch_summary["runner_rows"].sum()
    == len(bounded_variant_rows)
)

print("Bounded textual-variant chronology confirmed")
display(bounded_variant_assertions)

print("Switching summary")
display(bounded_variant_switch_summary)

print("Rows where the bounded label form changed")
bounded_variant_switch_events

Bounded textual-variant chronology confirmed


,horse,sire,dam,damsire,runner_rows,first_date,last_date,minimum_age,maximum_age,sex_values,distinct_courses,course_examples
0,Holly (FR),Voiladenuo (FR),Righty Malta (FR),Turgeon,14,2020-11-27,2024-12-14,3,7,F | M,13,Carlisle | Cheltenham | Exeter | Fakenham | Ha...
1,Holly (FR),Ut*voiladenuo (FR),Righty Malta (FR),Turgeon,1,2022-10-07,2022-10-07,5,5,M,1,Chepstow
2,Shielas Well (IRE),Saxon Warrior (JPN),AmcHitka (IRE),Cape Cross,15,2024-06-14,2025-07-15,2,3,F,11,Beverley | Catterick | Chester | Leicester | M...
3,Shielas Well (IRE),Saxon Warrior (JPN),Amchitka (IRE),Cape Cross,14,2025-06-26,2026-05-05,3,4,F,6,Leicester | Musselburgh | Ripon | Southwell (A...


Switching summary


,horse,runner_rows,sire_forms,dam_forms,damsire_forms,sire_switches,dam_switches,first_date,last_date
0,Holly (FR),15,2,1,1,2,0,2020-11-27,2024-12-14
1,Shielas Well (IRE),29,1,2,1,0,3,2024-06-14,2026-05-05


Rows where the bounded label form changed


,horse,date,course,off,age,sex,previous_sire,sire,previous_dam,dam,damsire,sire_changed,dam_changed
0,Holly (FR),2022-10-07,Chepstow,4:00,5,M,Voiladenuo (FR),Ut*voiladenuo (FR),Righty Malta (FR),Righty Malta (FR),Turgeon,True,False
1,Holly (FR),2022-11-20,Exeter,3:35,5,M,Ut*voiladenuo (FR),Voiladenuo (FR),Righty Malta (FR),Righty Malta (FR),Turgeon,True,False
2,Shielas Well (IRE),2025-06-26,Leicester,9:00,3,F,Saxon Warrior (JPN),Saxon Warrior (JPN),AmcHitka (IRE),Amchitka (IRE),Cape Cross,False,True
3,Shielas Well (IRE),2025-07-15,Thirsk,5:30,3,F,Saxon Warrior (JPN),Saxon Warrior (JPN),Amchitka (IRE),AmcHitka (IRE),Cape Cross,False,True
4,Shielas Well (IRE),2025-08-04,Ripon,4:40,3,F,Saxon Warrior (JPN),Saxon Warrior (JPN),AmcHitka (IRE),Amchitka (IRE),Cape Cross,False,True


### Interim interpretation — Bounded textual variants within coherent horse histories

The remaining bounded textual cases both occur within continuous and otherwise stable horse histories.

#### `Holly (FR)`

Fourteen rows identify the sire as:

* `Voiladenuo (FR)`.

One row dated 7 October 2022 identifies the sire as:

* `Ut*voiladenuo (FR)`.

At the following recorded run, the source returns to `Voiladenuo (FR)`.

Across both forms, the following remain unchanged:

* dam: `Righty Malta (FR)`;
* damsire: `Turgeon`;
* continuous age progression;
* one coherent racing history.

The `Ut*` form is therefore best classified as a **provisional source-prefix variant** of `Voiladenuo (FR)`. The raw value must remain preserved, and the relationship must not be generalized to unrelated labels containing `Ut*`.

#### `Shielas Well (IRE)`

The source alternates between:

* `AmcHitka (IRE)`;
* `Amchitka (IRE)`.

The switch occurs three times, including a reversal back to the earlier form before returning again to `Amchitka (IRE)`.

Across both forms, the following remain unchanged:

* sire: `Saxon Warrior (JPN)`;
* damsire: `Cape Cross`;
* sex: `F`;
* continuous recorded age progression;
* one coherent racing history.

The two dam forms differ only through letter case within the same character sequence. They are therefore best classified as a **provisional case-variant group**.

#### Governed consequence

The source-internal evidence supports provisional comparison groups for:

* `Voiladenuo (FR)` and `Ut*voiladenuo (FR)`;
* `AmcHitka (IRE)` and `Amchitka (IRE)`.

These groups:

* preserve every raw source label;
* record the bounded comparison method;
* do not assert an authoritative real-world identity;
* do not authorize general prefix removal or case correction;
* may be used to prevent these exact textual variants from being counted as independent pedigree contradictions.

After applying these bounded provisional groups, `Attention All (IRE)` should remain the only unresolved overlapping pedigree conflict among the eighteen temporally overlapping cases.


In [25]:
# Apply the two bounded textual-variant groups and recalculate the residue.
PROVISIONAL_SIRE_LABEL_GROUPS = {
    "voiladenuo_ut_prefix_candidate": {
        "Voiladenuo (FR)",
        "Ut*voiladenuo (FR)",
    },
}

PROVISIONAL_CASE_DAM_GROUPS = {
    "amchitka_case_candidate": {
        "AmcHitka (IRE)",
        "Amchitka (IRE)",
    },
}

final_overlap_classification_rows = overlap_rows.copy()

final_overlap_classification_rows["classified_sire"] = (
    final_overlap_classification_rows["sire"].map(
        lambda value: map_label_group(
            value,
            PROVISIONAL_SIRE_LABEL_GROUPS,
        )
    )
)

final_overlap_classification_rows["classified_dam"] = (
    final_overlap_classification_rows["dam"].map(
        lambda value: map_label_group(
            map_label_group(
                value,
                PROVISIONAL_DAM_LABEL_GROUPS,
            ),
            PROVISIONAL_CASE_DAM_GROUPS,
        )
    )
)

final_overlap_classification_rows["classified_damsire"] = (
    final_overlap_classification_rows["damsire"].map(
        lambda value: map_label_group(
            value,
            VERIFIED_PEDIGREE_LABEL_GROUPS,
        )
    )
)

final_overlap_contradictions = (
    final_overlap_classification_rows
    .groupby("horse", as_index=False)
    .agg(
        sire_forms=("classified_sire", "nunique"),
        dam_forms=("classified_dam", "nunique"),
        damsire_forms=("classified_damsire", "nunique"),
        runner_rows=("horse", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

final_overlap_contradictions["has_contradiction"] = (
    final_overlap_contradictions[
        ["sire_forms", "dam_forms", "damsire_forms"]
    ]
    .gt(1)
    .any(axis=1)
)

final_overlap_summary = pd.DataFrame(
    [
        {
            "measure": "temporally overlapping contradiction labels before bounded classification",
            "value": overlap_rows["horse"].nunique(),
        },
        {
            "measure": "resolved by verified Almutawakel equivalence",
            "value": 13,
        },
        {
            "measure": "resolved by provisional terminal-I dam groups",
            "value": 2,
        },
        {
            "measure": "resolved by provisional case-only dam group",
            "value": 1,
        },
        {
            "measure": "resolved by provisional Ut-prefix sire group",
            "value": 1,
        },
        {
            "measure": "remaining unresolved overlapping conflicts",
            "value": int(
                final_overlap_contradictions[
                    "has_contradiction"
                ].sum()
            ),
        },
    ]
)

remaining_final_overlap_conflicts = (
    final_overlap_contradictions.loc[
        final_overlap_contradictions[
            "has_contradiction"
        ]
    ]
    .sort_values(
        ["runner_rows", "horse"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(remaining_final_overlap_conflicts) == 1

assert (
    remaining_final_overlap_conflicts.iloc[0]["horse"]
    == "Attention All (IRE)"
)

print("Final overlapping pedigree classification confirmed")
display(final_overlap_summary)

print("Remaining unresolved overlapping conflict")
remaining_final_overlap_conflicts

Final overlapping pedigree classification confirmed


,measure,value
0,temporally overlapping contradiction labels be...,18
1,resolved by verified Almutawakel equivalence,13
2,resolved by provisional terminal-I dam groups,2
3,resolved by provisional case-only dam group,1
4,resolved by provisional Ut-prefix sire group,1
5,remaining unresolved overlapping conflicts,1


Remaining unresolved overlapping conflict


,horse,sire_forms,dam_forms,damsire_forms,runner_rows,first_date,last_date,has_contradiction
0,Attention All (IRE),1,2,2,8,2022-05-10,2024-02-03,True


### Conclusion — Temporally overlapping pedigree contradictions

The structured contradiction analysis identified eighteen exact horse labels whose competing pedigree assertions overlapped in time.

After applying bounded source-internal classifications and captured external verification:

* thirteen labels were resolved by the verified `Almutawakel` damsire-equivalence group;
* two labels were resolved by provisional terminal-`I` dam-variant groups;
* one label was resolved by a provisional case-only dam group;
* one label was resolved by the bounded `Ut*` sire-prefix group;
* the final apparent conflict, `Attention All (IRE)`, was resolved through external verification.

#### `Attention All (IRE)`

Seven source rows identify the pedigree as:

* sire: `Westerner (GB)`;
* dam: `Moon Light Shadow I GB`;
* damsire: `Compton Place`.

One source row, recorded at Newcastle on 6 January 2024, instead identifies:

* sire: `Westerner (GB)`;
* dam: `Moonlight Shadow GB`;
* damsire: `Fountain Of Youth`.

The published result for that same Newcastle race identifies the pedigree as:

* sire: `Westerner`;
* dam: `Moon Light Shadow I`;
* damsire: `Compton Place`.

The following source appearance on 3 February 2024 also returns to that pedigree.

The Newcastle source row is therefore not evidence of a second horse identity or a legitimate competing pedigree. It is an isolated source pedigree defect affecting both the dam and damsire fields.

The governed interpretation for `Attention All (IRE)` is:

* sire: `Westerner`;
* dam: `Moon Light Shadow I`;
* damsire: `Compton Place`.

The raw Newcastle values must remain preserved for lineage, but:

* `Moonlight Shadow GB` must not be treated as a verified alternative dam;
* `Fountain Of Youth` must not be retained as a competing damsire;
* the affected source row should carry an externally contradicted pedigree status;
* the verified pedigree should be exposed only through a governed downstream reconciliation layer;
* the immutable source row must not be overwritten.

The eighteen temporally overlapping labels therefore partition into:

| classification                                      | horse labels | status                        |
| --------------------------------------------------- | -----------: | ----------------------------- |
| verified `Almutawakel` label equivalence            |           13 | governed                      |
| provisional terminal-`I` dam variation              |            2 | reviewable candidate          |
| provisional case-only dam variation                 |            1 | reviewable candidate          |
| bounded `Ut*` sire-prefix variation                 |            1 | governed for this exact pair  |
| externally verified isolated source pedigree defect |            1 | governed correction candidate |
| unresolved overlapping conflicts                    |            0 | none                          |

This supports the following source-governance rule:

> Temporally overlapping pedigree differences may be reconciled only through reversible structural parsing, a narrowly bounded source-label rule or captured external verification. Materially different assertions must remain unresolved unless external evidence identifies the correct pedigree.

Raw `horse`, `sire`, `dam` and `damsire` values must remain unchanged in every class.

This completes the investigation of the eighteen temporally overlapping contradiction labels. None remains an unresolved horse-identity conflict.


### Stage 6 — Temporally separated pedigree groups

Of the 368 exact horse labels that retained structured pedigree contradictions, 350 had pedigree groups whose observed date ranges did not overlap.

This is materially different from the overlapping cases.

A temporally separated change may represent:

* reuse of the same registered horse name after an earlier horse’s protected-name period ended;
* distinct real horses carrying the same display name and country suffix;
* an historical source correction;
* incomplete observations creating an artificial gap;
* or a genuine source pedigree defect.

The source-wide evidence already shows examples with:

* completely different sire, dam and damsire assertions;
* gaps of several years between pedigree groups;
* recorded ages restarting at a younger value;
* internally coherent age progression within each group.

Those patterns are consistent with exact source horse labels being reused for different real-world horses.

This stage profiles all 350 temporally separated labels and measures:

* number of coherent pedigree groups;
* gap between consecutive groups;
* age before and after the transition;
* whether sire, dam and damsire all change;
* whether recorded sex is compatible across groups;
* whether the later group begins with an apparent age reset.

The purpose is to determine whether the source can support provisional **horse-occurrence identities** without treating the raw horse label as a permanent natural key.

No real-world horse entities are merged or invented.


In [26]:
# Find existing DataFrames that could support Stage 6.
candidate_frames = []

global_items = list(globals().items())

for variable_name, value in global_items:
    if not isinstance(value, pd.DataFrame):
        continue

    columns = set(value.columns)

    relevant_columns = {
        "horse",
        "sire",
        "dam",
        "structured_dam",
        "damsire",
        "date",
        "first_date",
        "last_date",
        "age",
        "minimum_age",
        "maximum_age",
        "sex",
        "sex_values",
        "runner_rows",
    }

    matched_columns = sorted(
        columns.intersection(relevant_columns)
    )

    if {"horse", "sire", "damsire"}.issubset(columns):
        candidate_frames.append(
            {
                "variable_name": variable_name,
                "rows": len(value),
                "matched_columns": " | ".join(
                    matched_columns
                ),
            }
        )

candidate_frames = (
    pd.DataFrame(candidate_frames)
    .sort_values(
        ["rows", "variable_name"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Candidate pedigree DataFrames")
candidate_frames

Candidate pedigree DataFrames


,variable_name,rows,matched_columns
0,contradictory_assertion_rows,96404,age | dam | damsire | date | horse | sex | sire
1,contradictory_assertion_rows_with_dam_structure,96404,age | dam | damsire | date | horse | sex | sire
2,structured_pedigree_rows,96404,age | dam | damsire | date | horse | sex | sire
3,_12,13099,dam | damsire | first_date | horse | last_date...
4,assertion_group_preview,13099,dam | damsire | first_date | horse | last_date...
5,contradictory_assertion_groups,13099,dam | damsire | first_date | horse | last_date...
6,remaining_assertion_rows,4974,age | dam | damsire | date | horse | sex | sire
7,_16,741,damsire | first_date | horse | last_date | max...
8,structured_pedigree_group_preview,741,damsire | first_date | horse | last_date | max...
9,structured_pedigree_groups,741,damsire | first_date | horse | last_date | max...


In [27]:
print("structured_pedigree_groups columns")
print(structured_pedigree_groups.columns.tolist())

display(structured_pedigree_groups.head(10))

structured_pedigree_groups columns
['horse', 'sire', 'dam_structured_key', 'damsire', 'runner_rows', 'provisional_races', 'first_date', 'last_date', 'minimum_age', 'maximum_age', 'distinct_sexes', 'sex_values', 'raw_dam_labels', 'dam_suffix_formats', 'course_examples']


,horse,sire,dam_structured_key,damsire,runner_rows,provisional_races,first_date,last_date,minimum_age,maximum_age,distinct_sexes,sex_values,raw_dam_labels,dam_suffix_formats,course_examples
0,A La Prochaine (FR),Laverock (IRE),"(parsed_suffix, Necessaire, FR)",Garde Royale,14,14,2015-03-14,2016-06-12,5,6,1,M,Necessaire (FR),parenthesised,Auteuil (FR) | Enghien (FR)
1,A La Prochaine (FR),Lope De Vega (IRE),"(parsed_suffix, Margies Music, FR)",Hurricane Run,2,2,2025-10-25,2026-05-06,2,3,1,F,Margies Music (FR),parenthesised,Chester | Newbury
2,Accrington Stanley (GB),Outstrip (GB),"(parsed_suffix, Round Midnight, GB)",Paco Boy,2,2,2022-08-11,2022-09-11,4,4,1,G,Round Midnight GB,bare,Chepstow | Salisbury
3,Accrington Stanley (GB),Outstrip (GB),"(parsed_suffix, Round Midnight I, GB)",Paco Boy,28,28,2020-08-06,2022-07-30,2,4,1,G,Round Midnight I GB,bare,Bath | Chelmsford (AW) | Chepstow | Kempton (A...
4,Acoustic Ave (USA),Macleans Music (USA),"(parsed_suffix, Rock Ave Road, USA)",Street Boss,1,1,2023-07-16,2023-07-16,3,3,1,C,Rock Ave Road (USA),parenthesised,Saratoga (USA)
5,Acoustic Ave (USA),Macleans Music (USA),"(parsed_suffix, Rock Ave. Road, USA)",Street Boss,4,4,2025-12-06,2026-05-09,5,6,1,G,Rock Ave. Road (USA),parenthesised,Aqueduct
6,Al Daayen (FR),Joshua Tree (IRE),"(parsed_suffix, Get The Ring, FR)",Linamix,12,12,2019-01-29,2019-11-27,3,3,1,F,Get The Ring (FR),parenthesised,Brighton | Deauville (FR) | Lingfield | Lingfi...
7,Al Daayen (FR),Zelzal (FR),"(parsed_suffix, Nigwah, FR)",Montjeu,9,9,2023-11-05,2025-05-29,2,4,2,"C, G",Nigwah (FR),parenthesised,Chantilly (FR) | Deauville (FR) | Longchamp (F...
8,Alcala (FR),Turgeon (USA),"(parsed_suffix, Pail Mel, FR)",Sleeping Car,33,33,2015-10-21,2022-02-03,5,12,1,G,Pail Mel (FR),parenthesised,Ascot | Ayr | Cheltenham | Chepstow | Fontwell
9,Alcala (FR),Wootton Bassett (GB),"(parsed_suffix, Almeria, FR)",Shamardal,4,4,2024-04-29,2025-01-21,3,4,1,F,Almeria (FR),parenthesised,Cagnes-Sur-Mer (FR) | Chantilly (FR) | Deauvil...


In [28]:
# Profile transitions between temporally separated structured pedigree groups.

required_group_columns = {
    "horse",
    "sire",
    "dam_structured_key",
    "damsire",
    "runner_rows",
    "first_date",
    "last_date",
    "minimum_age",
    "maximum_age",
    "sex_values",
}

missing_group_columns = required_group_columns.difference(
    structured_pedigree_groups.columns
)

assert not missing_group_columns, (
    "structured_pedigree_groups is missing required columns: "
    f"{sorted(missing_group_columns)}"
)

ordered_structured_groups = (
    structured_pedigree_groups
    .copy()
    .sort_values(
        ["horse", "first_date", "last_date"],
        kind="stable",
    )
    .reset_index(drop=True)
)

for column in ["first_date", "last_date"]:
    ordered_structured_groups[column] = pd.to_datetime(
        ordered_structured_groups[column]
    )

# Compare each group with the latest end date of all preceding groups
# for the same exact horse label.
ordered_structured_groups["previous_latest_end"] = (
    ordered_structured_groups
    .groupby("horse")["last_date"]
    .transform(lambda values: values.cummax().shift())
)

ordered_structured_groups["overlaps_previous_group"] = (
    ordered_structured_groups["previous_latest_end"].notna()
    & ordered_structured_groups["first_date"].le(
        ordered_structured_groups["previous_latest_end"]
    )
)

derived_temporal_summary = (
    ordered_structured_groups
    .groupby("horse", as_index=False)
    .agg(
        pedigree_groups=("horse", "size"),
        runner_rows=("runner_rows", "sum"),
        first_date=("first_date", "min"),
        last_date=("last_date", "max"),
        overlapping_transitions=(
            "overlaps_previous_group",
            "sum",
        ),
    )
)

derived_temporal_summary["all_groups_temporally_separate"] = (
    derived_temporal_summary["pedigree_groups"].gt(1)
    & derived_temporal_summary[
        "overlapping_transitions"
    ].eq(0)
)

temporally_separated_horses = set(
    derived_temporal_summary.loc[
        derived_temporal_summary[
            "all_groups_temporally_separate"
        ],
        "horse",
    ]
)

assert len(temporally_separated_horses) == 350, (
    "Expected 350 temporally separated labels, found "
    f"{len(temporally_separated_horses)}"
)

separated_groups = (
    ordered_structured_groups.loc[
        ordered_structured_groups["horse"].isin(
            temporally_separated_horses
        )
    ]
    .copy()
    .sort_values(
        ["horse", "first_date", "last_date"],
        kind="stable",
    )
    .reset_index(drop=True)
)

separated_groups["group_number"] = (
    separated_groups.groupby("horse").cumcount() + 1
)

next_columns = {
    "first_date": "next_first_date",
    "sire": "next_sire",
    "dam_structured_key": "next_dam_structured_key",
    "damsire": "next_damsire",
    "minimum_age": "next_minimum_age",
    "maximum_age": "next_maximum_age",
    "sex_values": "next_sex_values",
}

for source_column, next_column in next_columns.items():
    separated_groups[next_column] = (
        separated_groups
        .groupby("horse")[source_column]
        .shift(-1)
    )

separated_transitions = (
    separated_groups.loc[
        separated_groups["next_first_date"].notna()
    ]
    .copy()
)

separated_transitions["gap_days"] = (
    separated_transitions["next_first_date"]
    - separated_transitions["last_date"]
).dt.days

separated_transitions["sire_changed"] = (
    separated_transitions["sire"].ne(
        separated_transitions["next_sire"]
    )
)

separated_transitions["dam_changed"] = (
    separated_transitions["dam_structured_key"].ne(
        separated_transitions["next_dam_structured_key"]
    )
)

separated_transitions["damsire_changed"] = (
    separated_transitions["damsire"].ne(
        separated_transitions["next_damsire"]
    )
)

separated_transitions["all_pedigree_components_changed"] = (
    separated_transitions[
        [
            "sire_changed",
            "dam_changed",
            "damsire_changed",
        ]
    ].all(axis=1)
)

separated_transitions["age_reset"] = (
    separated_transitions["next_minimum_age"].notna()
    & separated_transitions["maximum_age"].notna()
    & separated_transitions["next_minimum_age"].lt(
        separated_transitions["maximum_age"]
    )
)

separated_transition_summary = pd.DataFrame(
    [
        {
            "measure": "temporally separated horse labels",
            "value": len(temporally_separated_horses),
        },
        {
            "measure": "structured pedigree groups",
            "value": len(separated_groups),
        },
        {
            "measure": "transitions between pedigree groups",
            "value": len(separated_transitions),
        },
        {
            "measure": "all pedigree components changed",
            "value": int(
                separated_transitions[
                    "all_pedigree_components_changed"
                ].sum()
            ),
        },
        {
            "measure": "recorded age reset at later group",
            "value": int(
                separated_transitions["age_reset"].sum()
            ),
        },
        {
            "measure": "full pedigree change and age reset",
            "value": int(
                (
                    separated_transitions[
                        "all_pedigree_components_changed"
                    ]
                    & separated_transitions["age_reset"]
                ).sum()
            ),
        },
        {
            "measure": "median gap days",
            "value": float(
                separated_transitions["gap_days"].median()
            ),
        },
        {
            "measure": "maximum gap days",
            "value": int(
                separated_transitions["gap_days"].max()
            ),
        },
    ]
)

strong_reuse_candidates = (
    separated_transitions.loc[
        separated_transitions[
            "all_pedigree_components_changed"
        ]
        & separated_transitions["age_reset"],
        [
            "horse",
            "group_number",
            "sire",
            "dam_structured_key",
            "damsire",
            "last_date",
            "maximum_age",
            "sex_values",
            "next_sire",
            "next_dam_structured_key",
            "next_damsire",
            "next_first_date",
            "next_minimum_age",
            "next_sex_values",
            "gap_days",
        ],
    ]
    .sort_values(
        ["gap_days", "horse"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert separated_transitions["gap_days"].gt(0).all()

assert (
    separated_groups.groupby("horse").size().sub(1).sum()
    == len(separated_transitions)
)

print("Temporally separated pedigree transitions confirmed")
display(separated_transition_summary)

print("Strong exact-label reuse candidates")
strong_reuse_candidates.head(25)

Temporally separated pedigree transitions confirmed


,measure,value
0,temporally separated horse labels,350.0
1,structured pedigree groups,703.0
2,transitions between pedigree groups,353.0
3,all pedigree components changed,259.0
4,recorded age reset at later group,205.0
5,full pedigree change and age reset,204.0
6,median gap days,2443.0
7,maximum gap days,4095.0


Strong exact-label reuse candidates


,horse,group_number,sire,dam_structured_key,damsire,last_date,maximum_age,sex_values,next_sire,next_dam_structured_key,next_damsire,next_first_date,next_minimum_age,next_sex_values,gap_days
0,OGorman (GB),1,Sleeping Indian (GB),"(parsed_suffix, Harryana, GB)",Efisio,2015-03-06,6,G,Sergei Prokofiev (CAN),"(parsed_suffix, Shirin Jaan, GB)",Sepoy,2026-04-21,2.0,C,4064
1,Hanohano (JPN),1,Admire Cozzene (JPN),"(parsed_suffix, Prominent Cut, JPN)",Tony Bin,2015-03-01,7,H,Maurice (JPN),"(parsed_suffix, Erimo Haruka, JPN)",Narita Top Road,2026-04-11,3.0,F,4059
2,Police Gazette (USA),1,Giants Causeway (USA),"(parsed_suffix, Black Speck, USA)",Arch,2015-04-01,6,G,War Front (USA),"(parsed_suffix, Akatea, IRE)",Shamardal,2026-04-10,3.0,C,4027
3,Seedling (GB),1,Cockney Rebel (IRE),"(parsed_suffix, Unseeded, GB)",Unfuwain,2015-04-11,6,G,Planteur (IRE),"(parsed_suffix, Sixtys Belle, GB)",Gold Well,2026-04-04,4.0,F,4011
4,Tavarua (IRE),1,Intense Focus (USA),"(parsed_suffix, Cloud Break, GB)",Dansili,2015-04-25,4,F,Ravens Pass (USA),"(parsed_suffix, Sannkala, FR)",Medicean,2026-04-08,3.0,F,4001
5,Found It (IRE),1,Heron Island (IRE),"(parsed_suffix, Iseefaith, IRE)",Perugino,2015-05-29,8,M,Maxios (GB),"(parsed_suffix, Findaway, IRE)",Westerner,2026-05-02,5.0,M,3991
6,Fireball (AUS),1,Beautiful Crown (USA),"(parsed_suffix, Hotspurs, AUS)",Flying Spur,2015-04-29,6,G,Snitzel (AUS),"(parsed_suffix, Advance Party, AUS)",Charge Forward,2026-02-07,2.0,C,3937
7,Pont Marie (FR),1,Great Journey (JPN),"(parsed_suffix, Cite Fleurie, IRE)",Mark Of Esteem,2015-01-10,5,H,Elarqam (GB),"(parsed_suffix, Ponte Sanangelo, FR)",Authorized,2025-08-31,3.0,F,3886
8,Vatican (AUS),1,Gods Own (AUS),"(parsed_suffix, Our Sistine, AUS)",Peintre Celebre,2015-05-02,6,G,Wootton Bassett (GB),"(parsed_suffix, Egyptian Missile, AUS)",Smart Missile,2025-12-13,2.0,C,3878
9,Synchronicity (IRE),1,High Chaparral (IRE),"(parsed_suffix, Sea Of Time, USA)",Gilded Time,2015-09-30,6,G,Night Of Thunder (IRE),"(parsed_suffix, Syndicate, GB)",Dansili,2026-04-17,3.0,F,3852


### Interim interpretation — Exact horse labels are reused across real-world horses

The temporally separated pedigree analysis provides direct source-wide evidence that an exact `horse` label cannot serve as a permanent natural key.

Across 350 exact horse labels:

* 703 structured pedigree groups were observed;
* 353 transitions occurred between consecutive groups;
* 259 transitions changed sire, structured dam and damsire together;
* 205 transitions restarted at a younger recorded age;
* 204 transitions combined a complete pedigree change with an age reset;
* the median gap between groups was 2,443 days;
* the maximum gap was 4,095 days.

The strongest cases cannot represent pedigree corrections within one horse’s life.

Examples include:

* `OGorman (GB)`;
* `Hanohano (JPN)`;
* `Police Gazette (USA)`;
* `Seedling (GB)`;
* `Fireball (AUS)`;
* `Australia Day (IRE)`;
* `Chamonix (IRE)`.

In these cases, the later group has:

* a different sire;
* a different dam;
* a different damsire;
* a younger recorded age;
* and often a different sex category.

The evidence therefore supports the conclusion that the source reuses identical complete horse labels, including the country suffix, for different real-world horses at different periods.

The governed identity rule must consequently be:

> An exact raw horse label identifies a source-reported name, not a permanently unique horse entity.

Neither of the following is safe as a permanent horse key:

* parsed display name alone;
* complete raw label including country suffix.

A source-level horse occurrence must instead be distinguished using a coherent evidence bundle that includes:

* raw horse label;
* structured pedigree assertion;
* observed date range;
* age progression;
* sex history where consistent;
* race and source-row lineage;
* identity confidence and review status.

A later authoritative horse entity may link one or more occurrence groups only when a governing authority identifier or bounded external verification supports that relationship.

The 204 complete-pedigree-change and age-reset transitions are strong label-reuse cases. The remaining transitions require further classification because they may include partial source corrections, label variants, missing ages or less complete evidence.


In [29]:
# Classify all temporally separated transitions by pedigree-change and age evidence.

separated_transitions["pedigree_components_changed"] = (
    separated_transitions[
        [
            "sire_changed",
            "dam_changed",
            "damsire_changed",
        ]
    ]
    .sum(axis=1)
)

separated_transitions["age_evidence"] = "age_not_comparable"

age_comparable = (
    separated_transitions["next_minimum_age"].notna()
    & separated_transitions["maximum_age"].notna()
)

separated_transitions.loc[
    age_comparable
    & separated_transitions["age_reset"],
    "age_evidence",
] = "younger_age_reset"

separated_transitions.loc[
    age_comparable
    & ~separated_transitions["age_reset"],
    "age_evidence",
] = "no_younger_age_reset"

transition_classification = (
    separated_transitions
    .groupby(
        [
            "pedigree_components_changed",
            "age_evidence",
        ],
        as_index=False,
    )
    .agg(
        transitions=("horse", "size"),
        horse_labels=("horse", "nunique"),
        median_gap_days=("gap_days", "median"),
        minimum_gap_days=("gap_days", "min"),
        maximum_gap_days=("gap_days", "max"),
    )
    .sort_values(
        [
            "pedigree_components_changed",
            "age_evidence",
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

non_strong_transition_examples = (
    separated_transitions.loc[
        ~(
            separated_transitions[
                "all_pedigree_components_changed"
            ]
            & separated_transitions["age_reset"]
        ),
        [
            "horse",
            "group_number",
            "sire",
            "dam_structured_key",
            "damsire",
            "last_date",
            "maximum_age",
            "sex_values",
            "next_sire",
            "next_dam_structured_key",
            "next_damsire",
            "next_first_date",
            "next_minimum_age",
            "next_sex_values",
            "gap_days",
            "pedigree_components_changed",
            "age_evidence",
        ],
    ]
    .sort_values(
        [
            "pedigree_components_changed",
            "gap_days",
            "horse",
        ],
        ascending=[False, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert transition_classification["transitions"].sum() == 353

assert len(non_strong_transition_examples) == 149

print("Temporally separated transition classes confirmed")
display(transition_classification)

print("Examples outside the strongest exact-label reuse class")
non_strong_transition_examples.head(30)

Temporally separated transition classes confirmed


,pedigree_components_changed,age_evidence,transitions,horse_labels,median_gap_days,minimum_gap_days,maximum_gap_days
0,3,no_younger_age_reset,55,55,2735.0,18,4095
1,3,younger_age_reset,204,204,2651.5,139,4064
2,2,no_younger_age_reset,5,5,49.0,29,2364
3,2,younger_age_reset,1,1,2586.0,2586,2586
4,1,no_younger_age_reset,88,86,217.5,9,2707


Examples outside the strongest exact-label reuse class


,horse,group_number,sire,dam_structured_key,damsire,last_date,maximum_age,sex_values,next_sire,next_dam_structured_key,next_damsire,next_first_date,next_minimum_age,next_sex_values,gap_days,pedigree_components_changed,age_evidence
0,Steel (USA),1,Mineshaft (USA),"(parsed_suffix, Picketline, USA)",Street Cry,2015-01-17,3,G,Tapit (USA),"(parsed_suffix, Snuggs And Kisses, USA)",Soto,2026-04-04,3.0,C,4095,3,no_younger_age_reset
1,Teen Spirit (FR),1,Antarctique (IRE),"(parsed_suffix, Bahia Belle, FR)",Sendawar,2015-07-02,4,F,Nirvana Du Berlais (FR),"(parsed_suffix, La Coquette, GER)",Kamsin,2026-02-23,4.0,G,3889,3,no_younger_age_reset
2,Attack (USA),1,Quality Road (USA),"(parsed_suffix, Seattle Tac, USA)",Seattle Slew,2015-04-17,3,R,Munnings (USA),"(parsed_suffix, Ammannati, IRE)",Galileo,2025-09-14,3.0,G,3803,3,no_younger_age_reset
3,Caldera (USA),1,Arch (USA),"(parsed_suffix, The Legend Grows, USA)",Siphon,2015-01-17,3,C,Liams Map (USA),"(parsed_suffix, Send Me On My Way, USA)",Tiznow,2025-02-16,3.0,C,3683,3,no_younger_age_reset
4,Roman (FR),1,Dylan Thomas (IRE),"(parsed_suffix, Happy Lodge, IRE)",Grand Lodge,2015-05-22,3,G,Starspangledbanner (AUS),"(parsed_suffix, Rosea, GER)",Nathaniel,2025-06-12,3.0,C,3674,3,no_younger_age_reset
5,Kingdom Of Alba (IRE),1,The Carbon Unit (USA),"(parsed_suffix, Crackling Rosie, IRE)",Dr Fong,2015-08-27,3,"C, G",Cotai Glory (GB),"(parsed_suffix, Alba Verde, GB)",Verglas,2025-09-06,3.0,G,3663,3,no_younger_age_reset
6,Iron Dome (USA),1,Silver Tree (USA),"(parsed_suffix, Final Assault, USA)",Evening Kris,2015-11-21,2,C,Into Mischief (USA),"(parsed_suffix, Speightful Affair, CAN)",Speightstown,2025-09-29,3.0,C,3600,3,no_younger_age_reset
7,Instant Replay (USA),1,Lemon Drop Kid (USA),"(parsed_suffix, Run Kate Run, USA)",Cherokee Run,2015-06-21,3,C,Maximum Security (USA),"(parsed_suffix, Academy Gal, USA)",Medaglia dOro,2025-03-22,3.0,C,3562,3,no_younger_age_reset
8,Medici (USA),1,Curlin (USA),"(parsed_suffix, Mrs Williams, GER)",Monsun,2016-06-26,3,C,Into Mischief (USA),"(parsed_suffix, Avenge, USA)",War Front,2026-03-21,3.0,C,3555,3,no_younger_age_reset
9,Boxing Clever (IRE),1,Teofilo (IRE),"(parsed_suffix, Sassy Gal, IRE)",Kings Best,2016-04-15,4,G,Shirocco (GER),"(parsed_suffix, Floral Fantasy, IRE)",Flemensfirth,2025-11-27,4.0,G,3513,3,no_younger_age_reset


### Stage 6a — Age continuity across separated pedigree groups

The initial age-reset test identified 204 transitions where:

* sire, dam and damsire all changed;
* and the later pedigree group began at a younger recorded age.

A further 55 full-pedigree transitions did not meet that narrow definition.

Inspection shows that many of these are still clear horse-label reuse cases. The later group may begin at the same age or an older age than the earlier group ended, but only after a gap of several years.

For example, an earlier horse recorded as age three in 2015 and a later horse recorded as age three in 2026 cannot represent one continuous horse history. If the records belonged to the same horse, its recorded age should have increased approximately with the elapsed calendar years.

This stage therefore compares:

* the elapsed time between pedigree groups;
* the earlier group’s maximum recorded age;
* the later group’s minimum recorded age;
* the approximate age expected under one continuous identity.

The calculation is diagnostic rather than authoritative. Recorded racing ages can be affected by jurisdictional conventions and incomplete observation periods. However, a large downward discrepancy between elapsed time and recorded age progression provides strong evidence that the exact horse label was reused.

No permanent horse identity is created from this calculation alone.


In [30]:
# Test whether recorded age progression is compatible with one continuous horse identity.

separated_transitions["elapsed_years_approx"] = (
    separated_transitions["gap_days"] / 365.25
)

separated_transitions["observed_age_change"] = (
    separated_transitions["next_minimum_age"]
    - separated_transitions["maximum_age"]
)

separated_transitions["expected_later_age_approx"] = (
    separated_transitions["maximum_age"]
    + separated_transitions["elapsed_years_approx"]
)

separated_transitions["age_continuity_shortfall"] = (
    separated_transitions["expected_later_age_approx"]
    - separated_transitions["next_minimum_age"]
)

# Allow a generous two-year tolerance for:
# - incomplete annual observations;
# - differing birthday conventions;
# - jurisdictional age rules;
# - use of group minimum and maximum ages.
separated_transitions["age_continuity_implausible"] = (
    separated_transitions["maximum_age"].notna()
    & separated_transitions["next_minimum_age"].notna()
    & separated_transitions["age_continuity_shortfall"].gt(2)
)

full_pedigree_transitions = separated_transitions.loc[
    separated_transitions[
        "all_pedigree_components_changed"
    ]
].copy()

full_pedigree_age_summary = pd.DataFrame(
    [
        {
            "measure": "full-pedigree transitions",
            "value": len(full_pedigree_transitions),
        },
        {
            "measure": "with comparable age evidence",
            "value": int(
                (
                    full_pedigree_transitions[
                        "maximum_age"
                    ].notna()
                    & full_pedigree_transitions[
                        "next_minimum_age"
                    ].notna()
                ).sum()
            ),
        },
        {
            "measure": "narrow younger-age resets",
            "value": int(
                full_pedigree_transitions[
                    "age_reset"
                ].sum()
            ),
        },
        {
            "measure": "implausible continuous-age progression",
            "value": int(
                full_pedigree_transitions[
                    "age_continuity_implausible"
                ].sum()
            ),
        },
        {
            "measure": "not implausible under two-year tolerance",
            "value": int(
                (
                    ~full_pedigree_transitions[
                        "age_continuity_implausible"
                    ]
                ).sum()
            ),
        },
        {
            "measure": "median age-continuity shortfall",
            "value": float(
                full_pedigree_transitions[
                    "age_continuity_shortfall"
                ].median()
            ),
        },
    ]
)

full_pedigree_age_exceptions = (
    full_pedigree_transitions.loc[
        ~full_pedigree_transitions[
            "age_continuity_implausible"
        ],
        [
            "horse",
            "sire",
            "dam_structured_key",
            "damsire",
            "last_date",
            "maximum_age",
            "next_sire",
            "next_dam_structured_key",
            "next_damsire",
            "next_first_date",
            "next_minimum_age",
            "gap_days",
            "elapsed_years_approx",
            "observed_age_change",
            "age_continuity_shortfall",
        ],
    ]
    .sort_values(
        ["gap_days", "horse"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(full_pedigree_transitions) == 259

print("Full-pedigree transition age continuity confirmed")
display(full_pedigree_age_summary)

print("Full-pedigree transitions not rejected by the two-year age tolerance")
full_pedigree_age_exceptions

Full-pedigree transition age continuity confirmed


,measure,value
0,full-pedigree transitions,259.000000
1,with comparable age evidence,259.000000
2,narrow younger-age resets,204.000000
3,implausible continuous-age progression,257.000000
4,not implausible under two-year tolerance,2.000000
5,median age-continuity shortfall,9.494867


Full-pedigree transitions not rejected by the two-year age tolerance


,horse,sire,dam_structured_key,damsire,last_date,maximum_age,next_sire,next_dam_structured_key,next_damsire,next_first_date,next_minimum_age,gap_days,elapsed_years_approx,observed_age_change,age_continuity_shortfall
0,Forest King (AUS),Rubick (AUS),"(parsed_suffix, Lady Of War, AUS)",Charge Forward,2025-10-26,3,Tiger Of Malay (AUS),"(parsed_suffix, Tibrogargan Miss, AUS)",Monashee Mountain,2026-03-14,2.0,139,0.380561,-1.0,1.380561
1,Felix Felicis (FR),Affinisea (IRE),"(parsed_suffix, Just Eile, IRE)",Presenting,2024-12-30,4,Olympic Glory (IRE),"(parsed_suffix, Sorina, FR)",Le Havre,2025-01-17,5.0,18,0.049281,1.0,-0.950719


### Interim interpretation — Full-pedigree transitions overwhelmingly indicate label reuse

All 259 temporally separated transitions with changes to sire, structured dam and damsire had comparable recorded-age evidence.

Using a deliberately generous two-year tolerance:

* 257 transitions had age progression incompatible with one continuous horse identity;
* only two transitions were not rejected by that diagnostic;
* the median age-continuity shortfall was approximately 9.5 years.

The 257 cases provide strong source-wide evidence of exact-label reuse.

In these transitions:

* the complete pedigree changes;
* the elapsed calendar period is inconsistent with the later recorded age;
* and the pedigree groups do not overlap in the observed source dates.

They should therefore be treated as distinct provisional horse occurrences sharing one exact source `horse` label.

The two exceptions are:

* `Forest King (AUS)`, with a 139-day gap;
* `Felix Felicis (FR)`, with an 18-day gap.

Their short gaps make them qualitatively different from the long-gap reuse cases. A complete pedigree change over such a short interval is more likely to represent:

* an isolated source pedigree defect;
* a horse-label assignment error;
* a source correction;
* or two distinct horses whose observations happen to occur close together.

The grouped summaries alone cannot distinguish those possibilities.

These two labels therefore require row-level chronology inspection before classification. Until that inspection is complete, neither transition should be automatically reconciled or treated as a governed identity split.


In [31]:
# Inspect the two short-gap full-pedigree exceptions at runner-row level.

full_pedigree_exception_horses = {
    "Forest King (AUS)",
    "Felix Felicis (FR)",
}

full_pedigree_exception_rows = (
    structured_pedigree_rows.loc[
        structured_pedigree_rows["horse"].isin(
            full_pedigree_exception_horses
        ),
        [
            "horse",
            "date",
            "course",
            "off",
            "age",
            "sex",
            "sire",
            "dam",
            "damsire",
        ],
    ]
    .copy()
)

full_pedigree_exception_rows["date"] = pd.to_datetime(
    full_pedigree_exception_rows["date"]
)

full_pedigree_exception_rows = (
    full_pedigree_exception_rows
    .sort_values(
        ["horse", "date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

exception_assertion_chronology = (
    full_pedigree_exception_rows
    .groupby(
        [
            "horse",
            "sire",
            "dam",
            "damsire",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        provisional_races=(
            "date",
            "size",
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_age=("age", "min"),
        maximum_age=("age", "max"),
        sex_values=(
            "sex",
            lambda values: ", ".join(
                sorted(
                    {
                        str(value)
                        for value in values.dropna()
                    }
                )
            ),
        ),
        course_examples=(
            "course",
            lambda values: " | ".join(
                list(dict.fromkeys(values.dropna()))[:8]
            ),
        ),
    )
    .sort_values(
        ["horse", "first_date", "last_date"],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert set(
    full_pedigree_exception_rows["horse"].unique()
) == full_pedigree_exception_horses

print("Short-gap full-pedigree exception chronology")
display(exception_assertion_chronology)

print("Complete runner-row chronology")
full_pedigree_exception_rows

Short-gap full-pedigree exception chronology


,horse,sire,dam,damsire,runner_rows,provisional_races,first_date,last_date,minimum_age,maximum_age,sex_values,course_examples
0,Felix Felicis (FR),Affinisea (IRE),Just Eile (IRE),Presenting,3,3,2024-10-04,2024-12-30,4,4,G,Saint-Cloud (FR) | Chantilly (FR)
1,Felix Felicis (FR),Olympic Glory (IRE),Sorina (FR),Le Havre,9,9,2025-01-17,2026-02-25,5,6,G,Deauville (FR) | Saint-Cloud (FR) | Deauville ...
2,Forest King (AUS),Rubick (AUS),Lady Of War (AUS),Charge Forward,1,1,2025-10-26,2025-10-26,3,3,G,Sha Tin
3,Forest King (AUS),Tiger Of Malay (AUS),Tibrogargan Miss (AUS),Monashee Mountain,1,1,2026-03-14,2026-03-14,2,2,G,Rosehill


Complete runner-row chronology


,horse,date,course,off,age,sex,sire,dam,damsire
0,Felix Felicis (FR),2024-10-04,Saint-Cloud (FR),4:45,4,G,Affinisea (IRE),Just Eile (IRE),Presenting
1,Felix Felicis (FR),2024-11-22,Chantilly (FR),1:15,4,G,Affinisea (IRE),Just Eile (IRE),Presenting
2,Felix Felicis (FR),2024-12-30,Chantilly (FR),1:12,4,G,Affinisea (IRE),Just Eile (IRE),Presenting
3,Felix Felicis (FR),2025-01-17,Deauville (FR),6:33,5,G,Olympic Glory (IRE),Sorina (FR),Le Havre
4,Felix Felicis (FR),2025-02-08,Deauville (FR),4:17,5,G,Olympic Glory (IRE),Sorina (FR),Le Havre
5,Felix Felicis (FR),2025-03-06,Saint-Cloud (FR),4:25,5,G,Olympic Glory (IRE),Sorina (FR),Le Havre
6,Felix Felicis (FR),2025-11-25,Deauville,16:25,5,G,Olympic Glory (IRE),Sorina (FR),Le Havre
7,Felix Felicis (FR),2025-12-29,Deauville,19:14,5,G,Olympic Glory (IRE),Sorina (FR),Le Havre
8,Felix Felicis (FR),2026-01-08,Deauville,16:07,6,G,Olympic Glory (IRE),Sorina (FR),Le Havre
9,Felix Felicis (FR),2026-01-22,Deauville,16:42,6,G,Olympic Glory (IRE),Sorina (FR),Le Havre


### Conclusion — Full-pedigree transitions and exact-label reuse

The 259 temporally separated transitions in which sire, structured dam and damsire all changed were tested against recorded-age continuity.

Using a deliberately generous two-year tolerance:

* 257 transitions had age progression incompatible with one continuous horse identity;
* the median age-continuity shortfall was approximately 9.5 years;
* only `Forest King (AUS)` and `Felix Felicis (FR)` required external resolution.

#### `Forest King (AUS)`

The source contains:

1. a three-year-old gelding by `Rubick (AUS)`, out of `Lady Of War (AUS)`, by `Charge Forward`, observed at Sha Tin on 26 October 2025; and
2. a two-year-old gelding by `Tiger Of Malay (AUS)`, out of `Tibrogargan Miss (AUS)`, by `Monashee Mountain`, observed at Rosehill on 14 March 2026.

External authority evidence confirms that these are two different real-world horses:

* the Hong Kong Jockey Club identifies its `FOREST KING` as the Australian gelding by Rubick out of Lady Of War;
* Racing Australia identifies the later `FOREST KING` as the Australian two-year-old by Tiger Of Malay out of Tibrogargan Miss.

This is a confirmed exact-label collision. The two source occurrence groups must remain separate and must not share a permanent horse identifier merely because their complete raw `horse` labels are identical.

#### `Felix Felicis (FR)`

The source initially appears to contain two pedigree groups separated by eighteen days:

1. three 2024 rows carrying `Affinisea (IRE) — Just Eile (IRE) — Presenting`;
2. subsequent rows carrying `Olympic Glory (IRE) — Sorina (FR) — Le Havre`.

External form and profile evidence identifies one gelding, foaled on 13 April 2020, by Olympic Glory out of Sorina, and includes the same October, November and December 2024 races found in the first source group.

The three earlier pedigree assertions are therefore isolated source pedigree defects. They are not evidence of a second horse or legitimate exact-label reuse.

The governed interpretation is:

* horse: `Felix Felicis (FR)`;
* sire: `Olympic Glory (IRE)`;
* dam: `Sorina (FR)`;
* damsire: `Le Havre`;
* affected raw rows: 4 October, 22 November and 30 December 2024.

The raw defective pedigree values must remain preserved for lineage, while the corrected interpretation may be exposed only through a governed reconciliation layer with external provenance.

#### Final classification

The 259 complete-pedigree transitions therefore divide into:

| classification                                                    | transitions | interpretation                             |
| ----------------------------------------------------------------- | ----------: | ------------------------------------------ |
| strong exact-label reuse from pedigree and age evidence           |         257 | distinct provisional horse occurrences     |
| externally confirmed exact-label collision: `Forest King (AUS)`   |           1 | distinct real-world horses                 |
| externally confirmed source pedigree defect: `Felix Felicis (FR)` |           1 | one horse with three defective source rows |

The full-pedigree branch therefore establishes:

> All 258 genuine full-pedigree identity transitions represent distinct horse occurrences sharing an exact raw source label. The remaining transition is an externally confirmed pedigree defect rather than an identity split.

This provides decisive evidence that even the complete source label, including its country suffix, is not a permanent natural horse key.


### Stage 6b — Partial-pedigree transitions

After separating the 259 complete-pedigree transitions, 94 temporally separated transitions remain:

* six change two pedigree components;
* 88 change only one pedigree component.

These cases cannot automatically be treated as horse-label reuse.

A partial change may instead represent:

* punctuation, spacing or terminal-`I` label variation;
* a source correction to one pedigree field;
* an isolated erroneous assertion;
* missing or inconsistent country suffix parsing;
* or a genuine collision where part of the pedigree happens to match.

This stage identifies exactly which pedigree components changed and profiles the affected labels before any reconciliation rule is applied.

Raw values remain authoritative source evidence and must not be overwritten.


In [32]:
# Profile temporally separated transitions with only one or two pedigree changes.

partial_pedigree_transitions = (
    separated_transitions.loc[
        separated_transitions[
            "pedigree_components_changed"
        ].isin([1, 2])
    ]
    .copy()
)

partial_pedigree_transitions["change_pattern"] = (
    partial_pedigree_transitions[
        [
            "sire_changed",
            "dam_changed",
            "damsire_changed",
        ]
    ]
    .rename(
        columns={
            "sire_changed": "sire",
            "dam_changed": "dam",
            "damsire_changed": "damsire",
        }
    )
    .apply(
        lambda row: " + ".join(
            component
            for component, changed in row.items()
            if changed
        ),
        axis=1,
    )
)

partial_transition_summary = (
    partial_pedigree_transitions
    .groupby(
        [
            "pedigree_components_changed",
            "change_pattern",
        ],
        as_index=False,
    )
    .agg(
        transitions=("horse", "size"),
        horse_labels=("horse", "nunique"),
        median_gap_days=("gap_days", "median"),
        minimum_gap_days=("gap_days", "min"),
        maximum_gap_days=("gap_days", "max"),
    )
    .sort_values(
        [
            "pedigree_components_changed",
            "transitions",
            "change_pattern",
        ],
        ascending=[False, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

partial_transition_examples = (
    partial_pedigree_transitions.loc[
        :,
        [
            "horse",
            "pedigree_components_changed",
            "change_pattern",
            "sire",
            "dam_structured_key",
            "damsire",
            "last_date",
            "maximum_age",
            "next_sire",
            "next_dam_structured_key",
            "next_damsire",
            "next_first_date",
            "next_minimum_age",
            "gap_days",
        ],
    ]
    .sort_values(
        [
            "pedigree_components_changed",
            "change_pattern",
            "gap_days",
            "horse",
        ],
        ascending=[False, True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(partial_pedigree_transitions) == 94
assert partial_transition_summary["transitions"].sum() == 94

print("Partial-pedigree transition patterns")
display(partial_transition_summary)

print("Partial-pedigree transition examples")
partial_transition_examples.head(40)

Partial-pedigree transition patterns


,pedigree_components_changed,change_pattern,transitions,horse_labels,median_gap_days,minimum_gap_days,maximum_gap_days
0,2,dam + damsire,6,6,95.0,29,2586
1,1,damsire,40,38,231.5,9,1034
2,1,dam,39,39,178.0,12,1236
3,1,sire,9,9,260.0,46,2707


Partial-pedigree transition examples


,horse,pedigree_components_changed,change_pattern,sire,dam_structured_key,damsire,last_date,maximum_age,next_sire,next_dam_structured_key,next_damsire,next_first_date,next_minimum_age,gap_days
0,Lyneham (FR),2,dam + damsire,Wootton Bassett (GB),"(parsed_suffix, Uklanie, FR)",Dalakhani,2018-09-15,3,Wootton Bassett (GB),"(parsed_suffix, Llanita, GB)",Rock Of Gibraltar,2025-10-14,2.0,2586
1,Marakan (IRE),2,dam + damsire,Arakan (USA),"(parsed_suffix, Templemartin Glen, IRE)",Anshan,2016-05-19,5,Arakan (USA),"(parsed_suffix, Goodthyne Miss, IRE)",Jolly Jake,2022-11-08,6.0,2364
2,Grand Sonata (USA),2,dam + damsire,Medaglia dOro (USA),"(parsed_suffix, A P Sonata, USA)",A P Indy,2025-10-10,6,Medaglia dOro (USA),"(parsed_suffix, A. P. Sonata, USA)",A.P. Indy,2026-02-28,7.0,141
3,New President (FR),2,dam + damsire,Sinndar (IRE),"(parsed_suffix, Sun Song I, FR)",Dr Fong,2025-09-27,8,Sinndar (IRE),"(parsed_suffix, Sun Song II, FR)",,2025-11-15,8.0,49
4,Diamond Tipp (IRE),2,dam + damsire,Diamond Boy (FR),"(parsed_suffix, Sound Out, IRE)",Great Palm,2024-07-05,4,Diamond Boy (FR),"(parsed_suffix, Soundout, IRE)",Oscar,2024-08-18,4.0,44
5,Colwyn Bay (FR),2,dam + damsire,Falco (USA),"(parsed_suffix, Eudora, IRE)",More Than Ready,2026-04-02,7,Falco (USA),"(parsed_suffix, Eudora I, IRE)",Kings Best,2026-05-01,7.0,29
6,Time Quest (GB),1,dam,Time Test (GB),"(parsed_suffix, Rainbows Edge, GB)",Rainbow Quest,2022-06-17,3,Time Test (GB),"(parsed_suffix, Rainbows Edge I, GB)",Rainbow Quest,2025-11-04,6.0,1236
7,Acoustic Ave (USA),1,dam,Macleans Music (USA),"(parsed_suffix, Rock Ave Road, USA)",Street Boss,2023-07-16,3,Macleans Music (USA),"(parsed_suffix, Rock Ave. Road, USA)",Street Boss,2025-12-06,5.0,874
8,Hurricane (FR),1,dam,Vale Of York (IRE),"(parsed_suffix, Resplendence, USA)",Elusive Quality,2024-01-19,5,Vale Of York (IRE),"(parsed_suffix, Resplendence I, USA)",Elusive Quality,2026-04-23,7.0,825
9,Theres Claude (GB),1,dam,Nathaniel (IRE),"(parsed_suffix, Storyland, USA)",Menifee,2024-02-09,6,Nathaniel (IRE),"(parsed_suffix, Storyland I, USA)",Menifee,2025-12-22,7.0,682


### Stage 6c — Structural comparison of partial-pedigree changes

The 94 temporally separated partial-pedigree transitions consist of:

* six `dam + damsire` changes;
* 40 damsire-only changes;
* 39 dam-only changes;
* nine sire-only changes.

The examples show that many apparent contradictions are likely source-label variants rather than pedigree changes.

Observed patterns include:

* terminal Roman-numeral additions such as `Storyland` and `Storyland I`;
* punctuation differences such as `Rock Ave Road` and `Rock Ave. Road`;
* spacing differences such as `Ticker TapeI` and `Ticker Tape I`;
* capitalization differences;
* minor apostrophe variation;
* and country-suffix differences.

These patterns must not be collapsed through unrestricted name normalization. Instead, each changed component is assigned a diagnostic relationship describing the narrowest reversible transformation that makes the labels match.

The diagnostic classes are:

* case or whitespace variation;
* punctuation or spacing variation;
* terminal-`I` variation;
* country-suffix difference;
* missing value;
* material difference.

These classes identify reviewable source-label families. They do not by themselves establish real-world entity equivalence.


In [33]:
import re


def compact_alphanumeric(value):
    """Casefold and remove all non-alphanumeric characters."""
    if pd.isna(value):
        return None

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).casefold(),
    )


def whitespace_casefold(value):
    """Normalize case and repeated whitespace only."""
    if pd.isna(value):
        return None

    return " ".join(str(value).casefold().split())


def remove_terminal_i(value):
    """
    Remove one terminal Roman-numeral I after punctuation and spacing
    have been compacted.

    This is diagnostic only and does not create a canonical label.
    """
    compact = compact_alphanumeric(value)

    if compact is None:
        return None

    if compact.endswith("i"):
        return compact[:-1]

    return compact


def label_relationship(left, right):
    """Describe the narrowest observed relationship between two labels."""
    if pd.isna(left) and pd.isna(right):
        return "both_missing"

    if pd.isna(left) or pd.isna(right):
        return "one_missing"

    left_text = str(left)
    right_text = str(right)

    if left_text == right_text:
        return "exact"

    if whitespace_casefold(left_text) == whitespace_casefold(right_text):
        return "case_or_whitespace"

    left_compact = compact_alphanumeric(left_text)
    right_compact = compact_alphanumeric(right_text)

    if left_compact == right_compact:
        return "punctuation_or_spacing"

    if (
        remove_terminal_i(left_text) == right_compact
        or left_compact == remove_terminal_i(right_text)
    ):
        return "terminal_i_variant"

    return "material"


def unpack_dam_structured_key(value):
    """
    Return parsed dam name and country from the structured tuple.

    Expected form:
        ('parsed_suffix', display_name, country)
    """
    if not isinstance(value, tuple) or len(value) < 3:
        return pd.Series(
            {
                "dam_name": None,
                "dam_country": None,
            }
        )

    return pd.Series(
        {
            "dam_name": value[1],
            "dam_country": value[2],
        }
    )


partial_relationships = partial_pedigree_transitions.copy()

current_dam_parts = (
    partial_relationships["dam_structured_key"]
    .apply(unpack_dam_structured_key)
    .add_prefix("current_")
)

next_dam_parts = (
    partial_relationships["next_dam_structured_key"]
    .apply(unpack_dam_structured_key)
    .add_prefix("next_")
)

partial_relationships = pd.concat(
    [
        partial_relationships.reset_index(drop=True),
        current_dam_parts.reset_index(drop=True),
        next_dam_parts.reset_index(drop=True),
    ],
    axis=1,
)

partial_relationships["sire_relationship"] = (
    partial_relationships.apply(
        lambda row: label_relationship(
            row["sire"],
            row["next_sire"],
        ),
        axis=1,
    )
)

partial_relationships["dam_name_relationship"] = (
    partial_relationships.apply(
        lambda row: label_relationship(
            row["current_dam_name"],
            row["next_dam_name"],
        ),
        axis=1,
    )
)

partial_relationships["dam_country_relationship"] = (
    partial_relationships.apply(
        lambda row: (
            "exact"
            if row["current_dam_country"]
            == row["next_dam_country"]
            else "country_suffix_difference"
        ),
        axis=1,
    )
)

partial_relationships["damsire_relationship"] = (
    partial_relationships.apply(
        lambda row: label_relationship(
            row["damsire"],
            row["next_damsire"],
        ),
        axis=1,
    )
)

partial_relationship_summary = (
    partial_relationships
    .groupby(
        [
            "change_pattern",
            "sire_relationship",
            "dam_name_relationship",
            "dam_country_relationship",
            "damsire_relationship",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        transitions=("horse", "size"),
        horse_labels=("horse", "nunique"),
        median_gap_days=("gap_days", "median"),
    )
    .sort_values(
        [
            "transitions",
            "change_pattern",
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

material_partial_transitions = (
    partial_relationships.loc[
        (
            partial_relationships["sire_changed"]
            & partial_relationships[
                "sire_relationship"
            ].eq("material")
        )
        | (
            partial_relationships["dam_changed"]
            & (
                partial_relationships[
                    "dam_name_relationship"
                ].eq("material")
                | partial_relationships[
                    "dam_country_relationship"
                ].eq("country_suffix_difference")
            )
        )
        | (
            partial_relationships["damsire_changed"]
            & partial_relationships[
                "damsire_relationship"
            ].isin(["material", "one_missing"])
        ),
        [
            "horse",
            "change_pattern",
            "sire",
            "next_sire",
            "sire_relationship",
            "dam_structured_key",
            "next_dam_structured_key",
            "dam_name_relationship",
            "dam_country_relationship",
            "damsire",
            "next_damsire",
            "damsire_relationship",
            "last_date",
            "next_first_date",
            "gap_days",
            "maximum_age",
            "next_minimum_age",
        ],
    ]
    .sort_values(
        [
            "change_pattern",
            "gap_days",
            "horse",
        ],
        ascending=[True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(partial_relationships) == 94
assert partial_relationship_summary["transitions"].sum() == 94

print("Partial-pedigree structural relationship classes")
display(partial_relationship_summary)

print("Transitions retaining a material or missing-value difference")
material_partial_transitions

Partial-pedigree structural relationship classes


,change_pattern,sire_relationship,dam_name_relationship,dam_country_relationship,damsire_relationship,transitions,horse_labels,median_gap_days
0,dam,exact,terminal_i_variant,exact,exact,34,34,184.5
1,damsire,exact,exact,exact,terminal_i_variant,20,18,217.5
2,damsire,exact,exact,exact,punctuation_or_spacing,16,16,305.5
3,sire,material,exact,exact,exact,5,5,260.0
4,damsire,exact,exact,exact,material,4,4,155.5
5,sire,punctuation_or_spacing,exact,exact,exact,4,4,338.5
6,dam,exact,punctuation_or_spacing,exact,exact,2,2,448.0
7,dam + damsire,exact,terminal_i_variant,exact,material,2,2,39.0
8,dam,exact,case_or_whitespace,exact,exact,1,1,138.0
9,dam,exact,exact,country_suffix_difference,exact,1,1,184.0


Transitions retaining a material or missing-value difference


,horse,change_pattern,sire,next_sire,sire_relationship,dam_structured_key,next_dam_structured_key,dam_name_relationship,dam_country_relationship,damsire,next_damsire,damsire_relationship,last_date,next_first_date,gap_days,maximum_age,next_minimum_age
0,Bonny Ezra (NZ),dam,Road To Rock (AUS),Road To Rock (AUS),exact,"(parsed_suffix, Ascolini, AUS)","(parsed_suffix, Ascolini, NZ)",exact,country_suffix_difference,Bertolini,Bertolini,exact,2022-06-25,2022-12-26,184,6,7.0
1,Almavillalobas (GB),dam,Master Carpenter (IRE),Master Carpenter (IRE),exact,"(parsed_suffix, Nation, USA)","(parsed_suffix, Nation II, USA)",material,exact,Rio Verde,Rio Verde,exact,2025-11-25,2026-02-12,79,5,6.0
2,Lyneham (FR),dam + damsire,Wootton Bassett (GB),Wootton Bassett (GB),exact,"(parsed_suffix, Uklanie, FR)","(parsed_suffix, Llanita, GB)",material,country_suffix_difference,Dalakhani,Rock Of Gibraltar,material,2018-09-15,2025-10-14,2586,3,2.0
3,Marakan (IRE),dam + damsire,Arakan (USA),Arakan (USA),exact,"(parsed_suffix, Templemartin Glen, IRE)","(parsed_suffix, Goodthyne Miss, IRE)",material,exact,Anshan,Jolly Jake,material,2016-05-19,2022-11-08,2364,5,6.0
4,New President (FR),dam + damsire,Sinndar (IRE),Sinndar (IRE),exact,"(parsed_suffix, Sun Song I, FR)","(parsed_suffix, Sun Song II, FR)",terminal_i_variant,exact,Dr Fong,,material,2025-09-27,2025-11-15,49,8,8.0
5,Diamond Tipp (IRE),dam + damsire,Diamond Boy (FR),Diamond Boy (FR),exact,"(parsed_suffix, Sound Out, IRE)","(parsed_suffix, Soundout, IRE)",punctuation_or_spacing,exact,Great Palm,Oscar,material,2024-07-05,2024-08-18,44,4,4.0
6,Colwyn Bay (FR),dam + damsire,Falco (USA),Falco (USA),exact,"(parsed_suffix, Eudora, IRE)","(parsed_suffix, Eudora I, IRE)",terminal_i_variant,exact,More Than Ready,Kings Best,material,2026-04-02,2026-05-01,29,7,7.0
7,Jimmy Chou Pecos Aa (FR),damsire,Walk In The Park (IRE),Walk In The Park (IRE),exact,"(parsed_suffix, Rose Chou, FR)","(parsed_suffix, Rose Chou, FR)",exact,exact,Ut*mangarose,Mangarose,material,2022-03-12,2022-12-26,289,6,6.0
8,Runninsonofagun (IRE),damsire,Inns Of Court (IRE),Inns Of Court (IRE),exact,"(parsed_suffix, High Society Lady, IRE)","(parsed_suffix, High Society Lady, IRE)",exact,exact,General Monash,Society Rock,material,2024-10-28,2025-06-18,233,2,3.0
9,Jimmy Chou Pecos AA (FR),damsire,Walk In The Park (IRE),Walk In The Park (IRE),exact,"(parsed_suffix, Rose Chou, FR)","(parsed_suffix, Rose Chou, FR)",exact,exact,Mangarose,Ut*mangarose,material,2024-02-18,2024-05-06,78,8,8.0


### Interim interpretation — Partial-pedigree changes are mostly label variation

Of the 94 temporally separated partial-pedigree transitions, most can be explained by narrowly bounded structural differences.

The dominant classes are:

* 34 dam transitions involving a terminal-`I` variant;
* 20 damsire transitions involving a terminal-`I` variant;
* 16 damsire transitions involving punctuation or spacing;
* four sire transitions involving punctuation or spacing;
* two dam transitions involving punctuation or spacing;
* one dam transition involving case or whitespace.

These 77 transitions do not provide evidence of different horse identities.

They remain reviewable source-label equivalence candidates because the unchanged pedigree components, coherent chronology and narrow text transformations all support a label-variation interpretation. Raw labels must nevertheless remain preserved.

A further two apparent material differences are already covered by the bounded `Ut*` rule:

* `Ut*mangarose` and `Mangarose`;
* `Ut*voiladenuo (FR)` and `Voiladenuo (FR)`.

Those pairs may be reconciled only for the exact observed labels. The rule does not authorise unrestricted removal of `Ut*` from other names.

The remaining material branch includes:

* one dam country-suffix difference;
* one materially different dam label;
* four `dam + damsire` transitions with at least one material difference;
* two additional materially different damsire assertions;
* five materially different sire assertions.

These cases require row-level chronology inspection before deciding whether they represent:

* exact-label reuse;
* a source pedigree defect;
* a bounded label variant;
* or an unresolved assertion.


In [34]:
# Inspect every partial-pedigree transition that retains a genuinely
# material difference after applying the already bounded Ut-prefix rule.

bounded_ut_pairs = {
    frozenset({"Ut*mangarose", "Mangarose"}),
    frozenset({"Ut*voiladenuo (FR)", "Voiladenuo (FR)"}),
}

def is_bounded_ut_pair(left, right):
    if pd.isna(left) or pd.isna(right):
        return False

    return frozenset({str(left), str(right)}) in bounded_ut_pairs


material_partial_review = material_partial_transitions.loc[
    ~material_partial_transitions.apply(
        lambda row: (
            is_bounded_ut_pair(
                row["damsire"],
                row["next_damsire"],
            )
            or is_bounded_ut_pair(
                row["sire"],
                row["next_sire"],
            )
        ),
        axis=1,
    )
].copy()

material_partial_horses = set(
    material_partial_review["horse"]
)

material_partial_rows = (
    structured_pedigree_rows.loc[
        structured_pedigree_rows["horse"].isin(
            material_partial_horses
        ),
        [
            "horse",
            "date",
            "course",
            "off",
            "age",
            "sex",
            "sire",
            "dam",
            "damsire",
        ],
    ]
    .copy()
)

material_partial_rows["date"] = pd.to_datetime(
    material_partial_rows["date"]
)

material_partial_rows = (
    material_partial_rows
    .sort_values(
        ["horse", "date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

material_partial_chronology = (
    material_partial_rows
    .groupby(
        [
            "horse",
            "sire",
            "dam",
            "damsire",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_age=("age", "min"),
        maximum_age=("age", "max"),
        sex_values=(
            "sex",
            lambda values: ", ".join(
                sorted(
                    {
                        str(value)
                        for value in values.dropna()
                    }
                )
            ),
        ),
        course_examples=(
            "course",
            lambda values: " | ".join(
                list(dict.fromkeys(values.dropna()))[:8]
            ),
        ),
    )
    .sort_values(
        ["horse", "first_date", "last_date"],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(material_partial_transitions) == 16
assert len(material_partial_review) == 13

print("Material partial-pedigree transitions after bounded Ut-prefix removal")
display(material_partial_review)

print("Row-level assertion chronology")
material_partial_chronology

Material partial-pedigree transitions after bounded Ut-prefix removal


,horse,change_pattern,sire,next_sire,sire_relationship,dam_structured_key,next_dam_structured_key,dam_name_relationship,dam_country_relationship,damsire,next_damsire,damsire_relationship,last_date,next_first_date,gap_days,maximum_age,next_minimum_age
0,Bonny Ezra (NZ),dam,Road To Rock (AUS),Road To Rock (AUS),exact,"(parsed_suffix, Ascolini, AUS)","(parsed_suffix, Ascolini, NZ)",exact,country_suffix_difference,Bertolini,Bertolini,exact,2022-06-25,2022-12-26,184,6,7.0
1,Almavillalobas (GB),dam,Master Carpenter (IRE),Master Carpenter (IRE),exact,"(parsed_suffix, Nation, USA)","(parsed_suffix, Nation II, USA)",material,exact,Rio Verde,Rio Verde,exact,2025-11-25,2026-02-12,79,5,6.0
2,Lyneham (FR),dam + damsire,Wootton Bassett (GB),Wootton Bassett (GB),exact,"(parsed_suffix, Uklanie, FR)","(parsed_suffix, Llanita, GB)",material,country_suffix_difference,Dalakhani,Rock Of Gibraltar,material,2018-09-15,2025-10-14,2586,3,2.0
3,Marakan (IRE),dam + damsire,Arakan (USA),Arakan (USA),exact,"(parsed_suffix, Templemartin Glen, IRE)","(parsed_suffix, Goodthyne Miss, IRE)",material,exact,Anshan,Jolly Jake,material,2016-05-19,2022-11-08,2364,5,6.0
4,New President (FR),dam + damsire,Sinndar (IRE),Sinndar (IRE),exact,"(parsed_suffix, Sun Song I, FR)","(parsed_suffix, Sun Song II, FR)",terminal_i_variant,exact,Dr Fong,,material,2025-09-27,2025-11-15,49,8,8.0
5,Diamond Tipp (IRE),dam + damsire,Diamond Boy (FR),Diamond Boy (FR),exact,"(parsed_suffix, Sound Out, IRE)","(parsed_suffix, Soundout, IRE)",punctuation_or_spacing,exact,Great Palm,Oscar,material,2024-07-05,2024-08-18,44,4,4.0
6,Colwyn Bay (FR),dam + damsire,Falco (USA),Falco (USA),exact,"(parsed_suffix, Eudora, IRE)","(parsed_suffix, Eudora I, IRE)",terminal_i_variant,exact,More Than Ready,Kings Best,material,2026-04-02,2026-05-01,29,7,7.0
8,Runninsonofagun (IRE),damsire,Inns Of Court (IRE),Inns Of Court (IRE),exact,"(parsed_suffix, High Society Lady, IRE)","(parsed_suffix, High Society Lady, IRE)",exact,exact,General Monash,Society Rock,material,2024-10-28,2025-06-18,233,2,3.0
10,Alderley Charlie (GB),damsire,Ask (GB),Ask (GB),exact,"(parsed_suffix, Alderley Heights, GB)","(parsed_suffix, Alderley Heights, GB)",exact,exact,Ut*windsor Heights,Windsor Heights,material,2024-12-30,2025-01-08,9,4,5.0
11,What A Whopper (IRE),sire,Bushranger (IRE),Churchill (IRE),material,"(parsed_suffix, Chica Whopa, IRE)","(parsed_suffix, Chica Whopa, IRE)",exact,exact,Oasis Dream,Oasis Dream,exact,2015-03-28,2022-08-25,2707,2,2.0


Row-level assertion chronology


,horse,sire,dam,damsire,runner_rows,first_date,last_date,minimum_age,maximum_age,sex_values,course_examples
0,Alderley Charlie (GB),Ask (GB),Alderley Heights GB,Ut*windsor Heights,2,2024-11-06,2024-12-30,4,4,G,Chepstow | Taunton
1,Alderley Charlie (GB),Ask (GB),Alderley Heights GB,Windsor Heights,4,2025-01-08,2025-03-07,5,5,G,Taunton | Exeter
2,Alderley Charlie (GB),Ask (GB),Alderley Heights (GB),Windsor Heights,6,2025-12-18,2026-05-20,5,6,G,Exeter | Taunton | Ffos Las
3,Almavillalobas (GB),Master Carpenter (IRE),Nation (USA),Rio Verde,7,2023-08-16,2025-11-25,3,5,"F, M",Kempton (AW) | Wolverhampton (AW) | Brighton |...
4,Almavillalobas (GB),Master Carpenter (IRE),Nation II (USA),Rio Verde,2,2026-02-12,2026-05-07,6,6,M,Lingfield (AW) | Windsor
5,Bonny Ezra (NZ),Road To Rock (AUS),Ascolini (AUS),Bertolini,1,2022-06-25,2022-06-25,6,6,G,Eagle Farm (AUS)
6,Bonny Ezra (NZ),Road To Rock (AUS),Ascolini (NZ),Bertolini,4,2022-12-26,2024-06-01,7,8,G,Randwick (AUS) | Newcastle (AUS) | Eagle Farm ...
7,Colwyn Bay (FR),Falco (USA),Eudora (IRE),More Than Ready,1,2026-04-02,2026-04-02,7,7,G,Clonmel
8,Colwyn Bay (FR),Falco (USA),Eudora I (IRE),Kings Best,1,2026-05-01,2026-05-01,7,7,G,Punchestown
9,Diamond Tipp (IRE),Diamond Boy (FR),Sound Out (IRE),Great Palm,1,2024-07-05,2024-07-05,4,4,F,Cork (IRE)


### Interim classification — Material partial-pedigree transitions

Row-level chronology distinguishes three broad classes among the thirteen material partial-pedigree transitions.

#### Strong exact-label reuse candidates

Three labels show long temporal separation, coherent but materially different pedigree groups and age histories consistent with separate horses:

* `Lyneham (FR)`;
* `Marakan (IRE)`;
* `What A Whopper (IRE)`.

Although one pedigree component happens to match in each case, the remaining evidence indicates distinct provisional horse occurrences sharing the same exact source label.

#### Bounded source-label or metadata variants

Five transitions are consistent with one continuous horse history and a narrowly identifiable source-label difference:

* `Alderley Charlie (GB)`: `Ut*windsor Heights` / `Windsor Heights`;
* `Hangry (IRE)`: `Galileo (FR)` / `Galileo (IRE)`;
* `LAziza Des Places (FR)`: `Alandi` / `Alanadi`;
* `Almavillalobas (GB)`: `Nation` / `Nation II`;
* `Bonny Ezra (NZ)`: dam country suffix `AUS` / `NZ`.

These remain provisional equivalence or correction candidates. The source-internal chronology supports continuity, but the correct governed form requires either a narrowly bounded rule or external verification.

#### Likely isolated pedigree defects

Five labels retain a materially different pedigree assertion within an otherwise continuous age and race chronology:

* `Diamond Tipp (IRE)`;
* `Colwyn Bay (FR)`;
* `New President (FR)`;
* `Runninsonofagun (IRE)`;
* `Herbert (NZ)`.

For these labels, the changed assertion may be:

* an isolated incorrect damsire;
* an incomplete pedigree;
* an incorrect sire;
* or a source correction introduced between appearances.

The source alone does not establish which assertion is correct. These cases should therefore remain explicit manual-verification candidates rather than being automatically normalized.

This classification is provisional. It narrows the external verification workload without overwriting any raw pedigree value.


In [35]:
material_partial_classification = pd.DataFrame(
    [
        {
            "horse": "Lyneham (FR)",
            "classification": "strong_exact_label_reuse_candidate",
            "reason": "long gap, different dam and damsire, age restart",
        },
        {
            "horse": "Marakan (IRE)",
            "classification": "strong_exact_label_reuse_candidate",
            "reason": "long gap, different dam and damsire, distinct sex history",
        },
        {
            "horse": "What A Whopper (IRE)",
            "classification": "strong_exact_label_reuse_candidate",
            "reason": "long gap, different sire, age restart and sex difference",
        },
        {
            "horse": "Alderley Charlie (GB)",
            "classification": "bounded_label_variant_candidate",
            "reason": "exact Ut-prefix pair with continuous age chronology",
        },
        {
            "horse": "Hangry (IRE)",
            "classification": "bounded_metadata_correction_candidate",
            "reason": "same pedigree except sire country suffix; continuous age chronology",
        },
        {
            "horse": "LAziza Des Places (FR)",
            "classification": "bounded_spelling_variant_candidate",
            "reason": "Alandi and Alanadi differ by one internal letter; all other evidence stable",
        },
        {
            "horse": "Almavillalobas (GB)",
            "classification": "bounded_terminal_numeral_candidate",
            "reason": "Nation and Nation II with otherwise continuous pedigree and age",
        },
        {
            "horse": "Bonny Ezra (NZ)",
            "classification": "dam_country_suffix_correction_candidate",
            "reason": "same dam name, sire and damsire; only dam country suffix changes",
        },
        {
            "horse": "Diamond Tipp (IRE)",
            "classification": "manual_verification_required",
            "reason": "single early Great Palm damsire followed by sustained Oscar assertion",
        },
        {
            "horse": "Colwyn Bay (FR)",
            "classification": "manual_verification_required",
            "reason": "two isolated rows disagree on dam label and damsire",
        },
        {
            "horse": "New President (FR)",
            "classification": "manual_verification_required",
            "reason": "terminal-numeral dam progression plus final missing damsire",
        },
        {
            "horse": "Runninsonofagun (IRE)",
            "classification": "manual_verification_required",
            "reason": "sustained General Monash then sustained Society Rock assertion",
        },
        {
            "horse": "Herbert (NZ)",
            "classification": "manual_verification_required",
            "reason": "same dam and damsire but materially different sire",
        },
    ]
)

assert len(material_partial_classification) == 13
assert set(material_partial_classification["horse"]) == set(
    material_partial_review["horse"]
)

material_partial_classification_summary = (
    material_partial_classification
    .groupby("classification", as_index=False)
    .agg(
        horse_labels=("horse", "size"),
        examples=(
            "horse",
            lambda values: " | ".join(values),
        ),
    )
    .sort_values(
        ["horse_labels", "classification"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Material partial-pedigree provisional classification")
display(material_partial_classification_summary)

material_partial_classification

Material partial-pedigree provisional classification


,classification,horse_labels,examples
0,manual_verification_required,5,Diamond Tipp (IRE) | Colwyn Bay (FR) | New Pre...
1,strong_exact_label_reuse_candidate,3,Lyneham (FR) | Marakan (IRE) | What A Whopper ...
2,bounded_label_variant_candidate,1,Alderley Charlie (GB)
3,bounded_metadata_correction_candidate,1,Hangry (IRE)
4,bounded_spelling_variant_candidate,1,LAziza Des Places (FR)
5,bounded_terminal_numeral_candidate,1,Almavillalobas (GB)
6,dam_country_suffix_correction_candidate,1,Bonny Ezra (NZ)


,horse,classification,reason
0,Lyneham (FR),strong_exact_label_reuse_candidate,"long gap, different dam and damsire, age restart"
1,Marakan (IRE),strong_exact_label_reuse_candidate,"long gap, different dam and damsire, distinct ..."
2,What A Whopper (IRE),strong_exact_label_reuse_candidate,"long gap, different sire, age restart and sex ..."
3,Alderley Charlie (GB),bounded_label_variant_candidate,exact Ut-prefix pair with continuous age chron...
4,Hangry (IRE),bounded_metadata_correction_candidate,same pedigree except sire country suffix; cont...
5,LAziza Des Places (FR),bounded_spelling_variant_candidate,Alandi and Alanadi differ by one internal lett...
6,Almavillalobas (GB),bounded_terminal_numeral_candidate,Nation and Nation II with otherwise continuous...
7,Bonny Ezra (NZ),dam_country_suffix_correction_candidate,"same dam name, sire and damsire; only dam coun..."
8,Diamond Tipp (IRE),manual_verification_required,single early Great Palm damsire followed by su...
9,Colwyn Bay (FR),manual_verification_required,two isolated rows disagree on dam label and da...


### Conclusion — Temporally separated partial-pedigree transitions

The 94 temporally separated transitions affecting only one or two pedigree components were classified using reversible structural comparison, chronology and age evidence.

They divide into the following groups.

#### Narrow source-label variation

Seventy-seven transitions are explained by bounded textual differences:

* 34 dam terminal-`I` variants;
* 20 damsire terminal-`I` variants;
* 16 damsire punctuation or spacing variants;
* four sire punctuation or spacing variants;
* two dam punctuation or spacing variants;
* one dam case or whitespace variant.

These transitions do not support separate horse identities.

They remain provisional label-equivalence candidates because the relationship is supported by:

* unchanged surrounding pedigree components;
* coherent age progression;
* non-overlapping but continuous chronology;
* and a narrowly reversible text transformation.

The raw labels must remain preserved, and no unrestricted global normalization rule is authorised.

#### Bounded `Ut*` variants

Three additional transitions are covered by exact observed `Ut*` pairs:

* `Ut*mangarose` / `Mangarose`;
* `Ut*voiladenuo (FR)` / `Voiladenuo (FR)`;
* `Ut*windsor Heights` / `Windsor Heights`.

These may be governed only as explicitly enumerated pairs. The evidence does not support removing `Ut*` from arbitrary pedigree labels.

#### Strong exact-label reuse candidates

Three labels retain materially different pedigree evidence and long separated histories consistent with distinct horses:

* `Lyneham (FR)`;
* `Marakan (IRE)`;
* `What A Whopper (IRE)`.

They should be represented as separate provisional horse occurrences sharing the same exact raw horse label.

#### Bounded metadata or spelling candidates

Five labels show a continuous horse history with one narrowly defined metadata or spelling difference:

* `Hangry (IRE)`: sire country suffix `FR` / `IRE`;
* `LAziza Des Places (FR)`: `Alandi` / `Alanadi`;
* `Almavillalobas (GB)`: `Nation` / `Nation II`;
* `Bonny Ezra (NZ)`: dam country suffix `AUS` / `NZ`;
* `Alderley Charlie (GB)`: bounded `Ut*` damsire pair.

These should remain reviewable correction or equivalence candidates until externally verified. They do not currently justify an identity split.

#### Manual-verification candidates

Five labels retain materially different pedigree assertions that cannot be resolved safely from the source alone:

* `Diamond Tipp (IRE)`;
* `Colwyn Bay (FR)`;
* `New President (FR)`;
* `Runninsonofagun (IRE)`;
* `Herbert (NZ)`.

These cases must preserve all competing raw assertions and carry an explicit review status. No canonical pedigree should be exposed until external evidence identifies the correct relationship.

#### Overall result

The 94 partial-pedigree transitions therefore partition into:

| classification                                                         | transitions |
| ---------------------------------------------------------------------- | ----------: |
| narrow structural label variants                                       |          77 |
| bounded `Ut*` variants                                                 |           3 |
| strong exact-label reuse candidates                                    |           3 |
| bounded metadata or spelling candidates                                |           5 |
| unresolved manual-verification candidates                              |           5 |
| additional transition within an already counted multi-transition label |           1 |
| **total**                                                              |      **94** |

The final count includes multiple transitions for some exact horse labels, so transition totals and unique-label totals are not interchangeable.

This branch confirms that partial pedigree differences must not be interpreted uniformly:

> A partial pedigree change may represent a label variant, metadata correction, source defect or genuine horse-label collision. Classification requires the changed component, chronology, age evidence and surrounding pedigree context.

Raw `horse`, `sire`, `dam` and `damsire` values remain immutable in every class.


### External verification — `New President (FR)`

The source contains three dam-label forms for `New President (FR)`:

* `Sun Song`;
* `Sun Song I`;
* `Sun Song II`.

The final `Sun Song II` assertion also omits the damsire, while earlier rows identify `Dr Fong`.

Official France Galop records resolve the apparent contradiction.

France Galop’s published official racing bulletins identify `New President` as:

* sire: `Sinndar`;
* dam: `Sun Song`;
* damsire: `Dr Fong`.

The same registered pedigree appears in official records across multiple seasons.

The governed interpretation is therefore:

* horse: `New President (FR)`;
* sire: `Sinndar (IRE)`;
* dam: `Sun Song (FR)`;
* damsire: `Dr Fong`;
* `Sun Song I` and `Sun Song II`: source-label variants of `Sun Song` for this exact pedigree family;
* blank damsire on 15 November 2025: incomplete source pedigree assertion.

This is not evidence of multiple real-world horses or competing registered pedigrees.

All raw labels and the blank damsire must remain preserved for lineage. A governed downstream layer may expose the France Galop-supported pedigree with explicit official provenance.

This verification is bounded to `New President (FR)` and does not authorise general removal of terminal Roman numerals from arbitrary dam labels.


### External verification — `Herbert (NZ)`

The source contains two sire labels for `Herbert (NZ)`:

* `Warning Flag (USA)`;
* `Sweet Orange (USA)`.

All other pedigree components remain stable:

* dam: `Ze One (AUS)`;
* damsire: `All American`.

The official New Zealand Stud Book resolves the apparent contradiction.

Its registered record for `Herbert (NZ)`, foaled on 18 October 2020 with life number `NZ00423062`, identifies the horse as:

* sire: `Sweet Orange (USA)`;
* dam: `Ze One (AUS)`;
* damsire: `All American (AUS)`.

The official Stud Book record for `Sweet Orange (USA)` separately records `Warning Flag (USA)` as another name used by the same stallion.

The two source sire values therefore refer to one real-world sire:

* registered breeding name: `Sweet Orange (USA)`;
* alternative racing name: `Warning Flag (USA)`.

This is not a pedigree correction, source defect or horse-identity split. It is a verified sire-alias relationship.

The governed interpretation for `Herbert (NZ)` is:

* horse: `Herbert (NZ)`;
* sire entity: `Sweet Orange (USA)`;
* equivalent source sire label: `Warning Flag (USA)`;
* dam: `Ze One (AUS)`;
* damsire: `All American (AUS)`.

Both raw sire labels must remain preserved for lineage. A governed identity layer may link them to the same verified sire entity using the official New Zealand Stud Book evidence.

This verification is specific to `Sweet Orange (USA)` and `Warning Flag (USA)` and does not authorise general alias matching from names alone.


### Pending official pedigree confirmations

Three pedigree discrepancies remain open because the available public evidence does not include the governing stud-book record needed to choose safely between competing published values.

#### `Diamond Tipp (IRE)`

The source contains:

* `Diamond Boy — Sound Out — Great Palm`;
* `Diamond Boy — Soundout — Oscar`.

The disagreement is currently between publication and data-provider chains. No Irish Stud Book or Weatherbys registration record has yet been obtained.

Status:

* official enquiry sent to Weatherbys Ireland;
* raw assertions preserved;
* no governed damsire correction applied;
* classification: `pending_official_confirmation`.

#### `Colwyn Bay (FR)`

The source contains:

* `Falco — Eudora — More Than Ready`;
* `Falco — Eudora I — King's Best`.

The `Eudora` / `Eudora I` difference is consistent with a terminal-`I` label variant, but the damsire difference is material.

Status:

* official enquiry sent to France Galop;
* raw assertions preserved;
* no governed damsire correction applied;
* classification: `pending_official_confirmation`.

#### `Runninsonofagun (IRE)`

The source contains two internally consistent chronological periods:

* seven earlier rows with `General Monash`;
* eleven later rows with `Society Rock`.

The stable switch suggests that an upstream pedigree record was changed between October 2024 and June 2025, but the source alone cannot establish whether that change corrected or introduced an error.

Status:

* official enquiry sent to Weatherbys Ireland;
* raw assertions preserved;
* no governed damsire correction applied;
* classification: `pending_official_confirmation`.

These cases must remain unresolved until a response or registration record is received from the relevant governing authority.

Commercial publication consensus may be retained as supporting evidence, but it must not govern the corrected pedigree where official stud-book confirmation is still pending.


### Stage 7 — From source labels to provisional horse occurrences

The contradiction analysis establishes that the source `horse` field is a reported label rather than a permanent horse identifier.

An identical complete label may represent:

* one horse with stable pedigree reporting;
* one horse whose pedigree labels change through punctuation, spacing, aliases or metadata corrections;
* one horse with one or more defective pedigree rows;
* or multiple real-world horses whose identical names and breeding-country suffixes are reused across time or jurisdictions.

The database therefore requires an intermediate identity layer between immutable source rows and any later authoritative horse entity.

#### Raw source label

The exact `horse` value stored on each runner row.

It must remain unchanged and traceable to the original row.

#### Structured pedigree assertion

The source-reported combination of:

* sire;
* reversibly parsed dam label and country suffix;
* damsire.

This is evidence attached to a runner appearance. It is not automatically a verified pedigree.

#### Provisional horse occurrence

A source-internal grouping of runner rows that appear to describe one continuous horse history.

A provisional occurrence may use:

* exact raw horse label;
* coherent structured pedigree assertion;
* non-contradictory chronology;
* plausible age progression;
* compatible sex history;
* and explicit bounded label-equivalence rules.

It must split where the evidence strongly indicates label reuse, including:

* materially different complete pedigrees;
* incompatible age continuity;
* distinct registry identities;
* or captured official verification.

#### Verified horse entity

A real-world horse identity supported by an authoritative identifier or governing registration record.

A verified entity may link:

* multiple raw horse labels;
* multiple provisional occurrences;
* aliases used in different jurisdictions;
* or source rows containing corrected pedigree information.

No verified entity should be created from name similarity alone.

#### Required status fields

Each provisional occurrence or reconciliation decision should retain:

* identity status;
* confidence;
* evidence basis;
* governing verification identifier where applicable;
* unresolved-conflict flag;
* first and last observed dates;
* source-row lineage;
* and the raw pedigree assertions included or excluded.

This model allows analysis to use coherent horse histories without pretending that names are globally unique or silently rewriting defective source data.


### Conclusion — Transition-level identity governance

The 353 temporally separated pedigree transitions now have explicit source-governance decisions.

They divide into:

| transition decision           | transitions | exact horse labels |
| ----------------------------- | ----------: | -----------------: |
| split provisional occurrence  |         261 |                261 |
| retain single occurrence      |          89 |                 86 |
| pending official confirmation |           3 |                  3 |
| **total**                     |     **353** |                  — |

#### Split provisional occurrence

Two hundred and sixty-one transitions require a new provisional horse occurrence.

These include:

* 258 genuine complete-pedigree identity transitions;
* three strong partial-pedigree label-reuse candidates.

The evidence supporting a split includes:

* materially different pedigree assertions;
* temporally separated histories;
* incompatible age continuity;
* sex-history differences where present;
* and, in some cases, external confirmation of distinct registered horses.

A split does not by itself create two verified real-world horse entities. It records that the source evidence cannot safely be represented as one continuous horse occurrence.

#### Retain single occurrence

Eighty-nine transitions across 86 exact horse labels remain within one provisional occurrence.

These include:

* punctuation, spacing and case variants;
* terminal-Roman-numeral variants;
* bounded `Ut*` label pairs;
* verified sire aliases;
* metadata corrections;
* incomplete pedigree assertions;
* and externally confirmed source pedigree defects.

Retaining one occurrence does not mean overwriting the raw assertion. Every source value remains attached to its original runner row, while the occurrence-level layer records the governed relationship between those assertions.

#### Pending official confirmation

Three transitions remain unresolved:

* `Diamond Tipp (IRE)`;
* `Colwyn Bay (FR)`;
* `Runninsonofagun (IRE)`.

Each has a material damsire disagreement that cannot be governed safely from publication consensus alone.

Official enquiries have been sent to the relevant stud-book or registration authority.

Until a response is received:

* both competing assertions remain preserved;
* no canonical damsire is assigned;
* no identity split is created;
* and each transition retains `pending_official_confirmation` status.

#### Governance consequence

The transition table provides the decision boundary required to construct provisional horse occurrences:

> A new occurrence begins only where an explicit governed transition requires an identity split. Label variants, aliases, metadata corrections and source defects remain within the existing occurrence, while unresolved transitions remain unsplit and visibly pending.

This prevents raw horse labels from being treated as permanent natural keys while also avoiding unnecessary fragmentation caused by formatting variation or correctable source defects.


### External verification — `Hangry (IRE)`

The source contains two sire-label forms for `Hangry (IRE)`:

* `Galileo (FR)`;
* `Galileo (IRE)`.

The remaining pedigree and horse history are stable:

* dam: `Magic Tree (UAE)`;
* damsire: `Timber Country`;
* coherent age progression;
* continuous sex and racing history.

Published Irish form records consistently identify `Hangry (IRE)` as:

* sire: `Galileo (IRE)`;
* dam: `Magic Tree (UAE)`;
* damsire: `Timber Country (USA)`.

International racing-authority records also identify the relevant stallion as `Galileo (IRE)`.

The `(FR)` suffix is therefore an incorrect country suffix attached to the same sire identity. It is not evidence of:

* a second sire;
* a pedigree change;
* or a separate horse occurrence.

The governed interpretation is:

* horse: `Hangry (IRE)`;
* sire: `Galileo (IRE)`;
* dam: `Magic Tree (UAE)`;
* damsire: `Timber Country (USA)`.

The raw `Galileo (FR)` value must remain preserved on its original rows for lineage. A governed downstream layer may expose `Galileo (IRE)` and classify the earlier suffix as a source metadata defect.

This decision is bounded to this identified sire relationship and does not authorise country-suffix replacement based on name similarity alone.


### External verification — `Bonny Ezra (NZ)`

The source contains two dam-label forms for `Bonny Ezra (NZ)`:

* `Ascolini (AUS)`;
* `Ascolini (NZ)`.

The remaining pedigree is stable:

* sire: `Road To Rock (AUS)`;
* damsire: `Bertolini (USA)`;
* continuous horse history.

New Zealand Stud Book material identifies the broodmare as:

* `Ascolini (NZ)`;
* foaled in 2006;
* by `Bertolini (USA)`;
* out of `Ascona (NZ)`.

Independent breeding records also identify Bonny Ezra’s dam as `Ascolini (NZ)`.

The governed interpretation is therefore:

* horse: `Bonny Ezra (NZ)`;
* sire: `Road To Rock (AUS)`;
* dam: `Ascolini (NZ)`;
* damsire: `Bertolini (USA)`.

The `(AUS)` form is an incorrect breeding-country suffix attached to the same dam identity. It is not evidence of:

* a different mare;
* a pedigree change;
* or a separate horse occurrence.

The raw `Ascolini (AUS)` value must remain preserved on its original rows. A governed downstream layer may expose `Ascolini (NZ)` and classify the alternative suffix as a source metadata defect.


### External verification — `Alderley Charlie (GB)`

The source contains two damsire-label forms for `Alderley Charlie (GB)`:

* `Windsor Heights`;
* `Ut*Windsor Heights`.

The surrounding pedigree remains stable:

* sire: `Ask`;
* dam: `Alderley Heights`;
* continuous age, sex and racing history.

`Windsor Heights` is an identifiable registered stallion. Published pedigree records consistently identify Alderley Charlie as:

* `Ask`;
* out of `Alderley Heights`;
* by `Windsor Heights`.

No evidence was found for a separate stallion registered as `Ut*Windsor Heights`.

The `Ut*` prefix is therefore treated as a source-system or registry marker attached to the underlying name. Its precise internal expansion is not established and must not be invented.

The governed interpretation is:

* verified damsire label: `Windsor Heights`;
* raw source variant: `Ut*Windsor Heights`;
* decision: `label_equivalence`;
* identity split: no.

Both raw forms remain preserved on their original runner rows. The governed layer may expose `Windsor Heights` while recording the prefixed form as a bounded source-label variant.


In [36]:
# Build explicit governance decisions for temporally separated pedigree transitions.

transition_governance = separated_transitions.copy()

transition_governance["transition_decision"] = "review_required"
transition_governance["decision_basis"] = "not_yet_classified"
transition_governance["identity_split"] = pd.NA
transition_governance["governing_verification_id"] = pd.NA

# Complete-pedigree changes generally represent exact-label reuse.
full_pedigree_mask = (
    transition_governance["pedigree_components_changed"].eq(3)
)

transition_governance.loc[
    full_pedigree_mask,
    [
        "transition_decision",
        "decision_basis",
        "identity_split",
    ],
] = [
    "split_provisional_occurrence",
    "complete_pedigree_change_with_separated_chronology",
    True,
]

# Felix Felicis is one horse with defective early pedigree rows.
felix_mask = transition_governance["horse"].eq(
    "Felix Felicis (FR)"
)

transition_governance.loc[
    felix_mask,
    [
        "transition_decision",
        "decision_basis",
        "identity_split",
    ],
] = [
    "retain_single_occurrence",
    "externally_verified_source_pedigree_defect",
    False,
]

# Forest King is a confirmed exact-label collision.
forest_king_mask = transition_governance["horse"].eq(
    "Forest King (AUS)"
)

transition_governance.loc[
    forest_king_mask,
    [
        "transition_decision",
        "decision_basis",
        "identity_split",
    ],
] = [
    "split_provisional_occurrence",
    "externally_confirmed_exact_label_collision",
    True,
]

# Apply classifications from the material partial-pedigree review.
partial_classification_lookup = (
    material_partial_classification
    .set_index("horse")["classification"]
    .to_dict()
)

partial_reason_lookup = (
    material_partial_classification
    .set_index("horse")["reason"]
    .to_dict()
)

for horse, classification in partial_classification_lookup.items():
    horse_mask = (
        transition_governance["horse"].eq(horse)
        & transition_governance[
            "pedigree_components_changed"
        ].isin([1, 2])
    )

    if classification == "strong_exact_label_reuse_candidate":
        decision = "split_provisional_occurrence"
        identity_split = True

    elif classification == "manual_verification_required":
        decision = "pending_official_confirmation"
        identity_split = pd.NA

    else:
        decision = "retain_single_occurrence"
        identity_split = False

    transition_governance.loc[
        horse_mask,
        [
            "transition_decision",
            "decision_basis",
            "identity_split",
        ],
    ] = [
        decision,
        partial_reason_lookup[horse],
        identity_split,
    ]

# Apply broad structural-variant treatment to remaining partial transitions.
structural_variant_mask = (
    transition_governance[
        "pedigree_components_changed"
    ].isin([1, 2])
    & transition_governance[
        "transition_decision"
    ].eq("review_required")
)

transition_governance.loc[
    structural_variant_mask,
    [
        "transition_decision",
        "decision_basis",
        "identity_split",
    ],
] = [
    "retain_single_occurrence",
    "bounded_structural_pedigree_label_variant",
    False,
]

# Completed official or authoritative external resolutions.

new_president_mask = transition_governance["horse"].eq(
    "New President (FR)"
)

transition_governance.loc[
    new_president_mask,
    [
        "transition_decision",
        "decision_basis",
        "identity_split",
        "governing_verification_id",
    ],
] = [
    "retain_single_occurrence",
    "official_pedigree_confirms_label_variation_and_missing_damsire",
    False,
    "NB19-HORSE-0002",
]

herbert_mask = transition_governance["horse"].eq(
    "Herbert (NZ)"
)

transition_governance.loc[
    herbert_mask,
    [
        "transition_decision",
        "decision_basis",
        "identity_split",
        "governing_verification_id",
    ],
] = [
    "retain_single_occurrence",
    "official_stud_book_confirms_sire_alias",
    False,
    "NB19-HORSE-0003",
]

bonny_ezra_mask = transition_governance["horse"].eq(
    "Bonny Ezra (NZ)"
)

transition_governance.loc[
    bonny_ezra_mask,
    [
        "transition_decision",
        "decision_basis",
        "identity_split",
    ],
] = [
    "retain_single_occurrence",
    "external_stud_book_evidence_confirms_dam_country_suffix_defect",
    False,
]

alderley_charlie_mask = transition_governance["horse"].eq(
    "Alderley Charlie (GB)"
)

transition_governance.loc[
    alderley_charlie_mask,
    [
        "transition_decision",
        "decision_basis",
        "identity_split",
    ],
] = [
    "retain_single_occurrence",
    "registered_damsire_confirms_ut_prefix_label_variant",
    False,
]

# Cases still awaiting governing-authority confirmation.

pending_authority_cases = {
    "Diamond Tipp (IRE)",
    "Colwyn Bay (FR)",
    "Runninsonofagun (IRE)",
    "LAziza Des Places (FR)",
    "Almavillalobas (GB)",
}

pending_authority_mask = transition_governance["horse"].isin(
    pending_authority_cases
)

transition_governance.loc[
    pending_authority_mask,
    [
        "transition_decision",
        "identity_split",
    ],
] = [
    "pending_official_confirmation",
    pd.NA,
]

transition_governance.loc[
    transition_governance["horse"].eq("LAziza Des Places (FR)"),
    "decision_basis",
] = (
    "competing_real_sire_entities_pending_french_stud_book_confirmation"
)

transition_governance.loc[
    transition_governance["horse"].eq("Almavillalobas (GB)"),
    "decision_basis",
] = (
    "probable_publisher_added_dam_numeral_pending_general_stud_book_confirmation"
)

transition_governance_summary = (
    transition_governance
    .groupby(
        [
            "transition_decision",
            "identity_split",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        transitions=("horse", "size"),
        horse_labels=("horse", "nunique"),
    )
    .sort_values(
        ["transition_decision", "identity_split"],
        kind="stable",
    )
    .reset_index(drop=True)
)

pending_transition_columns = [
    "horse",
    "group_number",
    "pedigree_components_changed",
    "sire_changed",
    "dam_changed",
    "damsire_changed",
    "last_date",
    "next_first_date",
    "decision_basis",
]

pending_official_transitions = (
    transition_governance.loc[
        transition_governance[
            "transition_decision"
        ].eq("pending_official_confirmation"),
        pending_transition_columns,
    ]
    .sort_values(
        ["horse", "group_number"],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(transition_governance) == 353

assert transition_governance[
    "transition_decision"
].ne("review_required").all()

assert set(pending_official_transitions["horse"]) == (
    pending_authority_cases
)

print("Transition-level identity governance decisions")
display(transition_governance_summary)

print("Transitions still pending official confirmation")
pending_official_transitions

Transition-level identity governance decisions


,transition_decision,identity_split,transitions,horse_labels
0,pending_official_confirmation,NaN,5,5
1,retain_single_occurrence,False,87,84
2,split_provisional_occurrence,True,261,261


Transitions still pending official confirmation


,horse,group_number,pedigree_components_changed,sire_changed,dam_changed,damsire_changed,last_date,next_first_date,decision_basis
0,Almavillalobas (GB),1,1,False,True,False,2025-11-25,2026-02-12,probable_publisher_added_dam_numeral_pending_g...
1,Colwyn Bay (FR),1,2,False,True,True,2026-04-02,2026-05-01,two isolated rows disagree on dam label and da...
2,Diamond Tipp (IRE),1,2,False,True,True,2024-07-05,2024-08-18,single early Great Palm damsire followed by su...
3,LAziza Des Places (FR),1,1,True,False,False,2025-09-13,2025-10-29,competing_real_sire_entities_pending_french_st...
4,Runninsonofagun (IRE),1,1,False,False,True,2024-10-28,2025-06-18,sustained General Monash then sustained Societ...


### Updated conclusion — Transition-level identity governance

The 353 temporally separated pedigree transitions now have explicit governance decisions.

| transition decision           | transitions | exact horse labels |
| ----------------------------- | ----------: | -----------------: |
| split provisional occurrence  |         261 |                261 |
| retain single occurrence      |          87 |                 84 |
| pending official confirmation |           5 |                  5 |
| **total**                     |     **353** |                  — |

#### Split provisional occurrence

Two hundred and sixty-one transitions are treated as boundaries between distinct horse occurrences sharing the same raw horse label.

These are cases where the evidence indicates that the same displayed horse name and breeding-country suffix have been used for different real horses.

The raw source label remains unchanged, but the histories must not be merged into one analytical horse record.

#### Retain single occurrence

Eighty-seven transitions across 84 horse labels remain within one horse occurrence.

These include:

* punctuation, spacing and case variants;
* terminal-Roman-numeral variants;
* bounded `Ut*` source markers;
* verified aliases;
* incorrect country suffixes;
* incomplete pedigree rows;
* and externally supported source defects.

The raw assertions remain preserved, while the governed layer records the supported pedigree relationship or label equivalence.

#### Pending official confirmation

Five transitions remain unresolved:

* `Almavillalobas (GB)` — `Nation (USA)` versus `Nation II (USA)`;
* `Colwyn Bay (FR)` — disagreement affecting dam and damsire;
* `Diamond Tipp (IRE)` — `Great Palm` versus `Oscar`;
* `L’Aziza des Places (FR)` — `Alandi (IRE)` versus `Alanadi (FR)`;
* `Runninsonofagun (IRE)` — `General Monash` versus `Society Rock`.

Each case has been referred to the relevant governing Stud Book or registration authority where a sufficiently authoritative public record was unavailable.

Until an official response is received:

* both source assertions remain preserved;
* no canonical pedigree value is assigned;
* no identity split is created;
* and the transition remains explicitly marked `pending_official_confirmation`.

This prevents publication consensus from being mistaken for governing evidence while allowing the rest of the identity analysis to proceed.


### Stage 8 — Governed pedigree reconciliation

Where the evidence identifies one continuous horse, competing pedigree labels should not remain as unresolved alternatives merely because they appeared in the source.

The governed layer should distinguish between:

* the immutable raw assertion recorded on each runner row;
* the verified or best-supported pedigree relationship;
* the type of source discrepancy;
* and the authority supporting the reconciliation.

A governed reconciliation may classify a discrepancy as:

* `label_equivalence` — two labels refer to the same registered horse;
* `country_suffix_defect` — the underlying name is correct but the breeding-country suffix is wrong;
* `publisher_disambiguation` — a numeral or marker was added by a publication rather than forming part of the registered name;
* `source_prefix_variant` — a bounded source-system marker is attached to the underlying name;
* `incorrect_entity_assignment` — the source selected a different real sire, dam or damsire;
* `incomplete_pedigree_assertion` — a source row omits a relationship confirmed elsewhere;
* or `pending_official_confirmation`.

The raw values must never be overwritten. Instead, each reconciliation should record:

* horse label;
* pedigree role affected;
* raw competing labels;
* governed label where established;
* reconciliation type;
* evidence status;
* verification identifier or authority;
* whether the discrepancy implies another horse identity;
* and whether database correction is permitted.

This separates identity governance from source correction:

> A horse may remain one identity while one or more of its source pedigree assertions are corrected, normalized or explicitly rejected in the governed layer.

Only genuine same-label collisions should require separate horse occurrences.


In [37]:
# Record pedigree reconciliations established during Notebook 19.

pedigree_reconciliations = pd.DataFrame(
    [
        {
            "horse": "Felix Felicis (FR)",
            "pedigree_role": "complete_pedigree",
            "raw_competing_labels": (
                "Olympic Glory — Sorina — Le Havre | "
                "Affinisea — Just Eile — Presenting"
            ),
            "governed_label": (
                "Olympic Glory — Sorina — Le Havre"
            ),
            "reconciliation_type": "incorrect_entity_assignment",
            "evidence_status": "externally_verified",
            "verification_id": pd.NA,
            "authority_or_source": (
                "external pedigree verification"
            ),
            "implies_distinct_horse": False,
            "database_correction_permitted": True,
            "notes": (
                "Three early rows contain a pedigree belonging "
                "to a different horse."
            ),
        },
        {
            "horse": "New President (FR)",
            "pedigree_role": "damsire",
            "raw_competing_labels": (
                "Sun Song | Sun Song I | Sun Song II | blank"
            ),
            "governed_label": "Dr Fong",
            "reconciliation_type": (
                "label_equivalence_and_incomplete_pedigree_assertion"
            ),
            "evidence_status": "officially_verified",
            "verification_id": "NB19-HORSE-0002",
            "authority_or_source": "France Galop",
            "implies_distinct_horse": False,
            "database_correction_permitted": True,
            "notes": (
                "The raw variants apply to the dam label; "
                "official pedigree confirms Dr Fong as damsire."
            ),
        },
        {
            "horse": "Herbert (NZ)",
            "pedigree_role": "sire",
            "raw_competing_labels": (
                "Warning Flag (USA) | Sweet Orange (USA)"
            ),
            "governed_label": "Sweet Orange (USA)",
            "reconciliation_type": "label_equivalence",
            "evidence_status": "officially_verified",
            "verification_id": "NB19-HORSE-0003",
            "authority_or_source": (
                "New Zealand Stud Book / LOVERACING.NZ"
            ),
            "implies_distinct_horse": False,
            "database_correction_permitted": True,
            "notes": (
                "Official record identifies Warning Flag as "
                "another name for Sweet Orange."
            ),
        },
        {
            "horse": "Bonny Ezra (NZ)",
            "pedigree_role": "dam",
            "raw_competing_labels": (
                "Ascolini (AUS) | Ascolini (NZ)"
            ),
            "governed_label": "Ascolini (NZ)",
            "reconciliation_type": "country_suffix_defect",
            "evidence_status": "authoritatively_supported",
            "verification_id": pd.NA,
            "authority_or_source": (
                "New Zealand Stud Book / LOVERACING.NZ"
            ),
            "implies_distinct_horse": False,
            "database_correction_permitted": True,
            "notes": (
                "The AUS suffix is incorrect; the registered "
                "broodmare is New Zealand-bred."
            ),
        },
        {
            "horse": "Alderley Charlie (GB)",
            "pedigree_role": "damsire",
            "raw_competing_labels": (
                "Ut*Windsor Heights | Windsor Heights"
            ),
            "governed_label": "Windsor Heights",
            "reconciliation_type": "source_prefix_variant",
            "evidence_status": "authoritatively_supported",
            "verification_id": pd.NA,
            "authority_or_source": (
                "Sport Horse Breeding of Great Britain"
            ),
            "implies_distinct_horse": False,
            "database_correction_permitted": True,
            "notes": (
                "Ut* is treated as a source or registry marker, "
                "not part of the registered stallion name."
            ),
        },
        {
            "horse": "Hangry (IRE)",
            "pedigree_role": "sire",
            "raw_competing_labels": (
                "Galileo (FR) | Galileo (IRE)"
            ),
            "governed_label": "Galileo (IRE)",
            "reconciliation_type": "country_suffix_defect",
            "evidence_status": "strongly_supported",
            "verification_id": pd.NA,
            "authority_or_source": (
                "consistent published pedigree and "
                "international stallion identity"
            ),
            "implies_distinct_horse": False,
            "database_correction_permitted": True,
            "notes": (
                "The FR suffix is an incorrect country label "
                "for the same sire."
            ),
        },
        {
            "horse": "Diamond Tipp (IRE)",
            "pedigree_role": "dam_and_damsire",
            "raw_competing_labels": (
                "Sound Out — Great Palm | Soundout — Oscar"
            ),
            "governed_label": pd.NA,
            "reconciliation_type": "pending_official_confirmation",
            "evidence_status": "pending_official_confirmation",
            "verification_id": pd.NA,
            "authority_or_source": "Weatherbys Ireland enquiry",
            "implies_distinct_horse": False,
            "database_correction_permitted": False,
            "notes": (
                "Official confirmation requested; competing "
                "publication chains remain preserved."
            ),
        },
        {
            "horse": "Colwyn Bay (FR)",
            "pedigree_role": "dam_and_damsire",
            "raw_competing_labels": (
                "Eudora — More Than Ready | "
                "Eudora I — King's Best"
            ),
            "governed_label": pd.NA,
            "reconciliation_type": "pending_official_confirmation",
            "evidence_status": "pending_official_confirmation",
            "verification_id": pd.NA,
            "authority_or_source": "France Galop enquiry",
            "implies_distinct_horse": False,
            "database_correction_permitted": False,
            "notes": (
                "Official confirmation requested because the "
                "disagreement affects two pedigree components."
            ),
        },
        {
            "horse": "Runninsonofagun (IRE)",
            "pedigree_role": "damsire",
            "raw_competing_labels": (
                "General Monash | Society Rock"
            ),
            "governed_label": pd.NA,
            "reconciliation_type": "pending_official_confirmation",
            "evidence_status": "pending_official_confirmation",
            "verification_id": pd.NA,
            "authority_or_source": "Weatherbys Ireland enquiry",
            "implies_distinct_horse": False,
            "database_correction_permitted": False,
            "notes": (
                "The source switches between two sustained "
                "damsire histories."
            ),
        },
        {
            "horse": "LAziza Des Places (FR)",
            "pedigree_role": "sire",
            "raw_competing_labels": (
                "Alandi (IRE) | Alanadi (FR)"
            ),
            "governed_label": pd.NA,
            "reconciliation_type": "pending_official_confirmation",
            "evidence_status": "pending_official_confirmation",
            "verification_id": pd.NA,
            "authority_or_source": "France Galop enquiry",
            "implies_distinct_horse": False,
            "database_correction_permitted": False,
            "notes": (
                "Both labels identify real stallions; official "
                "confirmation is required before correction."
            ),
        },
        {
            "horse": "Almavillalobas (GB)",
            "pedigree_role": "dam",
            "raw_competing_labels": (
                "Nation (USA) | Nation II (USA)"
            ),
            "governed_label": pd.NA,
            "reconciliation_type": "pending_official_confirmation",
            "evidence_status": "pending_official_confirmation",
            "verification_id": pd.NA,
            "authority_or_source": (
                "Weatherbys General Stud Book enquiry"
            ),
            "implies_distinct_horse": False,
            "database_correction_permitted": False,
            "notes": (
                "Nation II is probably a publisher-added numeral, "
                "but official confirmation is pending."
            ),
        },
    ]
)

reconciliation_summary = (
    pedigree_reconciliations
    .groupby(
        [
            "reconciliation_type",
            "evidence_status",
            "database_correction_permitted",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        reconciliations=("horse", "size"),
        horse_labels=("horse", "nunique"),
    )
    .sort_values(
        [
            "database_correction_permitted",
            "reconciliation_type",
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(pedigree_reconciliations) == 11
assert pedigree_reconciliations["horse"].is_unique

assert (
    pedigree_reconciliations[
        "evidence_status"
    ].eq("pending_official_confirmation").sum()
    == 5
)

assert (
    pedigree_reconciliations[
        "database_correction_permitted"
    ].sum()
    == 6
)

print("Explicit pedigree reconciliations")
display(reconciliation_summary)

print("Cases currently permitting governed correction")
pedigree_reconciliations.loc[
    pedigree_reconciliations[
        "database_correction_permitted"
    ],
    [
        "horse",
        "pedigree_role",
        "raw_competing_labels",
        "governed_label",
        "reconciliation_type",
        "evidence_status",
    ],
].reset_index(drop=True)

Explicit pedigree reconciliations


,reconciliation_type,evidence_status,database_correction_permitted,reconciliations,horse_labels
0,country_suffix_defect,authoritatively_supported,True,1,1
1,country_suffix_defect,strongly_supported,True,1,1
2,incorrect_entity_assignment,externally_verified,True,1,1
3,label_equivalence,officially_verified,True,1,1
4,label_equivalence_and_incomplete_pedigree_asse...,officially_verified,True,1,1
5,source_prefix_variant,authoritatively_supported,True,1,1
6,pending_official_confirmation,pending_official_confirmation,False,5,5


Cases currently permitting governed correction


,horse,pedigree_role,raw_competing_labels,governed_label,reconciliation_type,evidence_status
0,Felix Felicis (FR),complete_pedigree,Olympic Glory — Sorina — Le Havre | Affinisea ...,Olympic Glory — Sorina — Le Havre,incorrect_entity_assignment,externally_verified
1,New President (FR),damsire,Sun Song | Sun Song I | Sun Song II | blank,Dr Fong,label_equivalence_and_incomplete_pedigree_asse...,officially_verified
2,Herbert (NZ),sire,Warning Flag (USA) | Sweet Orange (USA),Sweet Orange (USA),label_equivalence,officially_verified
3,Bonny Ezra (NZ),dam,Ascolini (AUS) | Ascolini (NZ),Ascolini (NZ),country_suffix_defect,authoritatively_supported
4,Alderley Charlie (GB),damsire,Ut*Windsor Heights | Windsor Heights,Windsor Heights,source_prefix_variant,authoritatively_supported
5,Hangry (IRE),sire,Galileo (FR) | Galileo (IRE),Galileo (IRE),country_suffix_defect,strongly_supported


### Conclusion — Governed pedigree reconciliation

Eleven material pedigree discrepancies have been converted into explicit governed reconciliation records.

Six cases currently permit downstream correction or normalization:

| horse                   | governed outcome                                                       |
| ----------------------- | ---------------------------------------------------------------------- |
| `Felix Felicis (FR)`    | reject the incorrect early complete pedigree                           |
| `New President (FR)`    | retain one dam identity and restore `Dr Fong` as damsire               |
| `Herbert (NZ)`          | treat `Warning Flag` and `Sweet Orange` as verified sire aliases       |
| `Bonny Ezra (NZ)`       | correct the dam suffix to `Ascolini (NZ)`                              |
| `Alderley Charlie (GB)` | remove the bounded `Ut*` source prefix from the governed damsire label |
| `Hangry (IRE)`          | correct the sire suffix to `Galileo (IRE)`                             |

These reconciliations do not overwrite the source rows. They establish a governed interpretation alongside the immutable raw assertions.

Five cases remain pending official confirmation:

* `Almavillalobas (GB)`;
* `Colwyn Bay (FR)`;
* `Diamond Tipp (IRE)`;
* `L’Aziza des Places (FR)`;
* `Runninsonofagun (IRE)`.

For those cases:

* no canonical pedigree value is assigned;
* database correction remains prohibited;
* competing raw assertions remain visible;
* and the enquiry authority is recorded.

The reconciliation register demonstrates that pedigree disagreement and horse-identity disagreement are separate problems.

A source row may contain:

* the correct horse but an incorrect country suffix;
* the correct horse but a publisher-added numeral;
* a legitimate alias;
* an incomplete pedigree;
* or an entirely incorrect pedigree assertion.

Only the last category potentially overlaps with horse-identity splitting, and even then the surrounding chronology and external evidence must be assessed before treating the rows as different horses.


### Stage 9 — Assigning provisional horse occurrences

The pedigree discrepancies that can currently be reconciled have now been separated from genuine same-label horse collisions.

A provisional occurrence may therefore be assigned using the governed transition decisions.

For each exact raw horse label:

* the first structured pedigree group begins occurrence `1`;
* `split_provisional_occurrence` starts a new occurrence because the same label represents a different horse history;
* `retain_single_occurrence` keeps the adjoining groups together because the discrepancy has been classified as a label variant, metadata defect, alias, incomplete assertion or incorrect source pedigree;
* `pending_official_confirmation` remains within one occurrence provisionally, but the unresolved boundary must remain visible.

The resulting identifier is source-internal and provisional. It does not replace an official registration or life number.

Its purpose is to prevent histories belonging to different horses from being merged merely because they share:

* the same displayed name;
* and the same breeding-country suffix.

At the same time, it avoids splitting one real horse because of:

* spelling or punctuation;
* Roman numerals;
* source prefixes;
* aliases;
* incorrect country suffixes;
* missing pedigree fields;
* or known source defects.

Each occurrence must retain:

* the raw horse label;
* occurrence sequence within that label;
* included structured pedigree groups;
* first and last observed dates;
* observed age and sex history;
* runner-row lineage;
* the governed boundaries that created the occurrence;
* pending official-confirmation boundaries;
* and any verification identifiers.

A provisional occurrence must not be presented as a verified real-world horse entity unless it is later linked to an authoritative registration identifier.


In [38]:
# Assign provisional horse-occurrence identifiers from governed boundaries.

required_group_columns = {
    "horse",
    "group_number",
    "sire",
    "dam_structured_key",
    "damsire",
    "runner_rows",
    "first_date",
    "last_date",
    "minimum_age",
    "maximum_age",
    "sex_values",
}

missing_group_columns = required_group_columns.difference(
    separated_groups.columns
)

assert not missing_group_columns, (
    "separated_groups is missing required columns: "
    f"{sorted(missing_group_columns)}"
)

required_governance_columns = {
    "horse",
    "group_number",
    "transition_decision",
    "identity_split",
    "decision_basis",
    "governing_verification_id",
}

missing_governance_columns = required_governance_columns.difference(
    transition_governance.columns
)

assert not missing_governance_columns, (
    "transition_governance is missing required columns: "
    f"{sorted(missing_governance_columns)}"
)

# A transition recorded against group N governs the boundary
# between group N and group N + 1.
occurrence_boundaries = (
    transition_governance[
        [
            "horse",
            "group_number",
            "transition_decision",
            "identity_split",
            "decision_basis",
            "governing_verification_id",
        ]
    ]
    .copy()
)

occurrence_boundaries["target_group_number"] = (
    occurrence_boundaries["group_number"] + 1
)

occurrence_boundaries = occurrence_boundaries.rename(
    columns={
        "transition_decision": "boundary_decision",
        "identity_split": "split_before_group",
        "decision_basis": "boundary_basis",
        "governing_verification_id": "boundary_verification_id",
    }
)

occurrence_groups = (
    separated_groups
    .merge(
        occurrence_boundaries[
            [
                "horse",
                "target_group_number",
                "boundary_decision",
                "split_before_group",
                "boundary_basis",
                "boundary_verification_id",
            ]
        ],
        how="left",
        left_on=["horse", "group_number"],
        right_on=["horse", "target_group_number"],
        validate="one_to_one",
    )
    .drop(columns=["target_group_number"])
    .sort_values(
        ["horse", "group_number"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# The first group for each label has no preceding transition.
occurrence_groups["boundary_decision"] = (
    occurrence_groups["boundary_decision"]
    .fillna("first_group_for_label")
)

occurrence_groups["boundary_basis"] = (
    occurrence_groups["boundary_basis"]
    .fillna("start_of_exact_horse_label_history")
)

# Only explicitly governed identity splits begin a new occurrence.
occurrence_groups["split_before_group"] = (
    occurrence_groups["split_before_group"]
    .fillna(False)
    .astype(bool)
)

occurrence_groups["occurrence_sequence"] = (
    occurrence_groups
    .groupby("horse", sort=False)["split_before_group"]
    .cumsum()
    .astype(int)
    + 1
)

occurrence_groups["provisional_occurrence_id"] = (
    occurrence_groups["horse"]
    + "::"
    + occurrence_groups[
        "occurrence_sequence"
    ].astype(str).str.zfill(2)
)

provisional_occurrences = (
    occurrence_groups
    .groupby(
        [
            "horse",
            "occurrence_sequence",
            "provisional_occurrence_id",
        ],
        as_index=False,
        sort=False,
    )
    .agg(
        pedigree_groups=("group_number", "size"),
        runner_rows=("runner_rows", "sum"),
        first_date=("first_date", "min"),
        last_date=("last_date", "max"),
        minimum_age=("minimum_age", "min"),
        maximum_age=("maximum_age", "max"),
        sex_values=(
            "sex_values",
            lambda values: " | ".join(
                dict.fromkeys(
                    str(value)
                    for value in values
                    if pd.notna(value)
                )
            ),
        ),
        pending_boundaries=(
            "boundary_decision",
            lambda values: sum(
                value == "pending_official_confirmation"
                for value in values
            ),
        ),
        governing_verifications=(
            "boundary_verification_id",
            lambda values: " | ".join(
                dict.fromkeys(
                    str(value)
                    for value in values
                    if pd.notna(value)
                )
            ),
        ),
    )
    .sort_values(
        ["horse", "occurrence_sequence"],
        kind="stable",
    )
    .reset_index(drop=True)
)

occurrence_assignment_summary = pd.DataFrame(
    [
        {
            "measure": "exact horse labels",
            "value": occurrence_groups["horse"].nunique(),
        },
        {
            "measure": "structured pedigree groups",
            "value": len(occurrence_groups),
        },
        {
            "measure": "governed split boundaries",
            "value": int(
                occurrence_groups["split_before_group"].sum()
            ),
        },
        {
            "measure": "provisional occurrences",
            "value": len(provisional_occurrences),
        },
        {
            "measure": "labels split into multiple occurrences",
            "value": int(
                provisional_occurrences
                .groupby("horse")
                .size()
                .gt(1)
                .sum()
            ),
        },
        {
            "measure": "occurrences containing pending boundaries",
            "value": int(
                provisional_occurrences[
                    "pending_boundaries"
                ].gt(0).sum()
            ),
        },
    ]
)

assert occurrence_groups["horse"].nunique() == 350
assert len(occurrence_groups) == 703
assert occurrence_groups["split_before_group"].sum() == 261
assert len(provisional_occurrences) == 611

assert provisional_occurrences[
    "provisional_occurrence_id"
].is_unique

assert (
    provisional_occurrences[
        "pending_boundaries"
    ].gt(0).sum()
    == 5
)

print("Provisional occurrence assignment confirmed")
display(occurrence_assignment_summary)

print("Examples of labels split into multiple horse occurrences")

split_horse_labels = (
    provisional_occurrences
    .groupby("horse")
    .size()
    .loc[lambda values: values.gt(1)]
    .index
)

provisional_occurrences.loc[
    provisional_occurrences["horse"].isin(split_horse_labels),
    [
        "horse",
        "occurrence_sequence",
        "provisional_occurrence_id",
        "pedigree_groups",
        "runner_rows",
        "first_date",
        "last_date",
        "minimum_age",
        "maximum_age",
    ],
].head(30)

Provisional occurrence assignment confirmed


,measure,value
0,exact horse labels,350
1,structured pedigree groups,703
2,governed split boundaries,261
3,provisional occurrences,611
4,labels split into multiple occurrences,261
5,occurrences containing pending boundaries,5


Examples of labels split into multiple horse occurrences


,horse,occurrence_sequence,provisional_occurrence_id,pedigree_groups,runner_rows,first_date,last_date,minimum_age,maximum_age
0,A La Prochaine (FR),1,A La Prochaine (FR)::01,1,14,2015-03-14,2016-06-12,5,6
1,A La Prochaine (FR),2,A La Prochaine (FR)::02,1,2,2025-10-25,2026-05-06,2,3
4,Al Daayen (FR),1,Al Daayen (FR)::01,1,12,2019-01-29,2019-11-27,3,3
5,Al Daayen (FR),2,Al Daayen (FR)::02,1,9,2023-11-05,2025-05-29,2,4
6,Alcala (FR),1,Alcala (FR)::01,1,33,2015-10-21,2022-02-03,5,12
7,Alcala (FR),2,Alcala (FR)::02,1,4,2024-04-29,2025-01-21,3,4
9,All Good (GB),1,All Good (GB)::01,1,1,2018-06-09,2018-06-09,4,4
10,All Good (GB),2,All Good (GB)::02,1,2,2025-09-19,2025-10-14,2,2
12,Amoretti (FR),1,Amoretti (FR)::01,1,1,2017-08-28,2017-08-28,13,13
13,Amoretti (FR),2,Amoretti (FR)::02,1,2,2023-05-25,2024-04-16,2,3


### Conclusion — Provisional horse-occurrence assignment

The governed transition decisions produce:

| measure                                      | value |
| -------------------------------------------- | ----: |
| exact raw horse labels assessed              |   350 |
| structured pedigree groups                   |   703 |
| governed split boundaries                    |   261 |
| provisional horse occurrences                |   611 |
| labels split into multiple occurrences       |   261 |
| occurrences containing unresolved boundaries |     5 |

The 350 exact horse labels do not represent 350 stable horse identities.

After applying the governed boundaries, they expand to 611 provisional occurrences because 261 labels are reused for distinct horse histories.

The split examples show a consistent pattern:

* an earlier horse history ends;
* several years pass;
* the same name and breeding-country suffix reappear;
* the pedigree differs materially;
* and the age restarts at a younger value incompatible with continuous progression.

For example:

* `A La Prochaine (FR)` separates into a 2015–2016 occurrence aged five to six and a 2025–2026 occurrence aged two to three;
* `Alcala (FR)` separates into a 2015–2022 occurrence aged five to twelve and a 2024–2025 occurrence aged three to four;
* `Anglophile (GB)` separates into a 2015–2016 occurrence aged four to five and a 2024–2025 occurrence aged two to three.

These are not minor formatting differences. They are distinct horses sharing the same source label.

The provisional occurrence identifier therefore prevents analytical histories from being merged incorrectly while preserving the original source label on every row.

Five provisional occurrences still contain a boundary awaiting official confirmation. Those boundaries remain unsplit for now, but the uncertainty is carried explicitly rather than hidden.

The identifier remains source-internal:

> `raw horse label + governed occurrence sequence`

It is suitable for controlled analysis within this dataset, but it must not be treated as an official registration number or globally permanent horse identifier.


In [39]:
# Inspect exact horse labels split into three or more provisional occurrences.

occurrence_counts_by_label = (
    provisional_occurrences
    .groupby("horse", as_index=False)
    .agg(
        provisional_occurrences=(
            "provisional_occurrence_id",
            "size",
        ),
        first_observed_date=("first_date", "min"),
        last_observed_date=("last_date", "max"),
        total_runner_rows=("runner_rows", "sum"),
    )
    .sort_values(
        [
            "provisional_occurrences",
            "horse",
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

multi_occurrence_labels = occurrence_counts_by_label.loc[
    occurrence_counts_by_label[
        "provisional_occurrences"
    ].ge(3)
].reset_index(drop=True)

multi_occurrence_details = (
    provisional_occurrences.loc[
        provisional_occurrences["horse"].isin(
            multi_occurrence_labels["horse"]
        ),
        [
            "horse",
            "occurrence_sequence",
            "provisional_occurrence_id",
            "pedigree_groups",
            "runner_rows",
            "first_date",
            "last_date",
            "minimum_age",
            "maximum_age",
            "sex_values",
        ],
    ]
    .sort_values(
        [
            "horse",
            "occurrence_sequence",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Labels assigned three or more provisional occurrences")
display(multi_occurrence_labels)

print("Occurrence-level details")
multi_occurrence_details

Labels assigned three or more provisional occurrences


,horse,provisional_occurrences,first_observed_date,last_observed_date,total_runner_rows


Occurrence-level details


,horse,occurrence_sequence,provisional_occurrence_id,pedigree_groups,runner_rows,first_date,last_date,minimum_age,maximum_age,sex_values


### Check — Labels with three or more provisional occurrences

No exact raw horse label produced three or more provisional horse occurrences.

Although five labels previously contained three or more structured pedigree groups, those groups did not all represent distinct horses.

After applying the transition-level governance:

* formatting variants;
* aliases;
* source defects;
* incomplete assertions;
* and other reconciled pedigree differences

remain within the same occurrence.

Every label requiring an identity split therefore produces exactly two provisional occurrences in the current dataset.

This provides an additional consistency check:

> The occurrence model is not mechanically creating a new horse for every changed pedigree assertion.

Instead, new occurrences are created only at boundaries explicitly governed as same-label reuse by different horses.


### Reader-facing conclusion — Horse and pedigree identity

The raw `horse` label is not a safe permanent horse identifier.

The analysis found three practical outcomes.

#### Corrected

Where reliable evidence establishes the right horse or pedigree, the clean analytical layer should use that corrected result.

The original source value remains available only for audit and provenance.

#### Different horse

Where the same displayed name and breeding-country suffix have been reused for genuinely different horses, the histories must be split.

These records must not be merged merely because the raw label matches.

#### Unresolved

Where the correct horse or pedigree cannot yet be established confidently, the next question is whether manual verification is practical.

Manual checking is worthwhile where:

* the number of affected horses is small;
* an official Stud Book or racing authority can be contacted;
* the result would materially affect later analysis;
* and the verification effort is proportionate.

Where practical, the case should be investigated and converted to either:

* `Corrected`; or
* `Different horse`.

Only cases that cannot be resolved at reasonable cost should remain unresolved.

Those records should then be flagged or excluded from horse- or pedigree-dependent analysis rather than guessed.

The practical database rule is therefore:

> Use the corrected horse and pedigree where known, split genuine same-name collisions, manually verify unresolved cases where practical, and exclude rather than guess where uncertainty remains.

This produces a clean analytical identity layer without discarding the raw source evidence needed to verify how each decision was reached.


## Final conclusion

### Bounded question

This notebook investigated what the runner-level `horse`, `sire`, `dam` and `damsire` fields represent, whether those labels are stable enough to support horse- and pedigree-level analysis, and what identity rules are required before the fields can be used safely.

### Conclusion

The raw `horse` field is a source-presented label, not a permanent horse identifier.

The same displayed horse name and breeding-country suffix can be reused for different real horses. Conversely, one real horse can appear beside incorrect or inconsistent pedigree assertions.

Horse identity and pedigree therefore require a governed analytical layer rather than direct use of the raw strings.

The 353 material transitions between structured pedigree histories produced three practical outcomes:

| analytical outcome | transitions | exact horse labels |
| ------------------ | ----------: | -----------------: |
| Corrected          |          87 |                 84 |
| Different horse    |         261 |                261 |
| Unresolved         |           5 |                  5 |
| **Total**          |     **353** |                  — |

### Corrected

Eighty-seven transitions across 84 exact horse labels belong to one continuous horse history.

Where the correct pedigree has been established, the analytical layer should use the corrected result rather than preserving competing values as equally valid.

Examples include:

* verified aliases;
* incorrect country suffixes;
* source prefixes;
* missing pedigree fields;
* and rows carrying a pedigree belonging to another horse.

The original source values must remain unchanged for lineage and audit, but they should not remain active alternatives in analysis once the correct interpretation is governed.

### Different horse

Two hundred and sixty-one transitions represent genuinely different horses sharing the same displayed name and breeding-country suffix.

These histories must be separated before horse-level analysis.

Applying those governed split boundaries to the 350 exact labels examined produced:

* 703 structured pedigree groups;
* 261 split boundaries;
* 611 provisional horse occurrences;
* and 261 labels divided into two distinct horse histories.

No label produced three or more provisional occurrences.

The occurrence identifier is a source-internal analytical key. It prevents different horses from being merged, but it is not an official registration or life number.

### Unresolved

Five cases remain unresolved:

* `Almavillalobas (GB)`;
* `Colwyn Bay (FR)`;
* `Diamond Tipp (IRE)`;
* `L’Aziza des Places (FR)`;
* `Runninsonofagun (IRE)`.

Manual verification is practical because the residue is small and the disputed relationships may affect later pedigree analysis.

Enquiries have been sent to the relevant Stud Book or registration authorities.

Until authoritative confirmation is received:

* both source assertions must remain preserved;
* no canonical pedigree value should be assigned;
* no identity split should be created merely from publication consensus;
* and the disputed field should be flagged or excluded from dependent analysis.

Each unresolved case should eventually become either:

* `Corrected`; or
* `Different horse`.

### Database consequence

The analytical database should preserve the raw fields unchanged and add governed identity and pedigree fields separately.

The minimum required structure is:

* raw horse, sire, dam and damsire labels;
* source database, table and row lineage;
* governed horse-occurrence identifier;
* governed sire, dam and damsire where established;
* analytical outcome: `Corrected`, `Different horse` or `Unresolved`;
* verification or evidence identifier;
* review status;
* and an unresolved flag.

The operational rule is:

> Use corrected horse and pedigree values where established, split genuinely different horses sharing the same label, manually verify the finite unresolved residue where practical, and exclude rather than guess while uncertainty remains.

### Confidence

Confidence is high in the central conclusion that raw horse labels cannot serve as permanent natural keys.

Confidence is also high in the 261 same-label splits because they are supported by materially different pedigrees, separated chronology and generally incompatible age progression.

Confidence in individual corrected pedigrees varies with the authority of the supporting evidence and must remain recorded in the verification register.

### Limitations

The study is bounded to:

* the supplied source database;
* runner rows selected by `rowid <> 1`;
* the source period represented in that database;
* and the labels and pedigree assertions present there.

The provisional occurrence identifiers are not globally unique horse identities.

They do not establish that all horses outside the identified contradiction set are correctly represented.

The notebook also does not prove that every source pedigree is correct merely because it remained stable.

Five material cases remain pending authority replies, and their governed values may change when those replies are received.

### What this result justifies

This work justifies:

* avoiding raw horse labels as permanent analytical keys;
* separating the identified same-label horse histories;
* applying governed pedigree corrections with retained provenance;
* and excluding unresolved relationships from dependent analysis.

It does not justify:

* overwriting the immutable source;
* treating name similarity as identity proof;
* claiming a provisional occurrence as an official horse identity;
* or assuming the source is professionally complete or error-free.

### Notebook status

The analytical investigation is complete.

Notebook 19 should be closed through the non-rerunnable archival construction-record route.

The executed notebook preserves the investigation and evidence trail. Durable use of the conclusions must be provided separately through governed outputs, reusable implementation, focused tests, independent source-wide validation and database-integration documentation.
